In [ ]:
%cd /content/drive/MyDrive/FreightQuote_AI

/content/drive/MyDrive/FreightQuote_AI


In [ ]:
!ls -la

total 64
drwx------ 10 root root  4096 Oct 24 20:57 .
drwx------ 14 root root  4096 Oct 24 22:03 ..
drwx------  2 root root  4096 Oct 24 20:57 agents
-rw-------  1 root root 15032 Oct 24 20:53 app.py
drwx------  2 root root  4096 Oct 24 20:57 data
drwx------  2 root root  4096 Oct 24 20:57 database
-rw-------  1 root root  2523 Oct 24 20:53 FreightQuote_AI_Colab.ipynb
-rw-------  1 root root    34 Oct 24 20:53 .gitignore
drwx------  2 root root  4096 Oct 24 20:57 logs
drwx------  2 root root  4096 Oct 24 20:57 pages
-rw-------  1 root root  1929 Oct 24 20:53 README.md
-rw-------  1 root root    64 Oct 24 20:53 requirements.txt
-rw-------  1 root root  3254 Oct 24 20:53 run_colab.py
drwx------  2 root root  4096 Oct 24 20:57 services
drwx------  2 root root  4096 Oct 24 20:57 tests
drwx------  2 root root  4096 Oct 24 20:57 utils


In [ ]:
%cd /content/drive/MyDrive/FreightQuote_AI
!pip install -r requirements.txt

/content/drive/MyDrive/FreightQuote_AI
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 104.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 125.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [streamlit]


In [ ]:
!cat run_colab.py

"""
Start FreightQuote AI in Google Colab: Streamlit in the background + ngrok tunnel.

Usage in a Colab cell:
    %run /content/FreightQuote_AI/run_colab.py
Optional: set PORT before running, e.g.  import os; os.environ["FQ_PORT"] = "8504"
"""
import os
import signal
import subprocess
import sys
import time
from pathlib import Path

import requests
from pyngrok import ngrok, conf

PROJECT_DIR = Path(__file__).resolve().parent
PORT = int(os.environ.get("FQ_PORT", "8503"))
LOG_FILE = PROJECT_DIR / "logs" / "streamlit_server.log"


def get_secret(name):
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception:
        value = None
    value = value or os.environ.get(name)
    return value.strip() if value else None


def stop_old():
    # Stop every old Streamlit server and tunnel (frees 8501/8502/8503...)
    subprocess.run(["pkill", "-f", "streamlit run"], check=False)
    try:
        ngrok.kill()
    except Exception:
        pass
   

In [ ]:
%cd /content/drive/MyDrive/FreightQuote_AI
!pip install -r requirements.txt
!pip install pyngrok requests

/content/drive/MyDrive/FreightQuote_AI


In [ ]:
%%writefile run_colab.py
"""
Start FreightQuote AI in Google Colab: Streamlit in the background + ngrok tunnel.

Usage in a Colab cell:
    %run /content/FreightQuote_AI/run_colab.py
Optional: set PORT before running, e.g.  import os; os.environ["FQ_PORT"] = "8504"
"""
import os
import signal
import subprocess
import sys
import time
from pathlib import Path

import requests
from pyngrok import ngrok, conf

PROJECT_DIR = Path(__file__).resolve().parent
PORT = int(os.environ.get("FQ_PORT", "8503"))
LOG_FILE = PROJECT_DIR / "logs" / "streamlit_server.log"


def get_secret(name):
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception:
        value = None
    value = value or os.environ.get(name)
    return value.strip() if value else None


def stop_old():
    # Stop every old Streamlit server and tunnel (frees 8501/8502/8503...)
    subprocess.run(["pkill", "-f", "streamlit run"], check=False)
    try:
        ngrok.kill()
    except Exception:
        pass
    time.sleep(2)


def start_streamlit():
    LOG_FILE.parent.mkdir(parents=True, exist_ok=True)
    log = open(LOG_FILE, "w")
    # Colab Secrets can only be read in the notebook kernel, not inside the
    # Streamlit subprocess, so pass them to the app as environment variables.
    env = os.environ.copy()
    for name in ("SMTP_EMAIL", "SMTP_APP_PASSWORD"):
        value = get_secret(name)
        if value:
            env[name] = value
    proc = subprocess.Popen(
        [sys.executable, "-m", "streamlit", "run", str(PROJECT_DIR / "app.py"),
         "--server.port", str(PORT),
         "--server.address", "127.0.0.1",
         "--server.headless", "true",
         "--browser.gatherUsageStats", "false"],
        cwd=str(PROJECT_DIR), env=env, stdout=log, stderr=log,
        preexec_fn=os.setsid,
    )
    health = f"http://127.0.0.1:{PORT}/_stcore/health"
    for _ in range(30):
        if proc.poll() is not None:
            break
        try:
            if requests.get(health, timeout=2).ok:
                print(f"Streamlit running on 127.0.0.1:{PORT} (pid {proc.pid})")
                return proc
        except requests.RequestException:
            pass
        time.sleep(1)
    print("Streamlit failed to start. Log:\n")
    print(LOG_FILE.read_text(errors="replace"))
    raise SystemExit(1)


def start_tunnel():
    token = get_secret("NGROK_AUTHTOKEN")
    if not token:
        raise SystemExit("NGROK_AUTHTOKEN missing. Add it in Colab Secrets and enable notebook access.")
    conf.get_default().auth_token = token
    # Connect to the local port using correct syntax
    tunnel = ngrok.connect(addr=PORT, proto="http")
    url = tunnel.public_url
    r = requests.get(url, headers={"ngrok-skip-browser-warning": "true"}, timeout=20)
    print("\n" + "=" * 55)
    print("FreightQuote AI URL:", url)
    print("Health check:", "PASS" if r.ok else f"HTTP {r.status_code}")
    print("=" * 55)
    return url


def check_smtp_secrets():
    for name in ("SMTP_EMAIL", "SMTP_APP_PASSWORD"):
        print(f"{name}: {'OK' if get_secret(name) else 'MISSING (OTP email will fail)'}")


if __name__ == "__main__":
    check_smtp_secrets()
    stop_old()
    start_streamlit()
    start_tunnel()

Overwriting run_colab.py


In [ ]:
from google.colab import userdata
from pyngrok import ngrok, conf
import requests

# Configure ngrok authentication
token = userdata.get("NGROK_AUTHTOKEN")
conf.get_default().auth_token = token

# Clear the failed tunnel and create a new one
ngrok.kill()
tunnel = ngrok.connect("8503", "http")

print("FreightQuote AI URL:", tunnel.public_url)

# Check the public URL
try:
    response = requests.get(
        tunnel.public_url,
        headers={"ngrok-skip-browser-warning": "true"},
        timeout=20
    )
    print("HTTP status:", response.status_code)
except requests.RequestException as e:
    print("URL created, but health check failed:", e)

FreightQuote AI URL: https://reptile-unspoken-tuition.ngrok-free.dev
HTTP status: 200


In [ ]:
import os
import smtplib
from google.colab import userdata

email = userdata.get("SMTP_EMAIL")
app_password = userdata.get("SMTP_APP_PASSWORD")

try:
    with smtplib.SMTP("smtp.gmail.com", 587, timeout=20) as server:
        server.starttls()
        server.login(email, app_password)

    print("SUCCESS: Gmail SMTP authentication working.")

except Exception as e:
    print("SMTP ERROR:", type(e).__name__, str(e))

SUCCESS: Gmail SMTP authentication working.


In [ ]:
!pip install bcrypt

In [ ]:
from pathlib import Path

p = Path("/content/drive/MyDrive/FreightQuote_AI/security_auth.py")

if p.exists():
    print("File already exists:", p)
else:
    p.write_text("# FreightQuote AI security authentication\n")
    print("File created:", p)

File created: /content/drive/MyDrive/FreightQuote_AI/security_auth.py


In [ ]:

import sqlite3
import bcrypt
import time
import hmac
import re
from pathlib import Path
DB_PATH = (
    Path("/content/drive/MyDrive/FreightQuote_AI")
    / "database"
    / "security_auth.db"
)

MAX_ATTEMPTS = 3
LOCK_SECONDS = 15 * 60  # 15 minutes
QUESTIONS = {
    "pet": "What is your pet's name?",
    "school": "What was your first school's name?",
    "city": "What is your favourite city?",
}


def connect_db():
    DB_PATH.parent.mkdir(parents=True, exist_ok=True)
    conn = sqlite3.connect(str(DB_PATH), timeout=10)
    conn.row_factory = sqlite3.Row
    return conn


def init_db():
    with connect_db() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS auth_users (
                email TEXT PRIMARY KEY,
                password_hash BLOB NOT NULL,
                pet_hash BLOB NOT NULL,
                school_hash BLOB NOT NULL,
                city_hash BLOB NOT NULL,
                failed_attempts INTEGER NOT NULL DEFAULT 0,
                locked_until REAL NOT NULL DEFAULT 0
            )
        """)
        conn.commit()


def normalize(value):
    return " ".join(value.strip().casefold().split())


def hash_value(value):
    return bcrypt.hashpw(
        normalize(value).encode("utf-8"),
        bcrypt.gensalt(rounds=12)
    )


def check_value(value, stored_hash):
    try:
        return bcrypt.checkpw(
            normalize(value).encode("utf-8"),
            bytes(stored_hash)
        )
    except (ValueError, TypeError):
        return False


def valid_password(password):
    return (
        len(password) >= 10
        and re.search(r"[A-Z]", password)
        and re.search(r"[a-z]", password)
        and re.search(r"\d", password)
    )


def register_user(email, password, pet, school, city):
    email = email.strip().lower()

    if not valid_password(password):
        return False, (
            "Password must be at least 10 characters "
            "and include uppercase, lowercase and a number."
        )

    if not all(x.strip() for x in [email, pet, school, city]):
        return False, "All fields are required."

    try:
        with connect_db() as conn:
            conn.execute("""
                INSERT INTO auth_users
                (email, password_hash, pet_hash, school_hash, city_hash)
                VALUES (?, ?, ?, ?, ?)
            """, (
                email,
                hash_value(password),
                hash_value(pet),
                hash_value(school),
                hash_value(city),
            ))
            conn.commit()

        return True, "Account created successfully."

    except sqlite3.IntegrityError:
        return False, "Unable to create account. Check the details."


def login_user(email, password):
    email = email.strip().lower()

    with connect_db() as conn:
        user = conn.execute(
            "SELECT * FROM auth_users WHERE email = ?",
            (email,)
        ).fetchone()

        if not user:
            return False, "Invalid email or password."

        now = time.time()

        if user["locked_until"] > now:
            remaining = int(user["locked_until"] - now)
            minutes = (remaining + 59) // 60
            return False, f"Account locked. Try again in {minutes} minute(s)."

        # Reset an expired lockout.
        if user["locked_until"] and user["locked_until"] <= now:
            conn.execute("""
                UPDATE auth_users
                SET failed_attempts = 0, locked_until = 0
                WHERE email = ?
            """, (email,))
            conn.commit()

        if check_value(password, user["password_hash"]):
            conn.execute("""
                UPDATE auth_users
                SET failed_attempts = 0, locked_until = 0
                WHERE email = ?
            """, (email,))
            conn.commit()
            return True, "Login successful."

        attempts = user["failed_attempts"] + 1

        if attempts >= MAX_ATTEMPTS:
            conn.execute("""
                UPDATE auth_users
                SET failed_attempts = ?, locked_until = ?
                WHERE email = ?
            """, (attempts, now + LOCK_SECONDS, email))
            conn.commit()
            return False, "Too many failed attempts. Account locked for 15 minutes."

        conn.execute("""
            UPDATE auth_users
            SET failed_attempts = ?
            WHERE email = ?
        """, (attempts, email))
        conn.commit()

        remaining = MAX_ATTEMPTS - attempts
        return False, f"Invalid email or password. Attempts remaining: {remaining}."


def reset_password(email, pet, school, city, new_password):
    email = email.strip().lower()

    if not valid_password(new_password):
        return False, (
            "New password must be at least 10 characters "
            "and include uppercase, lowercase and a number."
        )

    with connect_db() as conn:
        user = conn.execute(
            "SELECT * FROM auth_users WHERE email = ?",
            (email,)
        ).fetchone()

        if not user:
            return False, "Unable to verify account or security answers."

        # Verify all three answers.
        checks = [
            check_value(pet, user["pet_hash"]),
            check_value(school, user["school_hash"]),
            check_value(city, user["city_hash"]),
        ]

        if not all(checks):
            return False, "Unable to verify account or security answers."

        conn.execute("""
            UPDATE auth_users
            SET password_hash = ?,
                failed_attempts = 0,
                locked_until = 0
            WHERE email = ?
        """, (hash_value(new_password), email))
        conn.commit()

    return True, "Password reset successfully. You can now log in."


init_db()


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from pathlib import Path

project = Path("/content/drive/MyDrive/FreightQuote_AI")
app_file = project / "app.py"

print("File exists:", app_file.exists())
print("Path:", app_file)

File exists: True
Path: /content/drive/MyDrive/FreightQuote_AI/app.py


In [ ]:

import streamlit as st
from security_auth import (
    QUESTIONS,
    register_user,
    login_user,
    reset_password,
)

st.title("FreightQuote AI — Secure Login")

page = st.radio(
    "Choose an option",
    ["Login", "Register", "Forgot Password"],
    horizontal=True,
)

if page == "Register":
    st.subheader("Create Account")

    email = st.text_input("Email")
    password = st.text_input("Create password", type="password")

    st.caption("Set three security answers. Remember your exact answers.")

    pet = st.text_input(QUESTIONS["pet"])
    school = st.text_input(QUESTIONS["school"])
    city = st.text_input(QUESTIONS["city"])

    if st.button("Register"):
        ok, message = register_user(
            email, password, pet, school, city
        )
        (st.success if ok else st.error)(message)

elif page == "Login":
    st.subheader("Login")

    email = st.text_input("Email", key="login_email")
    password = st.text_input(
        "Password", type="password", key="login_password"
    )

    if st.button("Login"):
        ok, message = login_user(email, password)

        if ok:
            st.session_state["authenticated"] = True
            st.session_state["user_email"] = email.strip().lower()
            st.success(message)
            # Connect your existing dashboard here.
        else:
            st.error(message)

elif page == "Forgot Password":
    st.subheader("Reset Password Using Security Questions")

    email = st.text_input("Registered email address")
    pet = st.text_input(QUESTIONS["pet"])
    school = st.text_input(QUESTIONS["school"])
    city = st.text_input(QUESTIONS["city"])

    new_password = st.text_input(
        "New password", type="password"
    )
    confirm_password = st.text_input(
        "Confirm new password", type="password"
    )

    if st.button("Verify Answers & Reset Password"):
        if new_password != confirm_password:
            st.error("Passwords do not match.")
        else:
            ok, message = reset_password(
                email, pet, school, city, new_password
            )
            (st.success if ok else st.error)(message)


ImportError: cannot import name 'QUESTIONS' from 'security_auth' (/content/drive/MyDrive/FreightQuote_AI/security_auth.py)

In [ ]:
from pathlib import Path

file_path = Path(
    "/content/drive/MyDrive/FreightQuote_AI/security_auth.py"
)

print(file_path.read_text()[:3000])

# FreightQuote AI security authentication



In [ ]:
from pathlib import Path

file_path = Path(
    "/content/drive/MyDrive/FreightQuote_AI/security_auth.py"
)

code = r'''
import sqlite3
import bcrypt
import time
import re
from pathlib import Path

DB_PATH = Path(
    "/content/drive/MyDrive/FreightQuote_AI/database/security_auth.db"
)

MAX_ATTEMPTS = 3
LOCK_SECONDS = 15 * 60

QUESTIONS = {
    "pet": "What is your pet's name?",
    "school": "What was your first school's name?",
    "city": "What is your favourite city?",
}

def connect_db():
    DB_PATH.parent.mkdir(parents=True, exist_ok=True)
    conn = sqlite3.connect(str(DB_PATH), timeout=10)
    conn.row_factory = sqlite3.Row
    return conn

def init_db():
    with connect_db() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS auth_users (
                email TEXT PRIMARY KEY,
                password_hash BLOB NOT NULL,
                pet_hash BLOB NOT NULL,
                school_hash BLOB NOT NULL,
                city_hash BLOB NOT NULL,
                failed_attempts INTEGER NOT NULL DEFAULT 0,
                locked_until REAL NOT NULL DEFAULT 0
            )
        """)

def normalize(value):
    return " ".join(value.strip().casefold().split())

def hash_value(value):
    return bcrypt.hashpw(
        normalize(value).encode("utf-8"),
        bcrypt.gensalt(rounds=12)
    )

def check_value(value, stored_hash):
    try:
        return bcrypt.checkpw(
            normalize(value).encode("utf-8"),
            bytes(stored_hash)
        )
    except (ValueError, TypeError):
        return False

def valid_password(password):
    return (
        len(password) >= 10
        and bool(re.search(r"[A-Z]", password))
        and bool(re.search(r"[a-z]", password))
        and bool(re.search(r"\d", password))
    )

def register_user(email, password, pet, school, city):
    email = email.strip().lower()

    if not valid_password(password):
        return False, "Use at least 10 characters, uppercase, lowercase and a number."

    if not all(x.strip() for x in [email, pet, school, city]):
        return False, "All fields are required."

    try:
        with connect_db() as conn:
            conn.execute("""
                INSERT INTO auth_users
                (email, password_hash, pet_hash, school_hash, city_hash)
                VALUES (?, ?, ?, ?, ?)
            """, (
                email, hash_value(password), hash_value(pet),
                hash_value(school), hash_value(city)
            ))
        return True, "Account created successfully."
    except sqlite3.IntegrityError:
        return False, "Unable to create account. Check the details."

def login_user(email, password):
    email = email.strip().lower()

    with connect_db() as conn:
        user = conn.execute(
            "SELECT * FROM auth_users WHERE email = ?", (email,)
        ).fetchone()

        if not user:
            return False, "Invalid email or password."

        now = time.time()

        if user["locked_until"] > now:
            mins = int((user["locked_until"] - now + 59) // 60)
            return False, f"Account locked. Try again in {mins} minute(s)."

        if user["locked_until"] and user["locked_until"] <= now:
            conn.execute("""
                UPDATE auth_users
                SET failed_attempts = 0, locked_until = 0
                WHERE email = ?
            """, (email,))

        if check_value(password, user["password_hash"]):
            conn.execute("""
                UPDATE auth_users
                SET failed_attempts = 0, locked_until = 0
                WHERE email = ?
            """, (email,))
            return True, "Login successful."

        attempts = user["failed_attempts"] + 1

        if attempts >= MAX_ATTEMPTS:
            conn.execute("""
                UPDATE auth_users
                SET failed_attempts = ?, locked_until = ?
                WHERE email = ?
            """, (attempts, now + LOCK_SECONDS, email))
            return False, "Account locked for 15 minutes after 3 failed attempts."

        conn.execute("""
            UPDATE auth_users SET failed_attempts = ? WHERE email = ?
        """, (attempts, email))
        return False, f"Invalid email or password. Attempts remaining: {MAX_ATTEMPTS - attempts}."

def reset_password(email, pet, school, city, new_password):
    email = email.strip().lower()

    if not valid_password(new_password):
        return False, "Use at least 10 characters, uppercase, lowercase and a number."

    with connect_db() as conn:
        user = conn.execute(
            "SELECT * FROM auth_users WHERE email = ?", (email,)
        ).fetchone()

        if not user:
            return False, "Unable to verify account or security answers."

        if not all([
            check_value(pet, user["pet_hash"]),
            check_value(school, user["school_hash"]),
            check_value(city, user["city_hash"])
        ]):
            return False, "Unable to verify account or security answers."

        conn.execute("""
            UPDATE auth_users
            SET password_hash = ?, failed_attempts = 0, locked_until = 0
            WHERE email = ?
        """, (hash_value(new_password), email))

    return True, "Password reset successfully."

init_db()
'''

file_path.write_text(code, encoding="utf-8")
print("Saved:", file_path)
print("File size:", file_path.stat().st_size, "bytes")

Saved: /content/drive/MyDrive/FreightQuote_AI/security_auth.py
File size: 5143 bytes


In [ ]:
import sys
import importlib

project = "/content/drive/MyDrive/FreightQuote_AI"

if project not in sys.path:
    sys.path.insert(0, project)

import security_auth
importlib.reload(security_auth)

print("Questions:", security_auth.QUESTIONS)
print("Register:", callable(security_auth.register_user))
print("Login:", callable(security_auth.login_user))
print("Password reset:", callable(security_auth.reset_password))

Questions: {'pet': "What is your pet's name?", 'school': "What was your first school's name?", 'city': 'What is your favourite city?'}
Register: True
Login: True
Password reset: True


In [ ]:
from pathlib import Path
import shutil

project = Path("/content/drive/MyDrive/FreightQuote_AI")
app_file = project / "app.py"
backup_file = project / "app_backup_before_auth.py"

if not app_file.exists():
    print("ERROR: app.py not found")
elif backup_file.exists():
    print("Backup already exists:", backup_file)
else:
    shutil.copy2(app_file, backup_file)
    print("Backup created:", backup_file)

Backup already exists: /content/drive/MyDrive/FreightQuote_AI/app_backup_before_auth.py


In [ ]:
from pathlib import Path

app_file = Path(
    "/content/drive/MyDrive/FreightQuote_AI/app.py"
)

print(app_file.read_text(encoding="utf-8")[:12000])

"""
FreightQuote AI - Authentication module
Sign In (email or username) / Register / Forgot Password.

Register collects three security answers (pet name, first school, favorite city).
Sign In locks an account after too many wrong passwords.
Forgot Password works two ways:
    1. Security questions - no email setup needed
    2. Email OTP          - needs the two secrets below

Secrets (Colab Secrets or environment variables):
    SMTP_EMAIL         - Gmail address used to send OTP emails
    SMTP_APP_PASSWORD  - Gmail App Password (16 chars, spaces allowed)
"""

import os
import re
import ssl
import sqlite3
import secrets
import smtplib
from contextlib import contextmanager
from datetime import datetime, timedelta, timezone
from email.message import EmailMessage
from pathlib import Path

import bcrypt
import streamlit as st

# ---------------- Configuration ----------------

BASE_DIR = Path(__file__).resolve().parent
DB_PATH = Path(os.environ.get("FREIGHTQUOTE_DB", BASE_DIR / "freight

In [ ]:
from pathlib import Path

app_file = Path(
    "/content/drive/MyDrive/FreightQuote_AI/app.py"
)

lines = app_file.read_text(encoding="utf-8").splitlines()

print("\n".join(
    f"{i+1}: {line}"
    for i, line in enumerate(lines[370:], start=370)
))

371:         attempts = row["attempts"] if row else 0
372:         # an expired lockout starts a fresh count
373:         if row and row["locked_until"] and now_utc() >= parse_time(row["locked_until"]):
374:             attempts = 0
375:         attempts += 1
376: 
377:         locked_until = None
378:         if attempts >= MAX_ANSWER_ATTEMPTS:
379:             locked_until = (now_utc() + timedelta(minutes=ANSWER_LOCKOUT_MINUTES)).isoformat()
380:             attempts = 0
381: 
382:         conn.execute(
383:             """INSERT INTO answer_attempts (email, attempts, locked_until)
384:                VALUES (?, ?, ?)
385:                ON CONFLICT(email) DO UPDATE SET
386:                    attempts = excluded.attempts,
387:                    locked_until = excluded.locked_until""",
388:             (email, attempts, locked_until),
389:         )
390: 
391: 
392: def reset_with_answers(identity, answers, new_password, confirm):
393:     """Reset a password after all three securit

In [ ]:
from pathlib import Path

p = Path("/content/drive/MyDrive/FreightQuote_AI/app.py")
print(p.read_text(encoding="utf-8")[12000:])

_failure(email):
    with db() as conn:
        row = conn.execute(
            "SELECT attempts, locked_until FROM answer_attempts WHERE email = ?",
            (email,),
        ).fetchone()
        attempts = row["attempts"] if row else 0
        # an expired lockout starts a fresh count
        if row and row["locked_until"] and now_utc() >= parse_time(row["locked_until"]):
            attempts = 0
        attempts += 1

        locked_until = None
        if attempts >= MAX_ANSWER_ATTEMPTS:
            locked_until = (now_utc() + timedelta(minutes=ANSWER_LOCKOUT_MINUTES)).isoformat()
            attempts = 0

        conn.execute(
            """INSERT INTO answer_attempts (email, attempts, locked_until)
               VALUES (?, ?, ?)
               ON CONFLICT(email) DO UPDATE SET
                   attempts = excluded.attempts,
                   locked_until = excluded.locked_until""",
            (email, attempts, locked_until),
        )


def reset_with_answers(identity, an

In [ ]:

from pathlib import Path
import shutil
import sqlite3

project = Path("/content/drive/MyDrive/FreightQuote_AI")
app_file = project / "app.py"
db_file = project / "freightquote.db"
backup = project / "app_backup_before_security.py"

if not app_file.exists():
    raise FileNotFoundError(f"app.py not found: {app_file}")

if not backup.exists():
    shutil.copy2(app_file, backup)
    print("Backup created:", backup)
else:
    print("Existing backup preserved:", backup)

if not db_file.exists():
    print("Database not found at the expected path:", db_file)
    print("Stop here and check the database path before migrating.")
else:
    with sqlite3.connect(str(db_file)) as conn:
        columns = {
            row[1] for row in conn.execute("PRAGMA table_info(users)")
        }

        required = {
            "pet_hash": "BLOB",
            "school_hash": "BLOB",
            "city_hash": "BLOB",
            "failed_attempts": "INTEGER NOT NULL DEFAULT 0",
            "locked_until": "REAL NOT NULL DEFAULT 0",
        }

        for name, definition in required.items():
            if name not in columns:
                conn.execute(
                    f"ALTER TABLE users ADD COLUMN {name} {definition}"
                )
                print("Added column:", name)
            else:
                print("Already exists:", name)

    print("Migration complete. Existing users and records preserved.")


Backup created: /content/drive/MyDrive/FreightQuote_AI/app_backup_before_security.py
Added column: pet_hash
Added column: school_hash
Added column: city_hash
Added column: failed_attempts
Added column: locked_until
Migration complete. Existing users and records preserved.


In [ ]:

from pathlib import Path
import shutil
import sqlite3

project = Path("/content/drive/MyDrive/FreightQuote_AI")
app_file = project / "app.py"
db_file = project / "freightquote.db"
backup = project / "app_backup_before_security.py"

if not app_file.exists():
    raise FileNotFoundError(f"app.py not found: {app_file}")

if not backup.exists():
    shutil.copy2(app_file, backup)
    print("Backup created:", backup)
else:
    print("Existing backup preserved:", backup)

if not db_file.exists():
    print("Database not found at the expected path:", db_file)
    print("Stop here and check the database path before migrating.")
else:
    with sqlite3.connect(str(db_file)) as conn:
        columns = {
            row[1] for row in conn.execute("PRAGMA table_info(users)")
        }

        required = {
            "pet_hash": "BLOB",
            "school_hash": "BLOB",
            "city_hash": "BLOB",
            "failed_attempts": "INTEGER NOT NULL DEFAULT 0",
            "locked_until": "REAL NOT NULL DEFAULT 0",
        }

        for name, definition in required.items():
            if name not in columns:
                conn.execute(
                    f"ALTER TABLE users ADD COLUMN {name} {definition}"
                )
                print("Added column:", name)
            else:
                print("Already exists:", name)

    print("Migration complete. Existing users and records preserved.")


Existing backup preserved: /content/drive/MyDrive/FreightQuote_AI/app_backup_before_security.py
Already exists: pet_hash
Already exists: school_hash
Already exists: city_hash
Already exists: failed_attempts
Already exists: locked_until
Migration complete. Existing users and records preserved.


In [ ]:
import py_compile

app_path = "/content/drive/MyDrive/FreightQuote_AI/app.py"

py_compile.compile(app_path, doraise=True)
print("SUCCESS: app.py syntax is valid.")

SUCCESS: app.py syntax is valid.


In [ ]:
# 1. Install required packages
%pip install -q streamlit pyngrok bcrypt requests

import os
import sys
import time
import subprocess
import requests

from pathlib import Path
from google.colab import userdata
from pyngrok import ngrok, conf

# 2. Project path
PROJECT_DIR = Path("/content/drive/MyDrive/FreightQuote_AI")
APP_FILE = PROJECT_DIR / "app.py"
LOG_FILE = PROJECT_DIR / "logs" / "streamlit_server.log"
PORT = 8503

if not APP_FILE.exists():
    raise FileNotFoundError(f"app.py not found: {APP_FILE}")

# 3. Load secrets from Colab Secrets
NGROK_TOKEN = userdata.get("NGROK_AUTHTOKEN")
SMTP_EMAIL = userdata.get("SMTP_EMAIL")
SMTP_APP_PASSWORD = userdata.get("SMTP_APP_PASSWORD")

if not NGROK_TOKEN:
    raise ValueError(
        "Add NGROK_AUTHTOKEN in Colab Secrets and enable notebook access."
    )

# 4. Stop old local ngrok tunnels
ngrok.kill()

# 5. Prepare Streamlit environment
env = os.environ.copy()
env["NGROK_AUTHTOKEN"] = NGROK_TOKEN
env["SMTP_EMAIL"] = SMTP_EMAIL or ""
env["SMTP_APP_PASSWORD"] = SMTP_APP_PASSWORD or ""

# 6. Start Streamlit
LOG_FILE.parent.mkdir(parents=True, exist_ok=True)

log_file = open(LOG_FILE, "w")

proc = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", str(APP_FILE),
        "--server.port", str(PORT),
        "--server.address", "127.0.0.1",
        "--server.headless", "true",
        "--browser.gatherUsageStats", "false",
    ],
    cwd=str(PROJECT_DIR),
    env=env,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

# 7. Wait for Streamlit health check
health_url = f"http://127.0.0.1:{PORT}/_stcore/health"

for attempt in range(45):
    if proc.poll() is not None:
        log_file.flush()
        print(LOG_FILE.read_text(errors="replace")[-5000:])
        raise RuntimeError("Streamlit stopped. Check the log above.")

    try:
        response = requests.get(health_url, timeout=2)
        if response.ok:
            print("Streamlit is running.")
            break
    except requests.RequestException:
        pass

    time.sleep(1)
else:
    raise TimeoutError("Streamlit did not start within 45 seconds.")

# 8. Configure ngrok and create the public tunnel
conf.get_default().auth_token = NGROK_TOKEN

# Pass the port number, avoiding the malformed URL issue
tunnel = ngrok.connect(addr=PORT, proto="http")
PUBLIC_URL = tunnel.public_url

print("\n" + "=" * 50)
print("FreightQuote AI is ready!")
print("Public URL:", PUBLIC_URL)
print("Health check: PASS")
print("=" * 50)

Streamlit is running.

FreightQuote AI is ready!
Public URL: https://reptile-unspoken-tuition.ngrok-free.dev
Health check: PASS


In [ ]:
from pathlib import Path
import subprocess

PROJECT_DIR = Path("/content/drive/MyDrive/FreightQuote_AI")
REPO_URL = (
    "https://github.com/mdshahabuddin-ml/"
    "Maritime_Freightquote_Route_Optimization.git"
)

print("Project folder exists:", PROJECT_DIR.exists())
print("Git installed:", subprocess.run(
    ["git", "--version"], capture_output=True, text=True
).stdout.strip())

if PROJECT_DIR.exists():
    print("\nProject files:")
    for item in list(PROJECT_DIR.iterdir())[:30]:
        print(item.name)

    result = subprocess.run(
        ["git", "status", "--short", "--branch"],
        cwd=PROJECT_DIR,
        capture_output=True,
        text=True
    )
    print("\nGit status:")
    print(result.stdout or result.stderr)

In [ ]:
from pathlib import Path
import subprocess

PROJECT_DIR = Path("/content/drive/MyDrive/FreightQuote_AI")

print("Project folder exists:", PROJECT_DIR.exists())

if PROJECT_DIR.exists():
    print("\nProject files:")
    for item in list(PROJECT_DIR.iterdir())[:30]:
        print(item.name)

    status = subprocess.run(
        ["git", "status", "--short", "--branch"],
        cwd=str(PROJECT_DIR),
        capture_output=True,
        text=True
    )
    print("\nGit status:")
    print(status.stdout or status.stderr)

    remote = subprocess.run(
        ["git", "remote", "-v"],
        cwd=str(PROJECT_DIR),
        capture_output=True,
        text=True
    )
    print("\nGit remote:")
    print(remote.stdout or "No remote configured (or this is not a Git repository).")

Project folder exists: True

Project files:
requirements.txt
run_colab.py
README.md
.gitignore
FreightQuote_AI_Colab.ipynb
app.py
utils
tests
services
pages
logs
database
data
agents
freightquote.db
security_auth.py
__pycache__
app_backup_before_auth.py
app_backup_before_security.py

Git status:
fatal: not a git repository (or any parent up to mount point /content)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


Git remote:
No remote configured (or this is not a Git repository).


In [ ]:
from pathlib import Path
import subprocess

PROJECT_DIR = Path("/content/drive/MyDrive/FreightQuote_AI")

# Git repository initialize karo
result = subprocess.run(
    ["git", "init", "-b", "main"],
    cwd=str(PROJECT_DIR),
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

# Status check karo
status = subprocess.run(
    ["git", "status", "--short --branch"],
    cwd=str(PROJECT_DIR),
    capture_output=True,
    text=True
)

print(status.stdout)

Initialized empty Git repository in /content/drive/MyDrive/FreightQuote_AI/.git/





In [ ]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/FreightQuote_AI")

gitignore = PROJECT_DIR / ".gitignore"

print("----- .gitignore -----")
print(gitignore.read_text() if gitignore.exists() else "File not found")

----- .gitignore -----
*.db
logs/*.log
__pycache__/
.env



In [ ]:
from pathlib import Path
import subprocess

PROJECT_DIR = Path("/content/drive/MyDrive/FreightQuote_AI")

# Check which files Git is tracking or ignoring
for name in [
    "app.py",
    "security_auth.py",
    "app_backup_before_auth.py",
    "app_backup_before_security.py",
    "freightquote.db",
    ".env",
]:
    result = subprocess.run(
        ["git", "check-ignore", "-v", name],
        cwd=str(PROJECT_DIR),
        capture_output=True,
        text=True
    )
    print(f"\n{name}:")
    print(result.stdout.strip() or "Not ignored by Git")

# Show project status
result = subprocess.run(
    ["git", "status", "--short --branch"],
    cwd=str(PROJECT_DIR),
    capture_output=True,
    text=True
)
print("\nGit status:\n", result.stdout)


app.py:
Not ignored by Git

security_auth.py:
Not ignored by Git

app_backup_before_auth.py:
Not ignored by Git

app_backup_before_security.py:
Not ignored by Git

freightquote.db:
.gitignore:1:*.db	freightquote.db

.env:
.gitignore:4:.env	.env

Git status:
 


In [ ]:
from pathlib import Path
import subprocess

PROJECT_DIR = Path("/content/drive/MyDrive/FreightQuote_AI")
REPO_URL = "https://github.com/mdshahabuddin-ml/Maritime_Freightquote_Route_Optimization.git"

result = subprocess.run(
    ["git", "remote", "add", "origin", REPO_URL],
    cwd=str(PROJECT_DIR),
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr or "Remote command completed.")

check = subprocess.run(
    ["git", "remote", "-v"],
    cwd=str(PROJECT_DIR),
    capture_output=True,
    text=True
)

print(check.stdout)


Remote command completed.
origin	https://github.com/mdshahabuddin-ml/Maritime_Freightquote_Route_Optimization.git (fetch)
origin	https://github.com/mdshahabuddin-ml/Maritime_Freightquote_Route_Optimization.git (push)



In [ ]:
from pathlib import Path
import subprocess

PROJECT_DIR = Path("/content/drive/MyDrive/FreightQuote_AI")

# Stage normal project files, respecting .gitignore
result = subprocess.run(
    ["git", "add", "-n", "."],
    cwd=str(PROJECT_DIR),
    capture_output=True,
    text=True
)

print("Files Git would add:")
print(result.stdout or "No files listed")
print("Errors:", result.stderr or "None")

Files Git would add:
add '.gitignore'
add 'FreightQuote_AI_Colab.ipynb'
add 'README.md'
add 'agents/.gitkeep'
add 'app.py'
add 'app_backup_before_auth.py'
add 'app_backup_before_security.py'
add 'data/.gitkeep'
add 'database/.gitkeep'
add 'logs/.gitkeep'
add 'pages/.gitkeep'
add 'requirements.txt'
add 'run_colab.py'
add 'security_auth.py'
add 'services/.gitkeep'
add 'tests/.gitkeep'
add 'utils/.gitkeep'

Errors: None


In [ ]:
from pathlib import Path
import subprocess

PROJECT_DIR = Path("/content/drive/MyDrive/FreightQuote_AI")
gitignore = PROJECT_DIR / ".gitignore"

content = gitignore.read_text()
new_rules = "\n# Local backups and sensitive files\napp_backup_before_auth.py\napp_backup_before_security.py\nfreightquote.db\n.env\n"

if "app_backup_before_auth.py" not in content:
    gitignore.write_text(content.rstrip() + "\n" + new_rules)

print("Updated .gitignore:")
print(gitignore.read_text())

Updated .gitignore:
*.db
logs/*.log
__pycache__/
.env

# Local backups and sensitive files
app_backup_before_auth.py
app_backup_before_security.py
freightquote.db
.env



In [ ]:
import subprocess

PROJECT_DIR = "/content/drive/MyDrive/FreightQuote_AI"

result = subprocess.run(
    ["git", "add", "-n", "."],
    cwd=PROJECT_DIR,
    capture_output=True,
    text=True
)

print(result.stdout)
print("Errors:", result.stderr or "None")

add '.gitignore'
add 'FreightQuote_AI_Colab.ipynb'
add 'README.md'
add 'agents/.gitkeep'
add 'app.py'
add 'data/.gitkeep'
add 'database/.gitkeep'
add 'logs/.gitkeep'
add 'pages/.gitkeep'
add 'requirements.txt'
add 'run_colab.py'
add 'security_auth.py'
add 'services/.gitkeep'
add 'tests/.gitkeep'
add 'utils/.gitkeep'

Errors: None


In [ ]:
import subprocess

PROJECT_DIR = "/content/drive/MyDrive/FreightQuote_AI"

files_to_add = [
    ".gitignore",
    "README.md",
    "requirements.txt",
    "app.py",
    "security_auth.py",
    "run_colab.py",
    "FreightQuote_AI_Colab.ipynb",
    "agents/.gitkeep",
    "data/.gitkeep",
    "database/.gitkeep",
    "logs/.gitkeep",
    "pages/.gitkeep",
    "services/.gitkeep",
    "tests/.gitkeep",
    "utils/.gitkeep",
]

result = subprocess.run(
    ["git", "add", "--"] + files_to_add,
    cwd=PROJECT_DIR,
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)
print(result.stdout)
print(result.stderr or "No errors")

status = subprocess.run(
    ["git", "status", "--short"],
    cwd=PROJECT_DIR,
    capture_output=True,
    text=True
)
print("\nStaged files:\n", status.stdout)

Return code: 0

No errors

Staged files:
 A  .gitignore
A  FreightQuote_AI_Colab.ipynb
A  README.md
A  agents/.gitkeep
A  app.py
A  data/.gitkeep
A  database/.gitkeep
A  logs/.gitkeep
A  pages/.gitkeep
A  requirements.txt
A  run_colab.py
A  security_auth.py
A  services/.gitkeep
A  tests/.gitkeep
A  utils/.gitkeep



In [ ]:
from pathlib import Path
import subprocess
import re

PROJECT_DIR = Path("/content/drive/MyDrive/FreightQuote_AI")

patterns = {
    "Possible API key": re.compile(
        r"(?i)(api[_-]?key|secret[_-]?key)\s*[:=]\s*['\"][^'\"]{8,}"
    ),
    "Possible password": re.compile(
        r"(?i)(password|passwd|smtp_app_password)\s*[:=]\s*['\"][^'\"]{4,}"
    ),
    "Possible token": re.compile(
        r"(?i)(ngrok[_-]?authtoken|access[_-]?token)\s*[:=]\s*['\"][^'\"]{8,}"
    ),
}

result = subprocess.run(
    ["git", "diff", "--cached", "--name-only"],
    cwd=PROJECT_DIR,
    capture_output=True,
    text=True,
    check=True,
)

findings = []

for name in result.stdout.splitlines():
    path = PROJECT_DIR / name
    if not path.is_file() or path.suffix.lower() not in {
        ".py", ".ipynb", ".md", ".txt", ".json", ".yaml", ".yml"
    }:
        continue

    try:
        content = path.read_text(encoding="utf-8", errors="ignore")
    except OSError:
        continue

    for line_no, line in enumerate(content.splitlines(), start=1):
        for label, pattern in patterns.items():
            if pattern.search(line):
                findings.append((name, line_no, label))

if findings:
    print("Possible secret patterns found (values hidden):")
    for name, line_no, label in findings:
        print(f"{name}:{line_no} — {label}")
    print("\nReview these lines before committing.")
else:
    print("No common secret patterns detected.")
    print("This scan cannot guarantee that all secrets are absent.")

print("\nStaged file list:")
print(result.stdout)

No common secret patterns detected.
This scan cannot guarantee that all secrets are absent.

Staged file list:
.gitignore
FreightQuote_AI_Colab.ipynb
README.md
agents/.gitkeep
app.py
data/.gitkeep
database/.gitkeep
logs/.gitkeep
pages/.gitkeep
requirements.txt
run_colab.py
security_auth.py
services/.gitkeep
tests/.gitkeep
utils/.gitkeep



In [ ]:
import subprocess

PROJECT_DIR = "/content/drive/MyDrive/FreightQuote_AI"

result = subprocess.run(
    ["git", "ls-remote", "--heads", "origin"],
    cwd=PROJECT_DIR,
    capture_output=True,
    text=True
)

print("Remote branches:")
print(result.stdout or "No remote branches found.")
print("Error:", result.stderr or "None")

Remote branches:
8346b8ddb60cd77f74c622f2c996fd359dd50c23	refs/heads/main

Error: None


In [ ]:
import subprocess

PROJECT_DIR = "/content/drive/MyDrive/FreightQuote_AI"

# Remote changes download/check karo; local files merge nahi hongi
fetch = subprocess.run(
    ["git", "fetch", "origin"],
    cwd=PROJECT_DIR,
    capture_output=True,
    text=True
)

print("Fetch output:", fetch.stdout or "(no output)")
print("Fetch error:", fetch.stderr or "None")
print("Fetch return code:", fetch.returncode)

if fetch.returncode == 0:
    result = subprocess.run(
        ["git", "ls-tree", "--name-only", "origin/main"],
        cwd=PROJECT_DIR,
        capture_output=True,
        text=True
    )
    print("\nFiles already on GitHub:")
    print(result.stdout or "(No files listed)")

Fetch output: (no output)
Fetch error: From https://github.com/mdshahabuddin-ml/Maritime_Freightquote_Route_Optimization
 * [new branch]      main       -> origin/main

Fetch return code: 0

Files already on GitHub:
LICENSE
README.md



In [ ]:
import subprocess

PROJECT_DIR = "/content/drive/MyDrive/FreightQuote_AI"

for label, command in [
    ("LOCAL README", ["git", "show", ":README.md"]),
    ("GITHUB README", ["git", "show", "origin/main:README.md"]),
]:
    result = subprocess.run(
        command,
        cwd=PROJECT_DIR,
        capture_output=True,
        text=True
    )
    print(f"\n--- {label} ---")
    print(result.stdout[:5000] if result.returncode == 0 else result.stderr)


--- LOCAL README ---
# FreightQuote AI — Authentication Module

Streamlit app with Sign In (email **or** username), Register, and Forgot Password with a 6-digit email OTP.

## Folder
```
FreightQuote_AI/
├── app.py                     # the Streamlit app
├── run_colab.py               # starts Streamlit in background + ngrok tunnel
├── FreightQuote_AI_Colab.ipynb# clean 4-cell notebook
├── requirements.txt
├── agents/ data/ database/ logs/ pages/ services/ tests/ utils/
```

## Run in Google Colab
1. Colab Secrets (key icon, notebook access ON): `NGROK_AUTHTOKEN`, `SMTP_EMAIL`, `SMTP_APP_PASSWORD` (Gmail App Password).
2. Open `FreightQuote_AI_Colab.ipynb`, run cells 1 → 3, upload `FreightQuote_AI.zip` when asked.
3. Open the printed URL.

## Run locally
```
pip install -r requirements.txt
set SMTP_EMAIL=you@gmail.com            (Linux/mac: export ...)
set SMTP_APP_PASSWORD=xxxx xxxx xxxx xxxx
streamlit run app.py
```

## What was fixed
- Streamlit now starts in the **background** (ol

In [ ]:
import subprocess
from pathlib import Path

PROJECT_DIR = "/content/drive/MyDrive/FreightQuote_AI"

# Unstage README.md safely using 'git rm --cached' which works even when there are no commits yet
subprocess.run(
    ["git", "rm", "--cached", "--ignore-unmatch", "README.md"],
    cwd=PROJECT_DIR,
    check=True
)

# Ensure the separate project documentation copy exists
root = Path(PROJECT_DIR)
new_readme = root / "README_FreightQuote_AI.md"
if not new_readme.exists() and (root / "README.md").exists():
    new_readme.write_text((root / "README.md").read_text(encoding="utf-8"), encoding="utf-8")

# Stage the separate documentation file
subprocess.run(
    ["git", "add", "--", "README_FreightQuote_AI.md"],
    cwd=PROJECT_DIR,
    check=True
)

# Review staged files
result = subprocess.run(
    ["git", "status", "--short"],
    cwd=PROJECT_DIR,
    capture_output=True,
    text=True
)
print(result.stdout)

A  .gitignore
A  FreightQuote_AI_Colab.ipynb
A  README_FreightQuote_AI.md
A  agents/.gitkeep
AM app.py
A  data/.gitkeep
A  database/.gitkeep
A  logs/.gitkeep
A  pages/.gitkeep
A  requirements.txt
A  run_colab.py
A  security_auth.py
A  services/.gitkeep
A  tests/.gitkeep
A  utils/.gitkeep
?? .streamlit/
?? README.md
?? app.py.before_customer_button.bak
?? app.py.before_rbac_phase1.bak
?? app.py.before_role_buttons.bak
?? app.py.pre_dashboard_backup
?? dashboard_views.py
?? dashboard_views.py.before_quote_views.bak
?? dashboard_views.py.phase1_customer_backup
?? dashboard_views.py.pre_quotes_backup
?? dashboard_views.py.pre_visible_quote_button
?? fq_ui.py
?? quote_service.py
?? quote_ui.py
?? quote_ui.py.before_quote_views.bak
?? ui_upgrade/



In [ ]:
import subprocess

PROJECT_DIR = "/content/drive/MyDrive/FreightQuote_AI"

for command in [
    ["git", "status", "--short", "--branch"],
    ["git", "diff", "--cached", "--name-status"],
    ["git", "log", "-1", "--oneline"],
]:
    result = subprocess.run(
        command,
        cwd=PROJECT_DIR,
        capture_output=True,
        text=True
    )
    print("\n$", " ".join(command))
    print(result.stdout or result.stderr or "(no output)")


$ git status --short --branch
## No commits yet on main
A  .gitignore
A  FreightQuote_AI_Colab.ipynb
A  README.md
A  agents/.gitkeep
A  app.py
A  data/.gitkeep
A  database/.gitkeep
A  logs/.gitkeep
A  pages/.gitkeep
A  requirements.txt
A  run_colab.py
A  security_auth.py
A  services/.gitkeep
A  tests/.gitkeep
A  utils/.gitkeep


$ git diff --cached --name-status
A	.gitignore
A	FreightQuote_AI_Colab.ipynb
A	README.md
A	agents/.gitkeep
A	app.py
A	data/.gitkeep
A	database/.gitkeep
A	logs/.gitkeep
A	pages/.gitkeep
A	requirements.txt
A	run_colab.py
A	security_auth.py
A	services/.gitkeep
A	tests/.gitkeep
A	utils/.gitkeep


$ git log -1 --oneline
fatal: your current branch 'main' does not have any commits yet



In [ ]:
import subprocess

PROJECT_DIR = "/content/drive/MyDrive/FreightQuote_AI"

commands = [
    ["git", "status", "--short", "--branch"],
    ["git", "diff", "--cached", "--name-status"],
    ["git", "log", "-1", "--oneline"],
]

for cmd in commands:
    result = subprocess.run(
        cmd,
        cwd=PROJECT_DIR,
        capture_output=True,
        text=True
    )
    print("\n$", " ".join(cmd))
    print(result.stdout or result.stderr or "(no output)")


$ git status --short --branch
## No commits yet on main
A  .gitignore
A  FreightQuote_AI_Colab.ipynb
A  README.md
A  agents/.gitkeep
A  app.py
A  data/.gitkeep
A  database/.gitkeep
A  logs/.gitkeep
A  pages/.gitkeep
A  requirements.txt
A  run_colab.py
A  security_auth.py
A  services/.gitkeep
A  tests/.gitkeep
A  utils/.gitkeep


$ git diff --cached --name-status
A	.gitignore
A	FreightQuote_AI_Colab.ipynb
A	README.md
A	agents/.gitkeep
A	app.py
A	data/.gitkeep
A	database/.gitkeep
A	logs/.gitkeep
A	pages/.gitkeep
A	requirements.txt
A	run_colab.py
A	security_auth.py
A	services/.gitkeep
A	tests/.gitkeep
A	utils/.gitkeep


$ git log -1 --oneline
fatal: your current branch 'main' does not have any commits yet



In [ ]:
import os

PROJECT_DIR = "/content/drive/MyDrive/FreightQuote_AI"

print("Project exists:", os.path.exists(PROJECT_DIR))
print("Files:", os.listdir(PROJECT_DIR)[:20]
      if os.path.exists(PROJECT_DIR) else "Folder not found")

Project exists: True
Files: ['requirements.txt', 'run_colab.py', 'README.md', '.gitignore', 'FreightQuote_AI_Colab.ipynb', 'app.py', 'utils', 'tests', 'services', 'pages', 'logs', 'database', 'data', 'agents', 'freightquote.db', 'security_auth.py', '__pycache__', 'app_backup_before_auth.py', 'app_backup_before_security.py', '.git']


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from pathlib import Path
import subprocess

PROJECT_DIR = Path("/content/drive/MyDrive/FreightQuote_AI")
REPO_URL = "https://github.com/mdshahabuddin-ml/FreightQuote_AI.git"

def run_git(args):
    result = subprocess.run(
        ["git"] + args,
        cwd=PROJECT_DIR,
        capture_output=True,
        text=True
    )
    print(f"\n$ git {' '.join(args)}")
    print(result.stdout or "")
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError("Git command failed. Stop here and share the error.")
    return result.stdout.strip()

# Confirm the project directory
assert PROJECT_DIR.is_dir(), f"Project folder not found: {PROJECT_DIR}"

# Point origin to the renamed repository
run_git(["remote", "set-url", "origin", REPO_URL])

# Preserve the local README separately; do not replace GitHub's README
local_readme = PROJECT_DIR / "README.md"
separate_readme = PROJECT_DIR / "README_FreightQuote_AI.md"

if local_readme.exists() and not separate_readme.exists():
    separate_readme.write_text(
        local_readme.read_text(encoding="utf-8"),
        encoding="utf-8"
    )

# Remove README.md from the index only; keep the local file on Drive
run_git(["rm", "--cached", "--ignore-unmatch", "--", "README.md"])

# Stage project files, excluding the original README and local secrets/backups
files = [
    ".gitignore",
    "FreightQuote_AI_Colab.ipynb",
    "README_FreightQuote_AI.md",
    "agents/.gitkeep",
    "app.py",
    "data/.gitkeep",
    "database/.gitkeep",
    "logs/.gitkeep",
    "pages/.gitkeep",
    "requirements.txt",
    "run_colab.py",
    "security_auth.py",
    "services/.gitkeep",
    "tests/.gitkeep",
    "utils/.gitkeep",
]

run_git(["add", "--"] + files)

# Check Git identity before committing
name = subprocess.run(
    ["git", "config", "user.name"],
    cwd=PROJECT_DIR, capture_output=True, text=True
).stdout.strip()

email = subprocess.run(
    ["git", "config", "user.email"],
    cwd=PROJECT_DIR, capture_output=True, text=True
).stdout.strip()

if not name or not email:
    print("\nGit author identity is missing.")
    print("Run these two commands in a separate cell, using your own details:")
    print('!git -C "/content/drive/MyDrive/FreightQuote_AI" config user.name "Your Name"')
    print('!git -C "/content/drive/MyDrive/FreightQuote_AI" config user.email "YOUR_GITHUB_EMAIL"')
else:
    run_git(["status", "--short"])
    run_git(["commit", "-m", "Add FreightQuote AI project"])


$ git remote set-url origin https://github.com/mdshahabuddin-ml/FreightQuote_AI.git


$ git rm --cached --ignore-unmatch -- README.md
rm 'README.md'


$ git add -- .gitignore FreightQuote_AI_Colab.ipynb README_FreightQuote_AI.md agents/.gitkeep app.py data/.gitkeep database/.gitkeep logs/.gitkeep pages/.gitkeep requirements.txt run_colab.py security_auth.py services/.gitkeep tests/.gitkeep utils/.gitkeep


Git author identity is missing.
Run these two commands in a separate cell, using your own details:
!git -C "/content/drive/MyDrive/FreightQuote_AI" config user.name "Your Name"
!git -C "/content/drive/MyDrive/FreightQuote_AI" config user.email "YOUR_GITHUB_EMAIL"


In [ ]:
from pathlib import Path
import ast

PROJECT_DIR = Path("/content/drive/MyDrive/FreightQuote_AI")

if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        f"Project folder not found: {PROJECT_DIR}"
    )

print("PROJECT:", PROJECT_DIR)
print("\n=== EXISTING PROJECT FILES ===")

for path in sorted(PROJECT_DIR.rglob("*")):
    if not path.is_file():
        continue

    # Avoid listing secrets, database contents, backups and Git internals
    if any(part in {".git", "__pycache__"} for part in path.parts):
        continue
    if path.suffix.lower() in {".db", ".sqlite", ".sqlite3", ".pyc"}:
        continue
    if path.name.startswith(".env"):
        continue
    if "backup" in path.name.lower():
        continue

    print(path.relative_to(PROJECT_DIR))

print("\n=== PYTHON FILES: FUNCTIONS AND CLASSES ===")

for path in sorted(PROJECT_DIR.rglob("*.py")):
    if any(part in {".git", "__pycache__"} for part in path.parts):
        continue
    if "backup" in path.name.lower():
        continue

    try:
        source = path.read_text(encoding="utf-8")
        tree = ast.parse(source)
        items = []

        for node in ast.walk(tree):
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
                items.append(f"function: {node.name}")
            elif isinstance(node, ast.ClassDef):
                items.append(f"class: {node.name}")

        print(f"\n{path.relative_to(PROJECT_DIR)}")
        print("  " + (", ".join(sorted(set(items))) or "No functions/classes found"))

    except Exception as exc:
        print(f"\n{path.relative_to(PROJECT_DIR)}: could not inspect ({type(exc).__name__})")

print("\n=== AUDIT COMPLETE: NO FILES MODIFIED ===")

PROJECT: /content/drive/MyDrive/FreightQuote_AI

=== EXISTING PROJECT FILES ===
.gitignore
FreightQuote_AI_Colab.ipynb
README.md
README_FreightQuote_AI.md
agents/.gitkeep
app.py
data/.gitkeep
database/.gitkeep
logs/.gitkeep
logs/streamlit_server.log
pages/.gitkeep
requirements.txt
run_colab.py
security_auth.py
services/.gitkeep
tests/.gitkeep
utils/.gitkeep

=== PYTHON FILES: FUNCTIONS AND CLASSES ===

app.py
  function: _locked_minutes, function: _record_answer_failure, function: authenticate, function: check_secret, function: clear_login_failures, function: db, function: find_user, function: get_secret, function: hash_secret, function: init_db, function: login_lock_minutes, function: normalize_answer, function: normalize_email, function: now_utc, function: parse_time, function: record_login_failure, function: register_user, function: request_otp, function: reset_password, function: reset_with_answers, function: send_otp_email, function: valid_password, function: verify_otp

run_colab

In [ ]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/FreightQuote_AI")

FILES_TO_INSPECT = {
    "app.py": [
        "init_db",
        "find_user",
        "authenticate",
        "register_user",
        "request_otp",
        "send_otp_email",
        "verify_otp",
        "reset_password",
        "record_login_failure",
        "clear_login_failures",
    ],
    "security_auth.py": [
        "init_db",
        "login_user",
        "register_user",
        "reset_password",
    ],
    "run_colab.py": [
        "check_smtp_secrets",
        "start_streamlit",
    ],
}

for filename, function_names in FILES_TO_INSPECT.items():
    file_path = PROJECT_DIR / filename

    print(f"\n{'=' * 65}\nFILE: {filename}\n{'=' * 65}")

    if not file_path.exists():
        print("File not found.")
        continue

    lines = file_path.read_text(encoding="utf-8").splitlines()

    for function_name in function_names:
        start = next(
            (
                i for i, line in enumerate(lines)
                if line.startswith(f"def {function_name}(")
                or line.startswith(f"async def {function_name}(")
            ),
            None,
        )

        if start is None:
            print(f"\n--- {function_name}: not found ---")
            continue

        end = start + 1
        while end < len(lines):
            line = lines[end]
            if line and not line[0].isspace() and (
                line.startswith("def ")
                or line.startswith("async def ")
                or line.startswith("class ")
            ):
                break
            end += 1

        print(f"\n--- {function_name} (line {start + 1}) ---")
        print("\n".join(lines[start:end]))

print("\nREAD-ONLY INSPECTION COMPLETE. No files were changed.")


FILE: app.py

--- init_db (line 84) ---
def init_db():
    DB_PATH.parent.mkdir(parents=True, exist_ok=True)
    with db() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS users (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                username TEXT NOT NULL UNIQUE,
                email TEXT NOT NULL UNIQUE,
                password_hash BLOB NOT NULL,
                role TEXT NOT NULL DEFAULT 'Customer',
                created_at TEXT NOT NULL,
                pet_hash BLOB,
                school_hash BLOB,
                city_hash BLOB
            )
        """)

        # Upgrade an older database that was created without the security columns.
        existing = {row["name"] for row in conn.execute("PRAGMA table_info(users)")}
        for column, _ in SECURITY_QUESTIONS:
            if column not in existing:
                conn.execute(f"ALTER TABLE users ADD COLUMN {column} BLOB")

        conn.execute("""
            CREATE TABLE IF NO

In [ ]:
from pathlib import Path
import ast

PROJECT_DIR = Path("/content/drive/MyDrive/FreightQuote_AI")
NEW_FILE = PROJECT_DIR / "dashboard_views.py"

# Safety: never overwrite an existing file
if NEW_FILE.exists():
    print(f"{NEW_FILE.name} already exists. Inspecting content instead.")
else:
    code = r'''
import streamlit as st

ROLE_FEATURES = {
    "Admin": [
        "User and role administration",
        "System health and configuration",
        "Audit trail review",
        "Agent and RAG configuration",
    ],
    "Operations Manager": [
        "Shipment and route operations",
        "Operational alerts",
        "Agent activity overview",
        "Performance reports",
    ],
    "Freight Broker": [
        "Freight quote workspace",
        "Route and shipment information",
        "Customer quote requests",
        "Quote history",
    ],
    "Auditor": [
        "Read-only audit review",
        "Authentication event review",
        "Policy and decision citations",
        "Compliance reports",
    ],
    "Customer": [
        "Request a freight quote",
        "View quote status",
        "Review shipment information",
        "Manage customer requests",
    ],
}

def render_dashboard(user):
    """Render a role-specific dashboard layout for an authenticated user."""
    if not user:
        st.error("Please sign in to view your dashboard.")
        st.stop()

    username = user.get("username", "User")
    role = user.get("role", "Customer")

    if role not in ROLE_FEATURES:
        st.error("Your account has an unrecognized role. Contact an administrator.")
        st.stop()

    st.title("FreightQuote AI Dashboard")
    st.caption(f"Signed in as {username} · Role: {role}")
    st.divider()

    st.subheader(f"Welcome, {username}")
    st.markdown("### Your workspace")

    columns = st.columns(2)
    for index, feature in enumerate(ROLE_FEATURES[role]):
        with columns[index % 2]:
            st.container(border=True).markdown(
                f"**{feature}**\n\n"
                "Workspace module — integration pending."
            )

    st.divider()
    st.markdown("### Advanced AI capabilities")

    ai_columns = st.columns(2)
    with ai_columns[0]:
        st.markdown("**Live Agents & Digital Twin**")
        st.info(
            "The interface will be connected to the existing agent "
            "and simulator services after they are audited."
        )
    with ai_columns[1]:
        st.markdown("**Hybrid RAG & Citations**")
        st.info(
            "Policy retrieval, live-data sources and citation display "
            "will be connected after their implementations are verified."
        )

    st.caption(
        "This is a dashboard UI scaffold. "
        "It does not yet execute operational actions or enforce backend permissions."
    )
'''
    # Validate and write
    ast.parse(code)
    NEW_FILE.write_text(code.lstrip(), encoding="utf-8")
    print("Created new file:", NEW_FILE)


dashboard_views.py already exists. Inspecting content instead.


In [ ]:
from pathlib import Path
import ast

PROJECT_DIR = Path("/content/drive/MyDrive/FreightQuote_AI")
NEW_FILE = PROJECT_DIR / "dashboard_views.py"

if not PROJECT_DIR.exists():
    raise FileNotFoundError(f"Project folder not found: {PROJECT_DIR}")

if NEW_FILE.exists():
    print("File already exists. No changes made:", NEW_FILE)
else:
    code = '''
import streamlit as st

ROLE_FEATURES = {
    "Admin": [
        "User and role administration",
        "System health and configuration",
        "Audit trail review",
        "Agent and RAG configuration",
    ],
    "Operations Manager": [
        "Shipment and route operations",
        "Operational alerts",
        "Agent activity overview",
        "Performance reports",
    ],
    "Freight Broker": [
        "Freight quote workspace",
        "Route and shipment information",
        "Customer quote requests",
        "Quote history",
    ],
    "Auditor": [
        "Read-only audit review",
        "Authentication event review",
        "Policy and decision citations",
        "Compliance reports",
    ],
    "Customer": [
        "Request a freight quote",
        "View quote status",
        "Review shipment information",
        "Manage customer requests",
    ],
}

def render_dashboard(user):
    if not user:
        st.error("Please sign in to view your dashboard.")
        return

    username = user.get("username", "User")
    role = user.get("role", "Customer")

    if role not in ROLE_FEATURES:
        st.error("Unrecognized role. Contact an administrator.")
        return

    st.title("FreightQuote AI Dashboard")
    st.caption(f"Signed in as {username} | Role: {role}")
    st.divider()

    st.subheader(f"Welcome, {username}")
    st.markdown("### Your workspace")

    columns = st.columns(2)
    for index, feature in enumerate(ROLE_FEATURES[role]):
        with columns[index % 2]:
            with st.container(border=True):
                st.markdown(f"**{feature}**")
                st.caption("Integration pending")

    st.divider()
    st.subheader("Advanced AI Capabilities")

    st.markdown("**Live Agents and Digital Twin**")
    st.info("Integration will follow an audit of existing agent services.")

    st.markdown("**Hybrid RAG and Citations**")
    st.info("Integration will follow an audit of existing retrieval services.")

    st.caption(
        "Dashboard UI scaffold only. Operational actions and backend "
        "authorization are not implemented in this module."
    )
'''

    # Validate before writing
    ast.parse(code)

    # Create only the new file
    NEW_FILE.write_text(code.lstrip(), encoding="utf-8")

    print("Created:", NEW_FILE)

# Verify syntax without importing or running the app
if NEW_FILE.exists():
    ast.parse(NEW_FILE.read_text(encoding="utf-8"))
    print("PASS: dashboard_views.py syntax is valid.")

print("Existing app.py and database were not modified.")

Created: /content/drive/MyDrive/FreightQuote_AI/dashboard_views.py
PASS: dashboard_views.py syntax is valid.
Existing app.py and database were not modified.


In [ ]:
from pathlib import Path
import ast

PROJECT_DIR = Path("/content/drive/MyDrive/FreightQuote_AI")
APP_FILE = PROJECT_DIR / "app.py"
DASHBOARD_FILE = PROJECT_DIR / "dashboard_views.py"
BACKUP_FILE = PROJECT_DIR / "app.py.pre_dashboard_backup"

assert APP_FILE.exists(), "app.py not found"
assert DASHBOARD_FILE.exists(), "dashboard_views.py not found"

app_source = APP_FILE.read_text(encoding="utf-8")
dashboard_source = DASHBOARD_FILE.read_text(encoding="utf-8")

# Validate both files without executing the app
ast.parse(app_source)
ast.parse(dashboard_source)

print("=== CURRENT LOGIN / DASHBOARD SECTION ===")
lines = app_source.splitlines()

for i, line in enumerate(lines):
    if 'if ss.user:' in line:
        start = max(0, i - 5)
        end = min(len(lines), i + 20)
        for number in range(start, end):
            print(f"{number + 1:4}: {lines[number]}")
        break
else:
    print("Could not find the expected signed-in section.")

print("\n=== BACKUP STATUS ===")
if BACKUP_FILE.exists():
    print("Backup already exists:", BACKUP_FILE)
    print("It will not be overwritten.")
else:
    BACKUP_FILE.write_text(app_source, encoding="utf-8")
    print("Backup created:", BACKUP_FILE)

print("\n=== VERIFICATION ===")
print("app.py syntax: PASS")
print("dashboard_views.py syntax: PASS")
print("No application code was changed.")

=== CURRENT LOGIN / DASHBOARD SECTION ===
 570: if ss.flash:
 571:     st.success(ss.flash)
 572:     ss.flash = None
 573: 
 574: # ---- Signed-in view ----
 575: if ss.user:
 576:     st.subheader(f"Welcome, {ss.user['username']}")
 577:     st.caption(f"{ss.user['email']} · Role: {ss.user['role']}")
 578:     st.info("Authentication complete. FreightQuote features will be connected here.")
 579:     if st.button("Sign Out", use_container_width=True):
 580:         ss.user = None
 581:         st.rerun()
 582:     st.stop()
 583: 
 584: # Widget state can only be changed before the widget is drawn.
 585: if ss.get("next_mode"):
 586:     ss.mode = ss.pop("next_mode")
 587: 
 588: mode = st.radio(
 589:     "Authentication",
 590:     ["Sign In", "Register", "Forgot Password"],
 591:     horizontal=True,
 592:     label_visibility="collapsed",
 593:     key="mode",
 594: )

=== BACKUP STATUS ===
Backup created: /content/drive/MyDrive/FreightQuote_AI/app.py.pre_dashboard_backup

=== VE

In [ ]:
from pathlib import Path
import ast

PROJECT_DIR = Path("/content/drive/MyDrive/FreightQuote_AI")
APP_FILE = PROJECT_DIR / "app.py"
BACKUP_FILE = PROJECT_DIR / "app.py.pre_dashboard_backup"
DASHBOARD_FILE = PROJECT_DIR / "dashboard_views.py"

# Safety checks
assert APP_FILE.exists(), "app.py not found"
assert BACKUP_FILE.exists(), "Backup missing. Stopping."
assert DASHBOARD_FILE.exists(), "dashboard_views.py missing"

source = APP_FILE.read_text(encoding="utf-8")
backup = BACKUP_FILE.read_text(encoding="utf-8")

# Ensure backup is the exact pre-integration version
if source != backup:
    raise RuntimeError(
        "app.py differs from its backup. No changes made; inspect first."
    )

old_section = '''# ---- Signed-in view ----
if ss.user:
    st.subheader(f"Welcome, {ss.user['username']}")
    st.caption(f"{ss.user['email']} · Role: {ss.user['role']}")
    st.info("Authentication complete. FreightQuote features will be connected here.")
    if st.button("Sign Out", use_container_width=True):
        ss.user = None
        st.rerun()
    st.stop()
'''

new_section = '''# ---- Signed-in view ----
if ss.user:
    from dashboard_views import render_dashboard

    render_dashboard(ss.user)

    if st.button("Sign Out", use_container_width=True):
        ss.user = None
        st.rerun()

    st.stop()
'''

if source.count(old_section) != 1:
    raise RuntimeError(
        "Expected signed-in section was not found exactly once. "
        "No changes made."
    )

updated = source.replace(old_section, new_section, 1)

# Validate the updated source before writing
ast.parse(updated)

# Write to a temporary file, then replace app.py
temp_file = PROJECT_DIR / "app.py.dashboard_tmp"
temp_file.write_text(updated, encoding="utf-8")

try:
    ast.parse(temp_file.read_text(encoding="utf-8"))
    APP_FILE.write_text(temp_file.read_text(encoding="utf-8"), encoding="utf-8")
finally:
    if temp_file.exists():
        temp_file.unlink()

# Verify
ast.parse(APP_FILE.read_text(encoding="utf-8"))

print("SUCCESS: Dashboard connected to the signed-in view.")
print("PASS: app.py syntax is valid.")
print("Backup preserved:", BACKUP_FILE)
print("Authentication functions and database schema were not intentionally changed.")

SUCCESS: Dashboard connected to the signed-in view.
PASS: app.py syntax is valid.
Backup preserved: /content/drive/MyDrive/FreightQuote_AI/app.py.pre_dashboard_backup
Authentication functions and database schema were not intentionally changed.


In [ ]:
from pathlib import Path
import ast

PROJECT_DIR = Path("/content/drive/MyDrive/FreightQuote_AI")

print("=== EXISTING QUOTE / SHIPMENT REFERENCES ===")

for filename in ["app.py", "dashboard_views.py", "security_auth.py"]:
    path = PROJECT_DIR / filename
    if not path.exists():
        continue

    source = path.read_text(encoding="utf-8")
    ast.parse(source)

    print(f"\n--- {filename} ---")
    keywords = (
        "quote", "shipment", "container", "origin", "destination",
        "CREATE TABLE", "sqlite3.connect", "DB_PATH"
    )

    for line_no, line in enumerate(source.splitlines(), 1):
        if any(word.lower() in line.lower() for word in keywords):
            print(f"{line_no}: {line.strip()[:180]}")

print("\nRead-only audit complete. No files modified.")

=== EXISTING QUOTE / SHIPMENT REFERENCES ===

--- app.py ---
2: FreightQuote AI - Authentication module
33: DB_PATH = Path(os.environ.get("FREIGHTQUOTE_DB", BASE_DIR / "freightquote.db"))
62: page_title="FreightQuote AI",
74: conn = sqlite3.connect(str(DB_PATH), timeout=10)
85: DB_PATH.parent.mkdir(parents=True, exist_ok=True)
88: CREATE TABLE IF NOT EXISTS users (
108: CREATE TABLE IF NOT EXISTS password_resets (
118: CREATE TABLE IF NOT EXISTS login_attempts (
125: CREATE TABLE IF NOT EXISTS answer_attempts (
195: message["Subject"] = "FreightQuote AI Password Reset OTP"
199: f"Your FreightQuote AI password reset code is: {otp}\n\n"
548: .block-container {max-width: 650px; padding-top: 2rem;}
561: <h1>🚢 FreightQuote AI</h1>
580: if st.button("Sign Out", use_container_width=True):
605: submit = st.form_submit_button("Sign In", type="primary", use_container_width=True)
629: submit = st.form_submit_button("Create Account", type="primary", use_container_width=True)
661: use_container_wid

In [ ]:
from pathlib import Path
import ast

PROJECT_DIR = Path("/content/drive/MyDrive/FreightQuote_AI")
NEW_FILE = PROJECT_DIR / "quote_service.py"

if not PROJECT_DIR.exists():
    raise FileNotFoundError(f"Project folder not found: {PROJECT_DIR}")

if NEW_FILE.exists():
    raise FileExistsError(
        "quote_service.py already exists. No files changed; inspect it first."
    )

code = r'''
import os
import sqlite3
import uuid
from pathlib import Path
from datetime import datetime, timezone

BASE_DIR = Path(__file__).resolve().parent
DB_PATH = Path(
    os.environ.get("FREIGHTQUOTE_DB", BASE_DIR / "freightquote.db")
)

VALID_CARGO_TYPES = {
    "General cargo",
    "Refrigerated cargo",
    "Hazardous cargo",
    "Bulk cargo",
    "Other",
}

VALID_CONTAINER_TYPES = {
    "20 ft",
    "40 ft",
    "40 ft High Cube",
    "Refrigerated",
    "Not sure",
}


def connect_db():
    DB_PATH.parent.mkdir(parents=True, exist_ok=True)
    conn = sqlite3.connect(str(DB_PATH), timeout=10)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON")
    return conn


def init_quotes_table():
    """Create the quote table only when explicitly called."""
    with connect_db() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS freight_quotes (
                id TEXT PRIMARY KEY,
                user_id INTEGER NOT NULL,
                username TEXT NOT NULL,
                origin_port TEXT NOT NULL,
                destination_port TEXT NOT NULL,
                cargo_type TEXT NOT NULL,
                container_type TEXT NOT NULL,
                quantity INTEGER NOT NULL,
                shipment_date TEXT NOT NULL,
                notes TEXT NOT NULL DEFAULT '',
                status TEXT NOT NULL DEFAULT 'Pending',
                created_at TEXT NOT NULL
            )
        """)


def create_quote(
    user,
    origin_port,
    destination_port,
    cargo_type,
    container_type,
    quantity,
    shipment_date,
    notes="",
):
    """Store a quote request for the currently authenticated user."""
    if not user or not user.get("id") or not user.get("username"):
        raise ValueError("An authenticated user is required.")

    origin_port = (origin_port or "").strip()
    destination_port = (destination_port or "").strip()
    notes = (notes or "").strip()

    if not origin_port or not destination_port:
        raise ValueError("Origin and destination ports are required.")

    if origin_port.casefold() == destination_port.casefold():
        raise ValueError("Origin and destination must be different.")

    if cargo_type not in VALID_CARGO_TYPES:
        raise ValueError("Invalid cargo type.")

    if container_type not in VALID_CONTAINER_TYPES:
        raise ValueError("Invalid container type.")

    if not isinstance(quantity, int) or isinstance(quantity, bool):
        raise ValueError("Quantity must be a whole number.")

    if not 1 <= quantity <= 10000:
        raise ValueError("Quantity must be between 1 and 10000.")

    if not shipment_date:
        raise ValueError("Shipment date is required.")

    quote_id = str(uuid.uuid4())
    created_at = datetime.now(timezone.utc).isoformat()

    with connect_db() as conn:
        conn.execute("""
            INSERT INTO freight_quotes (
                id, user_id, username, origin_port, destination_port,
                cargo_type, container_type, quantity, shipment_date,
                notes, status, created_at
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, 'Pending', ?)
        """, (
            quote_id,
            int(user["id"]),
            str(user["username"]),
            origin_port,
            destination_port,
            cargo_type,
            container_type,
            quantity,
            str(shipment_date),
            notes,
            created_at,
        ))

    return quote_id


def list_my_quotes(user):
    """Return only quote requests belonging to the authenticated user."""
    if not user or not user.get("id"):
        raise ValueError("An authenticated user is required.")

    with connect_db() as conn:
        rows = conn.execute("""
            SELECT id, origin_port, destination_port, cargo_type,
                   container_type, quantity, shipment_date, notes,
                   status, created_at
            FROM freight_quotes
            WHERE user_id = ?
            ORDER BY created_at DESC
        """, (int(user["id"]),)).fetchall()

    return [dict(row) for row in rows]
'''

# Syntax validation before creating the new file
ast.parse(code)

NEW_FILE.write_text(code.lstrip(), encoding="utf-8")

# Verify the saved file
ast.parse(NEW_FILE.read_text(encoding="utf-8"))

print("Created:", NEW_FILE)
print("PASS: quote_service.py syntax is valid.")
print("No database tables were created by this cell.")
print("Existing app.py and dashboard_views.py were not modified.")

Created: /content/drive/MyDrive/FreightQuote_AI/quote_service.py
PASS: quote_service.py syntax is valid.
No database tables were created by this cell.
Existing app.py and dashboard_views.py were not modified.


In [ ]:
from pathlib import Path
import ast

PROJECT_DIR = Path("/content/drive/MyDrive/FreightQuote_AI")
NEW_FILE = PROJECT_DIR / "quote_ui.py"

if NEW_FILE.exists():
    raise FileExistsError(
        "quote_ui.py already exists. No files were changed."
    )

code = r'''
import streamlit as st
from datetime import date

from quote_service import (
    init_quotes_table,
    create_quote,
    list_my_quotes,
    VALID_CARGO_TYPES,
    VALID_CONTAINER_TYPES,
)


def render_customer_quotes(user):
    if not user or not user.get("id"):
        st.error("Please sign in first.")
        return

    init_quotes_table()

    st.header("Request a Freight Quote")
    st.caption("Enter your shipment details to submit a quote request.")

    with st.form("customer_freight_quote_form"):
        col1, col2 = st.columns(2)

        with col1:
            origin = st.text_input(
                "Origin port",
                placeholder="e.g. Mumbai",
                max_chars=100,
            )
            cargo = st.selectbox(
                "Cargo type",
                sorted(VALID_CARGO_TYPES),
            )
            quantity = st.number_input(
                "Number of containers / units",
                min_value=1,
                max_value=10000,
                value=1,
                step=1,
            )

        with col2:
            destination = st.text_input(
                "Destination port",
                placeholder="e.g. Singapore",
                max_chars=100,
            )
            container = st.selectbox(
                "Container type",
                sorted(VALID_CONTAINER_TYPES),
            )
            shipment_date = st.date_input(
                "Planned shipment date",
                min_value=date.today(),
                value=date.today(),
            )

        notes = st.text_area(
            "Additional shipment details (optional)",
            max_chars=1000,
        )

        submitted = st.form_submit_button(
            "Submit Freight Quote Request",
            type="primary",
            use_container_width=True,
        )

    if submitted:
        try:
            quote_id = create_quote(
                user=user,
                origin_port=origin,
                destination_port=destination,
                cargo_type=cargo,
                container_type=container,
                quantity=int(quantity),
                shipment_date=shipment_date.isoformat(),
                notes=notes,
            )
            st.success("Quote request submitted successfully.")
            st.code(quote_id, language=None)
            st.info(
                "Status: Pending. This is a request record, "
                "not a confirmed price or booking."
            )
        except (ValueError, TypeError) as exc:
            st.error(str(exc))
        except Exception as exc:
            # Do not display database paths or internal exception details.
            st.error(
                "The quote could not be saved. "
                "Check the application logs for details."
            )
            print("Quote submission error:", type(exc).__name__)

    st.divider()
    st.subheader("My Quote Requests")

    try:
        quotes = list_my_quotes(user)

        if not quotes:
            st.info("You have not submitted any quote requests yet.")
        else:
            for quote in quotes:
                with st.container(border=True):
                    st.markdown(
                        f"**{quote['origin_port']} → "
                        f"{quote['destination_port']}**"
                    )
                    st.write("Request ID:", quote["id"])
                    st.write("Status:", quote["status"])
                    st.write("Cargo:", quote["cargo_type"])
                    st.write("Container:", quote["container_type"])
                    st.write("Quantity:", quote["quantity"])
                    st.write("Shipment date:", quote["shipment_date"])
                    if quote["notes"]:
                        st.write("Notes:", quote["notes"])

    except Exception as exc:
        st.error("Could not load quote requests. Check the application logs.")
        print("Quote retrieval error:", type(exc).__name__)
'''

ast.parse(code)
NEW_FILE.write_text(code.lstrip(), encoding="utf-8")
ast.parse(NEW_FILE.read_text(encoding="utf-8"))

print("Created:", NEW_FILE)
print("PASS: quote_ui.py syntax is valid.")
print("Existing app.py and dashboard_views.py were not modified.")
print("Database table will be created only when the UI function is called.")

Created: /content/drive/MyDrive/FreightQuote_AI/quote_ui.py
PASS: quote_ui.py syntax is valid.
Existing app.py and dashboard_views.py were not modified.
Database table will be created only when the UI function is called.


In [ ]:

from pathlib import Path
import ast

PROJECT_DIR = Path("/content/drive/MyDrive/FreightQuote_AI")
DASHBOARD = PROJECT_DIR / "dashboard_views.py"
BACKUP = PROJECT_DIR / "dashboard_views.py.pre_quotes_backup"

assert DASHBOARD.exists(), "dashboard_views.py not found"
assert (PROJECT_DIR / "quote_ui.py").exists(), "quote_ui.py not found"
assert (PROJECT_DIR / "quote_service.py").exists(), "quote_service.py not found"

source = DASHBOARD.read_text(encoding="utf-8")
ast.parse(source)

marker = '''    st.divider()
    st.subheader("Advanced AI Capabilities")
'''

addition = '''    if role == "Customer":
        from quote_ui import render_customer_quotes
        render_customer_quotes(user)

    st.divider()
    st.subheader("Advanced AI Capabilities")
'''

if source.count(marker) != 1:
    raise RuntimeError(
        "Expected dashboard section was not found exactly once. "
        "No changes made."
    )

if BACKUP.exists():
    print("Existing backup found; it will not be overwritten.")
else:
    BACKUP.write_text(source, encoding="utf-8")
    print("Backup created:", BACKUP)

updated = source.replace(marker, addition, 1)
ast.parse(updated)

temp = PROJECT_DIR / "dashboard_views.py.tmp"
try:
    temp.write_text(updated, encoding="utf-8")
    ast.parse(temp.read_text(encoding="utf-8"))
    DASHBOARD.write_text(temp.read_text(encoding="utf-8"), encoding="utf-8")
finally:
    if temp.exists():
        temp.unlink()

ast.parse(DASHBOARD.read_text(encoding="utf-8"))

print("PASS: Customer quote form connected.")
print("PASS: dashboard_views.py syntax is valid.")
print("Existing app.py and authentication code were not changed.")


Backup created: /content/drive/MyDrive/FreightQuote_AI/dashboard_views.py.pre_quotes_backup
PASS: Customer quote form connected.
PASS: dashboard_views.py syntax is valid.
Existing app.py and authentication code were not changed.


In [ ]:

from pathlib import Path
import ast

PROJECT = Path("/content/drive/MyDrive/FreightQuote_AI")

files_to_check = [
    "app.py",
    "dashboard_views.py",
    "quote_service.py",
    "quote_ui.py",
]

print("=== FreightQuote AI: Read-only integration audit ===")

for filename in files_to_check:
    path = PROJECT / filename
    print(f"\n{filename}: ", end="")

    if not path.exists():
        print("MISSING")
        continue

    try:
        source = path.read_text(encoding="utf-8")
        ast.parse(source)
        print("EXISTS | Syntax OK")

        if filename == "app.py":
            print("  Dashboard import:", "render_dashboard" in source)
        elif filename == "dashboard_views.py":
            print("  Quote UI import:", "render_customer_quotes" in source)
            print("  Customer role condition:", 'role == "Customer"' in source)
            print("  Quote section present:", "Request a Freight Quote" in source)
        elif filename == "quote_ui.py":
            print("  Form submission button:",
                  "Submit Freight Quote Request" in source)
            print("  My Quote Requests section:",
                  "My Quote Requests" in source)
        elif filename == "quote_service.py":
            print("  Quote creation function:", "def create_quote" in source)
            print("  Quote listing function:", "def list_my_quotes" in source)

    except Exception as exc:
        print(f"CHECK FAILED: {type(exc).__name__}: {exc}")

print("\nAudit complete. No files or database records were modified.")


=== FreightQuote AI: Read-only integration audit ===

app.py: EXISTS | Syntax OK
  Dashboard import: True

dashboard_views.py: EXISTS | Syntax OK
  Quote UI import: True
  Customer role condition: True
  Quote section present: False

quote_service.py: EXISTS | Syntax OK
  Quote creation function: True
  Quote listing function: True

quote_ui.py: EXISTS | Syntax OK
  Form submission button: True
  My Quote Requests section: True

Audit complete. No files or database records were modified.


In [ ]:

from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/FreightQuote_AI")

# Read-only inspection: no files or database records are changed.
checks = {
    "dashboard_views.py": [
        "def render_dashboard",
        "render_customer_quotes",
        "Advanced AI Capabilities",
    ],
    "quote_ui.py": [
        "def render_customer_quotes",
        "Request a Freight Quote",
        "Submit Freight Quote Request",
        "My Quote Requests",
    ],
    "app.py": [
        "render_dashboard(ss.user)",
        "st.stop()",
    ],
}

for filename, terms in checks.items():
    path = PROJECT / filename
    print(f"\n{'=' * 12} {filename} {'=' * 12}")

    if not path.exists():
        print("FILE MISSING")
        continue

    lines = path.read_text(encoding="utf-8").splitlines()

    for term in terms:
        matches = [
            i for i, line in enumerate(lines, start=1)
            if term in line
        ]
        print(f"\n{term}: {matches if matches else 'NOT FOUND'}")

    # Show dashboard integration and quote-rendering call sites.
    if filename in ("dashboard_views.py", "quote_ui.py"):
        for i, line in enumerate(lines, start=1):
            if any(term in line for term in (
                "def render_dashboard",
                "def render_customer_quotes",
                "render_customer_quotes(",
                "Advanced AI Capabilities",
            )):
                start = max(1, i - 3)
                end = min(len(lines), i + 5)
                print(f"\n--- Lines {start}-{end} ---")
                for n in range(start, end + 1):
                    print(f"{n}: {lines[n - 1]}")

print("\nRead-only diagnostic complete.")



============ dashboard_views.py ============

def render_dashboard: [36]

render_customer_quotes: [63, 64]

Advanced AI Capabilities: [67]

--- Lines 33-41 ---
33:     ],
34: }
35: 
36: def render_dashboard(user):
37:     if not user:
38:         st.error("Please sign in to view your dashboard.")
39:         return
40: 
41:     username = user.get("username", "User")

--- Lines 61-69 ---
61: 
62:     if role == "Customer":
63:         from quote_ui import render_customer_quotes
64:         render_customer_quotes(user)
65: 
66:     st.divider()
67:     st.subheader("Advanced AI Capabilities")
68: 
69:     st.markdown("**Live Agents and Digital Twin**")

--- Lines 64-72 ---
64:         render_customer_quotes(user)
65: 
66:     st.divider()
67:     st.subheader("Advanced AI Capabilities")
68: 
69:     st.markdown("**Live Agents and Digital Twin**")
70:     st.info("Integration will follow an audit of existing agent services.")
71: 
72:     st.markdown("**Hybrid RAG and Citations**")

===

In [ ]:

from pathlib import Path
import ast
import os
import sys

PROJECT = Path("/content/drive/MyDrive/FreightQuote_AI")

print("=== 1. Current notebook directory ===")
print(os.getcwd())

print("\n=== 2. Streamlit process information ===")
try:
    import psutil

    found = False
    for proc in psutil.process_iter(["pid", "cmdline", "cwd"]):
        try:
            cmd = proc.info.get("cmdline") or []
            if any("streamlit" in str(part).lower() for part in cmd):
                found = True
                print("PID:", proc.info.get("pid"))
                print("Command:", " ".join(cmd))
                print("Working directory:", proc.info.get("cwd"))
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass

    if not found:
        print("No visible Streamlit process found.")
except ImportError:
    print("psutil unavailable; process inspection skipped.")

print("\n=== 3. Quote UI function body ===")
path = PROJECT / "quote_ui.py"
tree = ast.parse(path.read_text(encoding="utf-8"))

for node in ast.walk(tree):
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) \
            and node.name == "render_customer_quotes":
        print(ast.get_source_segment(
            path.read_text(encoding="utf-8"), node
        ))
        break

print("\n=== 4. Quote UI import test ===")
sys.path.insert(0, str(PROJECT))
try:
    import quote_ui
    print("Import OK:", quote_ui.__file__)
    print("Render function callable:",
          callable(quote_ui.render_customer_quotes))
except Exception as exc:
    print("Import failed:", type(exc).__name__, str(exc))

print("\nDiagnostic complete. No files or database records modified.")


=== 1. Current notebook directory ===
/content/drive/MyDrive/FreightQuote_AI

=== 2. Streamlit process information ===
PID: 3371
Command: /usr/bin/python3 -m streamlit run /content/drive/MyDrive/FreightQuote_AI/app.py --server.port 8503 --server.address 127.0.0.1 --server.headless true --browser.gatherUsageStats false
Working directory: /content/drive/MyDrive/FreightQuote_AI

=== 3. Quote UI function body ===
def render_customer_quotes(user):
    if not user or not user.get("id"):
        st.error("Please sign in first.")
        return

    init_quotes_table()

    st.header("Request a Freight Quote")
    st.caption("Enter your shipment details to submit a quote request.")

    with st.form("customer_freight_quote_form"):
        col1, col2 = st.columns(2)

        with col1:
            origin = st.text_input(
                "Origin port",
                placeholder="e.g. Mumbai",
                max_chars=100,
            )
            cargo = st.selectbox(
                "Cargo 

In [ ]:

import requests
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/FreightQuote_AI")
URL = "http://127.0.0.1:8503"

print("=== Streamlit HTTP check ===")
try:
    response = requests.get(URL, timeout=10)
    print("HTTP status:", response.status_code)
    print("Content type:", response.headers.get("content-type"))
    print("Response length:", len(response.text))
except requests.RequestException as exc:
    print("Request failed:", type(exc).__name__, str(exc))

print("\n=== Relevant dashboard source ===")
path = PROJECT / "dashboard_views.py"
lines = path.read_text(encoding="utf-8").splitlines()

for start, end in [(45, 75)]:
    for number in range(start, min(end, len(lines)) + 1):
        print(f"{number}: {lines[number - 1]}")

print("\nDiagnostic finished. No files or database records modified.")


=== Streamlit HTTP check ===
HTTP status: 200
Content type: text/html; charset=utf-8
Response length: 6467

=== Relevant dashboard source ===
45:         st.error("Unrecognized role. Contact an administrator.")
46:         return
47: 
48:     st.title("FreightQuote AI Dashboard")
49:     st.caption(f"Signed in as {username} | Role: {role}")
50:     st.divider()
51: 
52:     st.subheader(f"Welcome, {username}")
53:     st.markdown("### Your workspace")
54: 
55:     columns = st.columns(2)
56:     for index, feature in enumerate(ROLE_FEATURES[role]):
57:         with columns[index % 2]:
58:             with st.container(border=True):
59:                 st.markdown(f"**{feature}**")
60:                 st.caption("Integration pending")
61: 
62:     if role == "Customer":
63:         from quote_ui import render_customer_quotes
64:         render_customer_quotes(user)
65: 
66:     st.divider()
67:     st.subheader("Advanced AI Capabilities")
68: 
69:     st.markdown("**Live Agents and Digi

In [ ]:

from pathlib import Path

P = Path("/content/drive/MyDrive/FreightQuote_AI")

print("=== Dashboard quote integration ===")
dashboard = (P / "dashboard_views.py").read_text(encoding="utf-8")
start = dashboard.find('if role == "Customer":')
end = dashboard.find('st.subheader("Advanced AI Capabilities")')

print(dashboard[start:end] if start >= 0 and end > start
      else "Customer integration block not found")

print("\n=== Quote UI function ===")
quote = (P / "quote_ui.py").read_text(encoding="utf-8")
start = quote.find("def render_customer_quotes")
print(quote[start:] if start >= 0 else "Quote UI function not found")

print("\nRead-only inspection complete.")


=== Dashboard quote integration ===
if role == "Customer":
        from quote_ui import render_customer_quotes
        render_customer_quotes(user)

    st.divider()
    

=== Quote UI function ===
def render_customer_quotes(user):
    if not user or not user.get("id"):
        st.error("Please sign in first.")
        return

    init_quotes_table()

    st.header("Request a Freight Quote")
    st.caption("Enter your shipment details to submit a quote request.")

    with st.form("customer_freight_quote_form"):
        col1, col2 = st.columns(2)

        with col1:
            origin = st.text_input(
                "Origin port",
                placeholder="e.g. Mumbai",
                max_chars=100,
            )
            cargo = st.selectbox(
                "Cargo type",
                sorted(VALID_CARGO_TYPES),
            )
            quantity = st.number_input(
                "Number of containers / units",
                min_value=1,
                max_value=10000,
 

In [ ]:

from pathlib import Path
import ast
import shutil

PROJECT = Path("/content/drive/MyDrive/FreightQuote_AI")
TARGET = PROJECT / "dashboard_views.py"
BACKUP = PROJECT / "dashboard_views.py.pre_visible_quote_button"

source = TARGET.read_text(encoding="utf-8")

old = '''    if role == "Customer":
        from quote_ui import render_customer_quotes
        render_customer_quotes(user)
'''

new = '''    if role == "Customer":
        st.divider()
        st.subheader("Customer Actions")

        if st.button(
            "Open Freight Quote Form",
            type="primary",
            use_container_width=True,
            key="open_freight_quote_form",
        ):
            st.session_state["show_freight_quote_form"] = True

        if st.session_state.get("show_freight_quote_form", False):
            from quote_ui import render_customer_quotes
            render_customer_quotes(user)
'''

if source.count(old) != 1:
    raise RuntimeError(
        "Expected integration block was not found exactly once. "
        "No files were changed."
    )

if not BACKUP.exists():
    shutil.copy2(TARGET, BACKUP)
    print("Backup created:", BACKUP)
else:
    print("Existing backup preserved:", BACKUP)

updated = source.replace(old, new, 1)
ast.parse(updated)

temp = TARGET.with_suffix(".py.tmp")
temp.write_text(updated, encoding="utf-8")
ast.parse(temp.read_text(encoding="utf-8"))
temp.replace(TARGET)

print("PASS: Visible quote button added.")
print("PASS: dashboard_views.py syntax is valid.")
print("Authentication and app.py were not modified.")
print("Refresh the app and sign in as Customer to test.")


Backup created: /content/drive/MyDrive/FreightQuote_AI/dashboard_views.py.pre_visible_quote_button
PASS: Visible quote button added.
PASS: dashboard_views.py syntax is valid.
Authentication and app.py were not modified.
Refresh the app and sign in as Customer to test.


In [ ]:

from pathlib import Path
import ast
import shutil

PROJECT = Path("/content/drive/MyDrive/FreightQuote_AI")
TARGET = PROJECT / "dashboard_views.py"
BACKUP = PROJECT / "dashboard_views.py.phase1_customer_backup"

source = TARGET.read_text(encoding="utf-8")

old = '''    st.subheader(f"Welcome, {username}")
    st.markdown("### Your workspace")

    columns = st.columns(2)
    for index, feature in enumerate(ROLE_FEATURES[role]):
        with columns[index % 2]:
            with st.container(border=True):
                st.markdown(f"**{feature}**")
                st.caption("Integration pending")

    if role == "Customer":
        st.divider()
        st.subheader("Customer Actions")

        if st.button(
            "Open Freight Quote Form",
            type="primary",
            use_container_width=True,
            key="open_freight_quote_form",
        ):
            st.session_state["show_freight_quote_form"] = True

        if st.session_state.get("show_freight_quote_form", False):
            from quote_ui import render_customer_quotes
            render_customer_quotes(user)
'''

new = '''    st.subheader(f"Welcome, {username}")

    if role == "Customer":
        st.markdown("### Customer Dashboard")
        st.caption("Choose an action to continue.")

        col1, col2 = st.columns(2)

        with col1:
            if st.button(
                "Request a Freight Quote",
                type="primary",
                use_container_width=True,
                key="customer_request_quote",
            ):
                st.session_state["customer_view"] = "quotes"

        with col2:
            if st.button(
                "My Quotes / Shipment Status",
                use_container_width=True,
                key="customer_view_quotes",
            ):
                st.session_state["customer_view"] = "quotes"

        with st.expander("Customer Profile"):
            st.write("Username:", username)
            st.write("Role:", role)
            st.caption("Profile editing will be implemented in a later step.")

        if st.session_state.get("customer_view") == "quotes":
            from quote_ui import render_customer_quotes
            render_customer_quotes(user)

    else:
        st.markdown("### Your workspace")
        columns = st.columns(2)
        for index, feature in enumerate(ROLE_FEATURES[role]):
            with columns[index % 2]:
                with st.container(border=True):
                    st.markdown(f"**{feature}**")
                    st.caption("Implementation scheduled in the next phase.")
'''

if source.count(old) != 1:
    raise RuntimeError(
        "Expected dashboard block was not found exactly once. "
        "No changes made. Send me the current dashboard_views.py block."
    )

if not BACKUP.exists():
    shutil.copy2(TARGET, BACKUP)
    print("Backup created:", BACKUP)
else:
    print("Existing backup preserved:", BACKUP)

updated = source.replace(old, new, 1)
ast.parse(updated)

temp = TARGET.with_suffix(".phase1.tmp")
try:
    temp.write_text(updated, encoding="utf-8")
    ast.parse(temp.read_text(encoding="utf-8"))
    temp.replace(TARGET)
finally:
    if temp.exists():
        temp.unlink()

print("PASS: Customer action buttons added.")
print("PASS: Dashboard syntax valid.")
print("Authentication code and app.py were not changed.")
print("Next: refresh the app and test both Customer buttons.")


Backup created: /content/drive/MyDrive/FreightQuote_AI/dashboard_views.py.phase1_customer_backup
PASS: Customer action buttons added.
PASS: Dashboard syntax valid.
Authentication code and app.py were not changed.
Next: refresh the app and test both Customer buttons.


In [ ]:

from pyngrok import ngrok, conf
from google.colab import userdata
import requests

# Use your existing Colab secret
token = userdata.get("NGROK_AUTHTOKEN")
if not token:
    raise RuntimeError("NGROK_AUTHTOKEN is missing from Colab Secrets.")

conf.get_default().auth_token = token

# Clear old tunnels
ngrok.kill()

# Verify that Streamlit is running locally
health = requests.get(
    "http://127.0.0.1:8503/_stcore/health",
    timeout=10,
)
print("Streamlit health:", health.status_code)
health.raise_for_status()

# Create the tunnel using the correct target URL
tunnel = ngrok.connect(addr="http://127.0.0.1:8503", proto="http")

print("\nFreightQuote AI URL:", tunnel.public_url)


Streamlit health: 200

FreightQuote AI URL: https://reptile-unspoken-tuition.ngrok-free.dev


In [ ]:
from pathlib import Path
import ast
import shutil

PROJECT = Path("/content/drive/MyDrive/FreightQuote_AI")
dashboard_path = PROJECT / "dashboard_views.py"
quote_path = PROJECT / "quote_ui.py"

# Under the Enterprise UI, quote rendering is handled inside `fq_ui.py`'s
# `page_customer_quotes` and delegates cleanly to `quote_ui.py`.
# We will update `quote_ui.py` to accept the `view_mode` argument safely.

if not quote_path.exists():
    raise FileNotFoundError(f"quote_ui.py not found at {quote_path}")

quote_source = quote_path.read_text(encoding="utf-8")

signature = "def render_customer_quotes(user):"
form_marker = '    st.header("Request a Freight Quote")'
history_marker = '    st.divider()\\n    st.subheader("My Quote Requests")'

# Adjust markers to match the current quote_ui.py structure
if "view_mode" in quote_source:
    print("quote_ui.py is already upgraded with view_mode. Skipping modification.")
else:
    if quote_source.count(signature) != 1:
        # Fallback to alternative signature spacing if any
        signature = "def render_customer_quotes(user):"
        if quote_source.count(signature) != 1:
            raise RuntimeError("Quote function signature mismatch; no files changed.")

    form_marker_clean = 'st.header("Request a Freight Quote")'
    history_marker_clean = 'st.subheader("My Quote Requests")'

    start = quote_source.index(form_marker_clean)
    history = quote_source.index(history_marker_clean, start)

    # Slice and reconstruct with proper indentation under view_mode conditional branches
    prefix = quote_source[:start]

    # Walk backwards from start to capture headings/captions
    header_start = quote_source.rfind("st.header", 0, start)
    if header_start != -1 and header_start > quote_source.index("def render_customer_quotes"):
        start = header_start

    form_block = quote_source[start:history]
    history_block = quote_source[history:]

    # Format blocks under the new conditionals
    form_block = "    if view_mode == 'request':\n" + "".join(
        "    " + line if line.strip() else line
        for line in form_block.splitlines(keepends=True)
    )
    history_block = "    if view_mode == 'quotes':\n" + "".join(
        "    " + line if line.strip() else line
        for line in history_block.splitlines(keepends=True)
    )

    updated_quote = quote_source.replace(
        signature,
        "def render_customer_quotes(user, view_mode='request'):",
        1,
    )

    # Replace original content with nested conditional blocks
    insertion_point = updated_quote.index(form_marker_clean)
    header_index = updated_quote.rfind("st.header", 0, insertion_point)
    if header_index != -1 and header_index > updated_quote.index("def render_customer_quotes"):
        insertion_point = header_index

    updated_quote = updated_quote[:insertion_point] + form_block + "\n" + history_block

    # Validate and save
    ast.parse(updated_quote)

    backup_quote = PROJECT / "quote_ui.py.before_quote_views.bak"
    if not backup_quote.exists():
        shutil.copy2(quote_path, backup_quote)

    quote_path.write_text(updated_quote, encoding="utf-8")
    print("PASS: quote_ui.py updated to support distinct view modes.")

print("Syntax check: PASS")

quote_ui.py is already upgraded with view_mode. Skipping modification.
Syntax check: PASS


In [ ]:

from pathlib import Path
import ast

project = Path("/content/drive/MyDrive/FreightQuote_AI")
p = project / "quote_ui.py"
source = p.read_text(encoding="utf-8")
ast.parse(source)

print("Function accepts view_mode:",
      "def render_customer_quotes(user, view_mode='request'):" in source)
print("Request view condition:",
      "if view_mode == 'request':" in source)
print("History view condition:",
      "if view_mode == 'quotes':" in source)

print("\nSyntax: PASS")
print("No files or database records modified.")


Function accepts view_mode: True
Request view condition: True
History view condition: True

Syntax: PASS
No files or database records modified.


In [ ]:

from pathlib import Path

p = Path("/content/drive/MyDrive/FreightQuote_AI/dashboard_views.py")
s = p.read_text(encoding="utf-8")
lines = s.splitlines()

for i, line in enumerate(lines):
    if any(term in line for term in (
        "customer_view",
        "render_customer_quotes",
        "customer_request_quote",
        "customer_view_quotes",
    )):
        start = max(0, i - 3)
        end = min(len(lines), i + 5)
        print(f"\n--- Lines {start + 1}-{end} ---")
        for n in range(start, end):
            print(f"{n + 1}: {lines[n]}")

print("\nRead-only inspection complete.")



--- Lines 62-69 ---
62:                 "Request a Freight Quote",
63:                 type="primary",
64:                 use_container_width=True,
65:                 key="customer_request_quote",
66:             ):
67:                 st.session_state["customer_view"] = "request"
68: 
69:         with col2:

--- Lines 64-71 ---
64:                 use_container_width=True,
65:                 key="customer_request_quote",
66:             ):
67:                 st.session_state["customer_view"] = "request"
68: 
69:         with col2:
70:             if st.button(
71:                 "My Quotes / Shipment Status",

--- Lines 70-77 ---
70:             if st.button(
71:                 "My Quotes / Shipment Status",
72:                 use_container_width=True,
73:                 key="customer_view_quotes",
74:             ):
75:                 st.session_state["customer_view"] = "quotes"
76: 
77:         with st.expander("Customer Profile"):

--- Lines 72-79 ---
72:                 

In [ ]:

from pathlib import Path

p = Path("/content/drive/MyDrive/FreightQuote_AI/quote_ui.py")
lines = p.read_text(encoding="utf-8").splitlines()

for i, line in enumerate(lines):
    if "if view_mode ==" in line:
        start = max(0, i - 2)
        end = min(len(lines), i + 12)
        print(f"\n--- Lines {start + 1}-{end} ---")
        for n in range(start, end):
            print(f"{n + 1}: {lines[n]}")

print("\nRead-only scope check complete.")



--- Lines 18-31 ---
18:     init_quotes_table()
19: 
20:     if view_mode == 'request':
21:         st.header("Request a Freight Quote")
22:         st.caption("Enter your shipment details to submit a quote request.")
23: 
24:         with st.form("customer_freight_quote_form"):
25:             col1, col2 = st.columns(2)
26: 
27:             with col1:
28:                 origin = st.text_input(
29:                     "Origin port",
30:                     placeholder="e.g. Mumbai",
31:                     max_chars=100,

--- Lines 98-111 ---
98:                 print("Quote submission error:", type(exc).__name__)
99: 
100:     if view_mode == 'quotes':
101:         st.divider()
102:         st.subheader("My Quote Requests")
103: 
104:         try:
105:             quotes = list_my_quotes(user)
106: 
107:             if not quotes:
108:                 st.info("You have not submitted any quote requests yet.")
109:             else:
110:                 for quote in quotes:
111:      

In [ ]:

from pathlib import Path
import ast
import sqlite3
import os

PROJECT = Path("/content/drive/MyDrive/FreightQuote_AI")

print("=== PHASE 2: ADMIN READ-ONLY AUDIT ===")

# 1. Inspect authentication and role-related functions
app_path = PROJECT / "app.py"
source = app_path.read_text(encoding="utf-8")
tree = ast.parse(source)

keywords = (
    "def init_db", "def register_user", "def authenticate",
    "def find_user", "def record_login_failure",
    "CREATE TABLE", "role", "audit"
)

print("\n--- Relevant app.py lines ---")
for number, line in enumerate(source.splitlines(), start=1):
    if any(word.lower() in line.lower() for word in keywords):
        print(f"{number}: {line.strip()[:180]}")

# 2. Determine the configured database path without printing credentials
configured = os.environ.get("FREIGHTQUOTE_DB")
db_path = Path(configured) if configured else PROJECT / "freightquote.db"
if not db_path.is_absolute():
    db_path = PROJECT / db_path

print("\n--- Database schema audit ---")
print("Configured DB file exists:", db_path.exists())

if db_path.exists():
    try:
        with sqlite3.connect(f"file:{db_path}?mode=ro", uri=True) as conn:
            tables = conn.execute("""
                SELECT name FROM sqlite_master
                WHERE type = 'table'
                ORDER BY name
            """).fetchall()

            for (table,) in tables:
                print(f"\nTable: {table}")
                columns = conn.execute(
                    f'PRAGMA table_info("{table}")'
                ).fetchall()
                print("Columns:", [row[1] for row in columns])
    except Exception as exc:
        print("Read-only DB inspection failed:", type(exc).__name__)
else:
    print("Database not found at the configured default path.")

# 3. Inspect existing role dashboard modules
print("\n--- Existing role-related files ---")
for name in (
    "dashboard_views.py", "security_auth.py",
    "quote_service.py", "quote_ui.py"
):
    path = PROJECT / name
    print(name, "EXISTS" if path.exists() else "MISSING")

print("\nAudit complete. No files or database records modified.")


=== PHASE 2: ADMIN READ-ONLY AUDIT ===

--- Relevant app.py lines ---
84: def init_db():
88: CREATE TABLE IF NOT EXISTS users (
93: role TEXT NOT NULL DEFAULT 'Customer',
108: CREATE TABLE IF NOT EXISTS password_resets (
118: CREATE TABLE IF NOT EXISTS login_attempts (
125: CREATE TABLE IF NOT EXISTS answer_attempts (
215: def find_user(identity):
225: def register_user(username, email, password, confirm, answers):
254: (username, email, password_hash, role, created_at,
286: def record_login_failure(email):
323: def authenticate(identity, password):
611: "email": user["email"], "role": user["role"]}

--- Database schema audit ---
Configured DB file exists: True

Table: answer_attempts
Columns: ['email', 'attempts', 'locked_until']

Table: freight_quotes
Columns: ['id', 'user_id', 'username', 'origin_port', 'destination_port', 'cargo_type', 'container_type', 'quantity', 'shipment_date', 'notes', 'status', 'created_at']

Table: login_attempts
Columns: ['email', 'failures', 'locked_until'

In [ ]:

from pathlib import Path
import os
import sqlite3

PROJECT = Path("/content/drive/MyDrive/FreightQuote_AI")
configured = os.environ.get("FREIGHTQUOTE_DB")
db_path = Path(configured) if configured else PROJECT / "freightquote.db"

if not db_path.is_absolute():
    db_path = PROJECT / db_path

print("=== ADMIN READ-ONLY CHECK ===")

if not db_path.exists():
    print("Database file not found.")
else:
    with sqlite3.connect(f"file:{db_path}?mode=ro", uri=True) as conn:
        print("\nRole counts:")
        rows = conn.execute("""
            SELECT role, COUNT(*)
            FROM users
            GROUP BY role
            ORDER BY role
        """).fetchall()

        for role, count in rows:
            print(f"{role}: {count}")

        print("\nQuote status counts:")
        rows = conn.execute("""
            SELECT status, COUNT(*)
            FROM freight_quotes
            GROUP BY status
            ORDER BY status
        """).fetchall()

        for status, count in rows:
            print(f"{status}: {count}")

        print("\nAdmin role exists:",
              conn.execute(
                  "SELECT 1 FROM users WHERE role = 'Admin' LIMIT 1"
              ).fetchone() is not None)

print("\nRead-only check complete. No records modified.")


=== ADMIN READ-ONLY CHECK ===

Role counts:
Customer: 2

Quote status counts:
Pending: 6

Admin role exists: False

Read-only check complete. No records modified.


In [ ]:

from pathlib import Path

project = Path("/content/drive/MyDrive/FreightQuote_AI")
path = project / "app.py"
source = path.read_text(encoding="utf-8")
lines = source.splitlines()

terms = (
    "Sign in",
    "Sign In",
    "Forgot Password",
    "Register",
    "st.radio",
    "st.session_state",
    "st.form",
    "demo",
)

hits = [
    i for i, line in enumerate(lines)
    if any(term.lower() in line.lower() for term in terms)
]

# Show the login UI section, without printing secrets or passwords.
if not hits:
    print("Login UI markers were not found.")
else:
    start = max(0, min(hits) - 8)
    end = min(len(lines), max(hits) + 35)
    for i in range(start, end):
        print(f"{i + 1}: {lines[i]}")

print("\nRead-only inspection complete.")


1: """
2: FreightQuote AI - Authentication module
3: Sign In (email or username) / Register / Forgot Password.
4: 
5: Register collects three security answers (pet name, first school, favorite city).
6: Sign In locks an account after too many wrong passwords.
7: Forgot Password works two ways:
8:     1. Security questions - no email setup needed
9:     2. Email OTP          - needs the two secrets below
10: 
11: Secrets (Colab Secrets or environment variables):
12:     SMTP_EMAIL         - Gmail address used to send OTP emails
13:     SMTP_APP_PASSWORD  - Gmail App Password (16 chars, spaces allowed)
14: """
15: 
16: import os
17: import re
18: import ssl
19: import sqlite3
20: import secrets
21: import smtplib
22: from contextlib import contextmanager
23: from datetime import datetime, timedelta, timezone
24: from email.message import EmailMessage
25: from pathlib import Path
26: 
27: import bcrypt
28: import streamlit as st
29: 
30: # ---------------- Configuration ----------------
3

In [ ]:

from pathlib import Path

p = Path("/content/drive/MyDrive/FreightQuote_AI/app.py")
source = p.read_text(encoding="utf-8")

print("Role buttons found:", all(
    label in source
    for label in [
        "Admin",
        "Operations Manager",
        "Freight Broker",
        "Auditor",
    ]
))
print("Login section found:", 'if mode == "Sign In":' in source)
print("Current app path:", p)


Role buttons found: False
Login section found: True
Current app path: /content/drive/MyDrive/FreightQuote_AI/app.py


In [ ]:

from pathlib import Path

p = Path("/content/drive/MyDrive/FreightQuote_AI/app.py")
s = p.read_text(encoding="utf-8")

for role in ["Admin", "Operations Manager", "Freight Broker", "Auditor"]:
    print(f"{role}: {'FOUND' if f'"{role}"' in s else 'MISSING'}")

print("Login form preserved:", 'with st.form("signin_form")' in s)


Admin: MISSING
Operations Manager: MISSING
Freight Broker: MISSING
Auditor: MISSING
Login form preserved: True


In [ ]:

from pathlib import Path
import ast
import shutil
import os

PROJECT = Path("/content/drive/MyDrive/FreightQuote_AI")
APP = PROJECT / "app.py"
BACKUP = PROJECT / "app.py.before_role_buttons.bak"
TEMP = PROJECT / "app.py.role_buttons.tmp"

if not APP.exists():
    raise FileNotFoundError(f"Existing app.py not found: {APP}")

source = APP.read_text(encoding="utf-8")
lines = source.splitlines(keepends=True)

# Idempotency: do not insert the same buttons twice.
if 'key="login_role_admin"' in source:
    print("Role buttons already exist. No duplicate changes made.")
else:
    # Find the real Sign In heading without replacing the login form.
    target_index = None
    indent = ""

    for i, line in enumerate(lines):
        if line.strip() in (
            'st.subheader("Sign in")',
            "st.subheader('Sign in')",
            'st.subheader("Sign In")',
            "st.subheader('Sign In')",
        ):
            target_index = i
            indent = line[:len(line) - len(line.lstrip())]
            break

    if target_index is None:
        raise RuntimeError(
            "Could not locate the Sign In heading safely. "
            "No files were changed."
        )

    block = [
        "",
        f'{indent}st.markdown("**Choose your workspace**")',
        f'{indent}role_cols = st.columns(2)',
        f'{indent}role_buttons = [',
        f'{indent}    ("Admin", "login_role_admin"),',
        f'{indent}    ("Operations Manager", "login_role_ops"),',
        f'{indent}    ("Freight Broker", "login_role_broker"),',
        f'{indent}    ("Auditor", "login_role_auditor"),',
        f'{indent}]',
        f'{indent}for index, (role_label, role_key) in enumerate(role_buttons):',
        f'{indent}    with role_cols[index % 2]:',
        f'{indent}        if st.button(',
        f'{indent}            role_label,',
        f'{indent}            key=role_key,',
        f'{indent}            use_container_width=True,',
        f'{indent}        ):',
        f'{indent}            st.session_state["selected_login_role"] = role_label',
        f'{indent}selected_role = st.session_state.get("selected_login_role")',
        f'{indent}if selected_role:',
        f'{indent}    st.caption(',
        f'{indent}        f"Selected workspace: {{selected_role}}. Sign in with your assigned account."',
        f'{indent}    )',
        "",
    ]

    # Preserve the existing login form and authentication logic.
    updated_lines = (
        lines[:target_index + 1]
        + [line + "\n" if not line.endswith("\n") else line for line in block]
        + lines[target_index + 1:]
    )
    updated_source = "".join(updated_lines)
    ast.parse(updated_source)

    # Reuse the existing backup; create it only if absent.
    if not BACKUP.exists():
        shutil.copy2(APP, BACKUP)
        print("Backup created:", BACKUP)
    else:
        print("Reusing existing backup:", BACKUP)

    try:
        TEMP.write_text(updated_source, encoding="utf-8")
        ast.parse(TEMP.read_text(encoding="utf-8"))
        TEMP.replace(APP)
    finally:
        if TEMP.exists():
            TEMP.unlink()

    print("PASS: Role buttons inserted into the existing Sign In section.")

# Verify the saved file.
saved = APP.read_text(encoding="utf-8")
ast.parse(saved)

for label in ("Admin", "Operations Manager", "Freight Broker", "Auditor"):
    print(f"{label}: {'FOUND' if label in saved else 'MISSING'}")

print("Python syntax: PASS")
print("Authentication functions were not intentionally modified.")
print("No database records were modified.")


Backup created: /content/drive/MyDrive/FreightQuote_AI/app.py.before_role_buttons.bak
PASS: Role buttons inserted into the existing Sign In section.
Admin: FOUND
Operations Manager: FOUND
Freight Broker: FOUND
Auditor: FOUND
Python syntax: PASS
Authentication functions were not intentionally modified.
No database records were modified.


In [ ]:
from pathlib import Path
import shutil
import ast

project_dir = Path("/content/drive/MyDrive/FreightQuote_AI")
app_file = project_dir / "app.py"
backup_file = project_dir / "app.py.before_customer_button.bak"

if not app_file.exists():
    raise FileNotFoundError(f"Cannot find: {app_file}")

source = app_file.read_text(encoding="utf-8")

# Don't insert the button twice.
if '"login_role_customer"' in source or "'login_role_customer'" in source:
    print("Customer button code already exists. No changes made.")
else:
    # Add Customer to the existing role_buttons list.
    old = '''        ("Auditor", "login_role_auditor"),
    ]'''
    new = '''        ("Auditor", "login_role_auditor"),
        ("Customer", "login_role_customer"),
    ]'''

    if old not in source:
        raise RuntimeError(
            "Could not find the expected role_buttons list. "
            "No changes were made; please share the current role_buttons code."
        )

    # Create one backup, without overwriting an existing backup.
    if not backup_file.exists():
        shutil.copy2(app_file, backup_file)
        print(f"Backup created: {backup_file}")
    else:
        print(f"Existing backup preserved: {backup_file}")

    updated = source.replace(old, new, 1)
    ast.parse(updated)
    app_file.write_text(updated, encoding="utf-8")
    print("PASS: Customer button added and Python syntax is valid.")

print("Next: refresh your FreightQuote AI browser page.")

Backup created: /content/drive/MyDrive/FreightQuote_AI/app.py.before_customer_button.bak
PASS: Customer button added and Python syntax is valid.
Next: refresh your FreightQuote AI browser page.


In [ ]:
from pathlib import Path
import ast
import shutil

PROJECT = Path("/content/drive/MyDrive/FreightQuote_AI")
APP = PROJECT / "app.py"
BACKUP = PROJECT / "app.py.before_rbac_phase1.bak"

if not APP.exists():
    raise FileNotFoundError(f"app.py not found: {APP}")

source = APP.read_text(encoding="utf-8")

# Under the Enterprise UI, the sign-in redirect block is inside app.py's updated UI section:
old_block = """                if submit and ss.auth_mode == "Sign In":
                    user, error = authenticate(identity, password)
                    if user:
                        selected_role = ss.get("selected_login_role")
                        actual_role = str(user["role"]).strip().casefold()
                        expected_role = ROLE_ALIASES.get(
                            str(selected_role).strip().casefold()) if selected_role else None
                        verified_role = ROLE_ALIASES.get(actual_role)

                        if expected_role is None:
                            st.warning("Please select a workspace before signing in.")
                        elif verified_role is None:
                            st.error("Your account has an unrecognized role. "
                                     "Please contact an administrator.")
                        elif expected_role != verified_role:
                            st.error(f"Access denied: this account is assigned the "
                                     f"{user['role']} role, not {selected_role}. "
                                     "Select the workspace assigned to your account.")
                        else:
                            ss.user = {"id": user["id"], "username": user["username"],
                                       "email": user["email"], "role": user["role"]}
                            st.rerun()
                    else:
                        st.error(error)"""

# We check if this RBAC check block already exists in our upgraded app.py
if "Access denied: this account is assigned the" in source:
    print("PASS: RBAC workspace role checking is already integrated into the Enterprise UI app.py!")
else:
    raise RuntimeError("Could not find the target Sign In block in the upgraded app.py. Please verify app.py layout.")

# Validate before writing
ast.parse(source)
print("PASS: Saved Python syntax is valid.")
print("PASS: No database records were modified.")

PASS: RBAC workspace role checking is already integrated into the Enterprise UI app.py!
PASS: Saved Python syntax is valid.
PASS: No database records were modified.


In [ ]:
from pathlib import Path

app_file = Path(
    "/content/drive/MyDrive/FreightQuote_AI/app.py"
)

source = app_file.read_text(encoding="utf-8")
start = source.find('# ---- Sign In ----')

if start == -1:
    start = source.find('if mode == "Sign In":')

if start == -1:
    start = source.find("if mode == 'Sign In':")

if start == -1:
    print("Sign In section marker not found.")
else:
    print(source[start:start + 2500])

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
print("Colab runtime is connected.")

In [ ]:
# ================= FreightQuote AI - Enterprise UI upgrade =================
# Creates the folder FreightQuote_AI/ui_upgrade/ (source copy + backups),
# writes fq_ui.py, and replaces ONLY the UI section of app.py.
# All sign-in / database logic is kept. Safe to re-run.
import ast, shutil, time, os
from pathlib import Path

if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

PROJECT = Path("/content/drive/MyDrive/FreightQuote_AI")
APP = PROJECT / "app.py"
UPGRADE_DIR = PROJECT / "ui_upgrade"
BACKUP_DIR = UPGRADE_DIR / "backups"
BACKUP_DIR.mkdir(parents=True, exist_ok=True)
MARKER = "# ---------------- UI ----------------"

FQ_UI = r'''"""
FreightQuote AI - enterprise UI layer.

Theme (CSS), login layout pieces, signed-in shell (sidebar + top bar),
the AI Copilot page and the generic module pages.

This module never writes to any database. Every query it runs opens SQLite
in read-only mode, skips credential tables/columns, and limits Customers to
their own rows.
"""

import html
import re
import sqlite3
import time
from pathlib import Path

import streamlit as st

BASE_DIR = Path(__file__).resolve().parent

# ---------------------------------------------------------------- copy ----
BRAND = "FreightQuote AI"
HERO_LEAD = ("Agents watch your data, raise tickets, and wait for a human to "
             "approve anything that matters.")
HERO_POINTS = [
    "Freight quote requests and live shipment status",
    "AI Copilot that answers from your live data, with sources",
    "Secure sign-in with lockout, OTP and security questions",
    "Role-based access for every workspace",
]

ROLE_BUTTONS = [
    ("Admin", "login_role_admin"),
    ("Operations Manager", "login_role_ops"),
    ("Freight Broker", "login_role_broker"),
    ("Auditor", "login_role_auditor"),
    ("Customer", "login_role_customer"),
]

# ------------------------------------------------------------- navigation ----
# (section, page id, label, material icon)
NAV = [
    ("Assistant", "copilot", "AI Copilot", "auto_awesome"),
    ("Agents", "route", "Route AI", "alt_route"),
    ("Agents", "quotes", "Spot Quotes", "attach_money"),
    ("Agents", "carriers", "Carriers", "local_shipping"),
    ("Agents", "weather", "Weather Risk", "thunderstorm"),
    ("Agents", "margin", "Margin", "trending_up"),
    ("Agents", "customs", "Customs", "description"),
    ("Insights", "reports", "Reports", "summarize"),
    ("Insights", "notifications", "Notifications", "notifications"),
    ("Insights", "graph", "Knowledge Graph", "hub"),
    ("Insights", "twin", "Digital Twin", "deployed_code"),
    ("Platform", "feeds", "Data Feed Center", "upload"),
    ("Platform", "admin", "Admin", "shield"),
    ("Platform", "profile", "My Profile", "person"),
]
PAGE_TITLES = {pid: label for _, pid, label, _ in NAV}
PAGE_TITLES["command"] = "Command Center"

PAGE_SUBTITLES = {
    "copilot": "Agents run first, then the Copilot answers from their fresh evidence in your live data.",
    "command": "Your workspace: quotes, shipments and account actions.",
    "route": "Ports, routes and shipment movement from your live data.",
    "quotes": "Spot quote requests, pricing and approval status.",
    "carriers": "Carrier records and performance from your live data.",
    "weather": "Weather and storm exposure across ports and vessels.",
    "margin": "Quote margins and pricing outliers.",
    "customs": "Customs holds and clearance status.",
    "reports": "Every data table the platform can read, at a glance.",
    "notifications": "Alerts, tickets and items waiting for a human.",
    "graph": "How your records connect across tables.",
    "twin": "A live mirror of shipments and network state.",
    "feeds": "Data sources feeding the platform.",
    "admin": "Users, roles and account lockouts. Read-only view.",
    "profile": "Your account details.",
}

# Keywords that decide which tables a module page shows.
PAGE_KEYWORDS = {
    "route": ["route", "port", "shipment", "vessel", "voyage", "lane"],
    "quotes": ["quote", "rate", "pricing", "request"],
    "carriers": ["carrier", "vendor", "fleet"],
    "weather": ["weather", "storm", "forecast", "wind", "wave"],
    "margin": ["margin", "quote", "price", "cost"],
    "customs": ["custom", "clearance", "hold", "declaration"],
    "notifications": ["notification", "alert", "ticket", "event", "audit", "log"],
    "graph": ["relation", "edge", "node", "graph", "entity"],
    "twin": ["twin", "simulation", "snapshot", "shipment", "vessel"],
    "feeds": ["feed", "source", "import", "upload", "ingest"],
}

ALL_ROLES = {"admin", "operations manager", "freight broker", "auditor", "customer"}
PAGE_ROLES = {
    "admin": {"admin"},
    "route": ALL_ROLES - {"customer"},
    "carriers": ALL_ROLES - {"customer"},
    "weather": ALL_ROLES - {"customer"},
    "margin": ALL_ROLES - {"customer"},
    "customs": ALL_ROLES - {"customer"},
    "graph": ALL_ROLES - {"customer"},
    "twin": ALL_ROLES - {"customer"},
    "feeds": ALL_ROLES - {"customer"},
    "reports": ALL_ROLES - {"customer"},
}

# -------------------------------------------------------------- data safety ----
SENSITIVE_TABLES = {
    "password_resets", "login_attempts", "answer_attempts", "auth_users",
    "sqlite_sequence", "sqlite_stat1", "sqlite_stat4",
}
SENSITIVE_COL_RE = re.compile(r"(hash|password|passwd|otp|secret|token|salt|api_?key)", re.I)
OWNER_ID_COLS = ("user_id", "customer_id", "owner_id", "created_by", "requested_by")
OWNER_EMAIL_COLS = ("email", "customer_email", "user_email", "requester_email")
STATUS_COLS = ("status", "state", "current_status", "quote_status", "shipment_status",
               "approval_status", "severity", "current_severity", "risk_level")
ATTENTION_RE = re.compile(
    r"(pending|hold|delay|reject|fail|error|overdue|open|review|await|block|"
    r"cancel|high|critical|risk|exception)", re.I)
DATE_COL_RE = re.compile(r"(_at$|date|time|created|updated)", re.I)

# ======================================================================= CSS ====

_TOKENS = """
:root{
  --fq-primary:#5b5bd6; --fq-primary-600:#4f46e5; --fq-primary-50:#eef0ff;
  --fq-primary-100:#e0e3ff; --fq-text:#1f2937; --fq-text-2:#374151;
  --fq-muted:#6b7280; --fq-subtle:#9ca3af; --fq-border:#e7e9ef;
  --fq-border-2:#dfe2e8; --fq-bg:#f7f8fb; --fq-card:#ffffff;
  --fq-green:#22c55e; --fq-red:#dc2626; --fq-amber:#d97706; --fq-teal:#0f766e;
}
"""

BASE_CSS = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&display=swap');
%(tokens)s
html, body, .stApp{font-family:'Inter',-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;}
.stApp :is(p,span,div,label,input,textarea,button,h1,h2,h3,h4,h5,h6,li,td,th,a,small,strong,summary)
  :not([data-testid="stIconMaterial"]):not(code *):not(pre *){font-family:'Inter',-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;}
.stApp{background:var(--fq-bg); color:var(--fq-text);}
#MainMenu, footer, [data-testid="stDeployButton"], [data-testid="stAppDeployButton"],
[data-testid="stStatusWidget"], [data-testid="stDecoration"], [data-testid="stMainMenu"]{display:none !important;}
header[data-testid="stHeader"]{background:transparent; pointer-events:none;}
header[data-testid="stHeader"] *{pointer-events:auto;}
[data-testid="stSidebarNav"], [data-testid="stSidebarNavSeparator"]{display:none !important;}
[data-testid="stMarkdownContainer"]:has(.fq-nav-label, .fq-brand, .fq-topbar, .fq-hero, .fq-answer, .fq-msg-user,
  .fq-demo-label, .fq-welcome, .fq-section-title, .fq-kpis, .fq-card, .fq-empty, .fq-auth-title, .fq-table-wrap){margin-bottom:0 !important;}

/* headings */
.stApp h1, .stApp h2, .stApp h3{color:var(--fq-text); letter-spacing:-.015em; font-weight:700;}
.stApp h2{font-size:1.25rem;} .stApp h3{font-size:1.05rem;}
[data-testid="stCaptionContainer"], .stCaption{color:var(--fq-muted) !important;}

/* labels + inputs */
[data-testid="stWidgetLabel"] p, [data-testid="stWidgetLabel"] label{font-size:12px !important; font-weight:500 !important; color:var(--fq-text-2) !important;}
[data-testid="stTextInputRootElement"], .stSelectbox [data-rac][role="group"], .stMultiSelect [data-rac][role="group"],
.stTextInput [data-baseweb="input"], .stNumberInput [data-baseweb="input"], .stDateInput [data-baseweb="input"],
.stTextArea [data-baseweb="textarea"], .stSelectbox [data-baseweb="select"] > div, .stMultiSelect [data-baseweb="select"] > div{
  background:#fff !important; border:1px solid var(--fq-border-2) !important; border-radius:8px !important; box-shadow:none !important;}
.stTextInput [data-baseweb="base-input"], .stTextArea [data-baseweb="base-input"], .stNumberInput [data-baseweb="base-input"]{background:#fff !important;}
[data-testid="stTextInputRootElement"]:focus-within, .stSelectbox [data-rac][role="group"]:focus-within,
.stTextInput [data-baseweb="input"]:focus-within, .stTextArea [data-baseweb="textarea"]:focus-within,
.stNumberInput [data-baseweb="input"]:focus-within, .stSelectbox [data-baseweb="select"] > div:focus-within{
  border-color:#a5b4fc !important; box-shadow:0 0 0 3px rgba(99,102,241,.13) !important;}
.stTextInput input, .stNumberInput input, .stTextArea textarea{font-size:13px !important; color:var(--fq-text) !important;}
.stTextInput input{padding:8px 11px !important; background:transparent !important;}
[data-testid="stTextInputRootElement"]{min-height:36px; overflow:hidden;}
.stSelectbox [data-rac][role="group"]{min-height:34px;}
.stSelectbox input{font-size:13px !important; color:var(--fq-text) !important;}
.stTextArea textarea{border:1px solid var(--fq-border-2) !important; border-radius:8px !important; background:#fff !important;}
.stSelectbox [data-baseweb="select"] div{font-size:13px;}

/* buttons */
.stButton button, .stFormSubmitButton button, .stDownloadButton button, .stLinkButton a{
  border-radius:8px !important; font-size:13px !important; font-weight:500 !important;
  min-height:34px; padding:6px 14px; box-shadow:none; transition:background .15s, border-color .15s, color .15s;}
.stButton button p, .stFormSubmitButton button p{font-size:13px !important; font-weight:inherit;}
button[data-testid="stBaseButton-primary"], button[data-testid="stBaseButton-primaryFormSubmit"]{
  background:var(--fq-primary) !important; border:1px solid var(--fq-primary) !important; color:#fff !important;}
button[data-testid="stBaseButton-primary"]:hover, button[data-testid="stBaseButton-primaryFormSubmit"]:hover{
  background:var(--fq-primary-600) !important; border-color:var(--fq-primary-600) !important;}
button[data-testid="stBaseButton-secondary"], button[data-testid="stBaseButton-secondaryFormSubmit"]{
  background:#fff !important; border:1px solid var(--fq-border-2) !important; color:var(--fq-text-2) !important;}
button[data-testid="stBaseButton-secondary"]:hover, button[data-testid="stBaseButton-secondaryFormSubmit"]:hover{
  border-color:#c7d2fe !important; color:var(--fq-primary) !important; background:#fafaff !important;}
button[data-testid="stBaseButton-tertiary"], button[data-testid="stBaseButton-tertiaryFormSubmit"]{
  color:var(--fq-primary) !important; padding:0 2px !important; min-height:0 !important;}
button[data-testid="stBaseButton-tertiary"]:hover, button[data-testid="stBaseButton-tertiaryFormSubmit"]:hover{
  color:var(--fq-primary-600) !important; text-decoration:underline;}
button:focus-visible{outline:2px solid #a5b4fc !important; outline-offset:2px;}

/* radio as segmented control */
.stRadio [role="radiogroup"]{gap:6px; flex-wrap:wrap;}
.stRadio [role="radiogroup"] > label{background:#fff; border:1px solid var(--fq-border-2); border-radius:8px; padding:5px 12px 5px 8px; margin:0;}
.stRadio [role="radiogroup"] > label:has(input:checked){border-color:#a5b4fc; background:var(--fq-primary-50);}
.stRadio [role="radiogroup"] > label p{font-size:12.5px !important;}

/* alerts, expanders, containers, tabs */
[data-testid="stAlert"]{border-radius:10px;}
[data-testid="stAlert"] p{font-size:13px;}
[data-testid="stExpander"] details{border:1px solid var(--fq-border) !important; border-radius:10px !important; background:#fff;}
[data-testid="stExpander"] summary p{font-size:13px; font-weight:600;}
[data-testid="stVerticalBlockBorderWrapper"]:has(> div > [data-testid="stVerticalBlock"]){border-radius:12px;}
.stTabs [data-baseweb="tab"] p{font-size:13px; font-weight:500;}
.stTabs [aria-selected="true"] p{color:var(--fq-primary);}
.stTabs [data-baseweb="tab-highlight"]{background:var(--fq-primary);}
hr{border-color:var(--fq-border) !important;}
[data-testid="stMetric"]{background:#fff; border:1px solid var(--fq-border); border-radius:12px; padding:12px 14px;}
[data-testid="stMetricLabel"] p{font-size:12px; color:var(--fq-muted);}
[data-testid="stDataFrame"]{border:1px solid var(--fq-border); border-radius:10px; overflow:hidden;}

/* shared components */
.fq-badge{display:inline-flex; align-items:center; justify-content:center; min-width:46px; padding:2px 10px;
  border-radius:999px; font-size:11px; font-weight:600; line-height:16px; border:1px solid; white-space:nowrap;}
.fq-badge.high{color:#dc2626; background:#fef2f2; border-color:#fecaca;}
.fq-badge.medium{color:#b45309; background:#fffbeb; border-color:#fde68a;}
.fq-badge.low{color:#0f766e; background:#f0fdfa; border-color:#99f6e4;}
.fq-badge.info{color:#4f46e5; background:#eef2ff; border-color:#c7d2fe;}
.fq-badge.none{color:#6b7280; background:#f9fafb; border-color:#e5e7eb;}
.fq-table-wrap{border:1px solid var(--fq-border); border-radius:10px; overflow:auto; background:#fff; max-width:100%;}
table.fq-table{width:100%; border-collapse:collapse; font-size:12px; color:var(--fq-text-2); margin:0;}
table.fq-table th, table.fq-table td{font-size:12px !important; line-height:1.4 !important;}
table.fq-table th{background:#fafbfc; color:var(--fq-text-2); font-weight:600; text-align:left; padding:8px 12px !important;
  border:0 !important; border-bottom:1px solid var(--fq-border) !important; white-space:nowrap;}
table.fq-table td{padding:6px 12px !important; border:0 !important; border-bottom:1px solid #f0f1f5 !important; white-space:nowrap; max-width:320px;
  overflow:hidden; text-overflow:ellipsis;}
table.fq-table tr:last-child td{border-bottom:0 !important;}
table.fq-table tr:hover td{background:#fafaff;}
.fq-card{background:#fff; border:1px solid var(--fq-border); border-radius:12px; padding:16px 18px; margin-bottom:12px;}
.fq-card .h{margin:0 0 4px; font-size:14px !important; font-weight:600; color:var(--fq-text);}
.fq-card p{margin:0; font-size:12.5px; color:var(--fq-muted);}
.fq-empty{background:#fff; border:1px dashed #d6d9e2; border-radius:12px; padding:28px 22px; text-align:center;}
.fq-empty .h{margin:0 0 6px; font-size:14px !important; font-weight:600; color:var(--fq-text);}
.fq-empty p{margin:0 auto; font-size:12.5px; color:var(--fq-muted); max-width:520px;}
""".replace("%(tokens)s", _TOKENS)

LOGIN_CSS = """
.block-container{padding:0 !important; max-width:100% !important;}
section[data-testid="stSidebar"], [data-testid="stSidebarCollapsedControl"], [data-testid="stExpandSidebarButton"]{display:none !important;}
header[data-testid="stHeader"]{display:none !important;}
.stApp{background:#fff;}
[data-testid="stMain"], .main{padding:0 !important;}
[data-testid="stMainBlockContainer"]{padding:0 !important;}
.st-key-fq_login{gap:0 !important;}
.st-key-fq_login > div > [data-testid="stHorizontalBlock"], .st-key-fq_login [data-testid="stHorizontalBlock"]:has(.fq-hero){
  gap:0 !important; min-height:100vh; align-items:stretch !important; flex-wrap:nowrap;}
[data-testid="stColumn"]:has(.fq-hero){
  background:linear-gradient(162deg,#7d86f6 0%,#6168e0 30%,#5459d4 52%,#3f7fc0 76%,#14a39b 100%);
  padding:0 !important; display:flex; flex-direction:column; justify-content:center; position:relative; overflow:hidden;}
[data-testid="stColumn"]:has(.fq-hero) > div{width:100%;}
[data-testid="stColumn"]:has(.st-key-fq_auth_panel){background:#fff; padding:0 !important;}
.fq-hero{padding:48px clamp(28px,4vw,44px); color:#fff; position:sticky; top:0; z-index:1; min-height:100vh;
  display:flex; flex-direction:column; justify-content:center; box-sizing:border-box;}
.fq-hero h1{color:#fff !important; font-size:clamp(26px,2.4vw,31px) !important; font-weight:700 !important; margin:0 0 16px !important;
  padding:0 !important; letter-spacing:-.02em; line-height:1.15;}
.fq-hero .lead{font-size:13px; line-height:1.6; color:rgba(255,255,255,.88); margin:0 0 20px; max-width:330px;}
.fq-hero ul{list-style:none; margin:0; padding:0;}
.fq-hero li{display:flex; align-items:center; gap:10px; font-size:12.5px; color:rgba(255,255,255,.92); margin:0 0 9px; padding:0;}
.fq-hero li::before{content:""; width:6px; height:6px; border-radius:50%; background:#7ee8e0; flex:0 0 6px; box-shadow:0 0 0 3px rgba(126,232,224,.18);}
.st-key-fq_auth_panel{max-width:400px; margin:0 auto; padding:%(pad_top)s 28px 40px !important; gap:12px !important;}
.fq-auth-title{font-size:28px; font-weight:700; color:#1f2937; letter-spacing:-.025em; line-height:1.1; margin:0 0 6px;}
.fq-auth-sub{font-size:11.5px !important; color:#6b7280; margin:0; line-height:1.4;}
.fq-auth-heading{font-size:15px; font-weight:600; color:#1f2937; margin:6px 0 0;}
.fq-status{display:flex; align-items:center; gap:9px; background:#eef1ff; border:1px solid #d3d9fe; color:#3c3f8f;
  border-radius:7px; padding:8px 12px; font-size:12px; margin:14px 0 6px;}
.fq-status .ring{width:11px; height:11px; border-radius:50%; border:2px solid #c7cdfb; border-top-color:#5b5bd6; flex:0 0 11px;
  animation:fqspin 1.1s linear infinite;}
.fq-status .ok{width:8px; height:8px; border-radius:50%; background:#22c55e; box-shadow:0 0 0 3px rgba(34,197,94,.18); flex:0 0 8px;}
@keyframes fqspin{to{transform:rotate(360deg)}}
.st-key-fq_auth_panel [data-testid="stForm"]{padding:0; border:0;}
.st-key-fq_auth_panel [data-testid="stForm"] [data-testid="stVerticalBlock"]{gap:10px;}
.st-key-fq_auth_panel .stTextInput [data-baseweb="input"]{min-height:38px;}
.st-key-fq_auth_links{justify-content:flex-end; align-items:center; gap:8px !important;}
.st-key-fq_auth_links > div:not(:first-child)::before{content:"\\00B7"; color:#9ca3af; margin-right:8px; font-size:13px;}
.st-key-fq_auth_links > div{display:flex; align-items:center; width:auto !important;}
.st-key-fq_auth_links button p{font-size:12.5px !important; font-weight:500 !important;}
.fq-demo-label{font-size:11.5px; color:#6b7280; margin:8px 0 0;}
.st-key-fq_role_chips{gap:7px !important; flex-wrap:wrap;}
.st-key-fq_role_chips > div{width:auto !important;}
.st-key-fq_role_chips button{min-height:26px !important; padding:3px 11px !important; border-radius:999px !important;}
.st-key-fq_role_chips button p{font-size:11.5px !important;}
.st-key-fq_role_chips button[data-testid="stBaseButton-secondary"]{border-color:#e3e5ea !important; color:#374151 !important;}
@media (max-width: 860px){
  .fq-hero{padding:36px 26px;}
  .fq-hero h1{font-size:26px !important;}
  .st-key-fq_auth_panel{padding:48px 22px 36px !important;}
}
@media (max-width: 640px){
  .st-key-fq_login [data-testid="stHorizontalBlock"]:has(.fq-hero){flex-wrap:wrap; min-height:0;}
  [data-testid="stColumn"]:has(.fq-hero), [data-testid="stColumn"]:has(.st-key-fq_auth_panel){width:100% !important; flex:1 1 100% !important; min-width:100% !important;}
  .fq-hero{padding:30px 20px 26px; min-height:0;}
  .fq-hero ul{display:none;}
  .fq-hero .lead{margin:0;}
  .st-key-fq_auth_panel{padding:28px 18px 36px !important;}
}
"""

APP_CSS = """
.block-container, [data-testid="stMainBlockContainer"]{padding:0 28px 40px !important; max-width:100% !important;}
section[data-testid="stSidebar"]{background:#fff !important; border-right:1px solid var(--fq-border);}
[data-testid="stSidebar"][data-collapsed="false"], section[data-testid="stSidebar"][aria-expanded="true"]{width:244px !important; min-width:244px !important; max-width:244px !important;}
[data-testid="stSidebarHeader"]{height:auto !important; min-height:0 !important; padding:8px 10px 0 !important; margin:0 !important;}
[data-testid="stSidebarContent"]{padding:0 !important;}
[data-testid="stSidebarUserContent"]{padding:0 10px 18px !important;}
[data-testid="stSidebarUserContent"] [data-testid="stVerticalBlock"]{gap:2px !important;}
.fq-brand{display:flex; align-items:center; gap:10px; padding:2px 4px 12px;}
.fq-logo{width:34px; height:34px; border-radius:10px; flex:0 0 34px; display:flex; align-items:center; justify-content:center;
  background:linear-gradient(140deg,#7178f2,#5b5bd6 60%,#4f46e5); box-shadow:0 3px 10px rgba(91,91,214,.28);}
.fq-brand .name{font-size:13.5px; font-weight:700; color:#111827; line-height:1.2; letter-spacing:-.01em;}
.fq-brand .who{font-size:10.5px; color:#6b7280; line-height:1.3; margin-top:1px; overflow:hidden; text-overflow:ellipsis; white-space:nowrap; max-width:172px;}
.st-key-fq_search{position:relative; margin-bottom:6px;}
.st-key-fq_search [data-baseweb="input"]{background:#f9fafb !important; border-color:#eceef2 !important; min-height:32px;}
.st-key-fq_search [data-baseweb="base-input"]{background:#f9fafb !important;}
.st-key-fq_search input{font-size:12px !important; padding:6px 52px 6px 10px !important;}
.st-key-fq_search::after{content:"Ctrl K"; position:absolute; right:8px; top:50%; transform:translateY(-50%); font-size:10px; color:#6b7280;
  border:1px solid #e5e7eb; background:#fff; border-radius:5px; padding:1px 6px; pointer-events:none; font-weight:500;}
.fq-nav-label{display:block; font-size:10.5px; font-weight:600; letter-spacing:.07em; color:#9ca3af; text-transform:uppercase;
  line-height:16px; margin:0; padding:14px 8px 4px;}
.st-key-fq_nav .stButton button{justify-content:flex-start !important; background:transparent !important; border:0 !important;
  color:#374151 !important; min-height:32px !important; padding:6px 10px !important; border-radius:8px !important; box-shadow:none !important;}
.st-key-fq_nav .stButton button > div, .st-key-fq_nav .stButton button > span{justify-content:flex-start !important;}
.st-key-fq_nav .stButton button:hover{background:#f4f5f8 !important; color:#111827 !important;}
.st-key-fq_nav .stButton button p{font-size:13px !important; font-weight:500 !important;}
.st-key-fq_nav .stButton button [data-testid="stIconMaterial"]{font-size:17px !important; color:#6b7280; margin-right:4px;}
.st-key-fq_nav .st-key-nav_%(active)s button, .st-key-fq_nav .st-key-nav_%(active)s button:hover{background:var(--fq-primary-50) !important; color:#4338ca !important;}
.st-key-fq_nav .st-key-nav_%(active)s button p{font-weight:600 !important;}
.st-key-fq_nav .st-key-nav_%(active)s button [data-testid="stIconMaterial"]{color:#4f46e5 !important;}
.st-key-fq_nav .st-key-nav_command{margin-top:12px;}
.st-key-fq_nav .st-key-nav_command button, .st-key-fq_nav .st-key-nav_command button:hover{background:linear-gradient(135deg,#6366f1,#7f80f3) !important; color:#fff !important;
  box-shadow:0 4px 12px rgba(99,102,241,.28) !important; min-height:36px !important;}
.st-key-fq_nav .st-key-nav_command button p, .st-key-fq_nav .st-key-nav_command button:hover p{font-weight:600 !important; color:#fff !important;}
.st-key-fq_nav .st-key-nav_command button [data-testid="stIconMaterial"], .st-key-fq_nav .st-key-nav_command button:hover [data-testid="stIconMaterial"]{color:#fff !important;}
.st-key-fq_nav .st-key-nav_command button:hover{filter:brightness(1.05);}
%(command_active)s
.st-key-fq_signout{margin-top:22px;}
.st-key-fq_signout button{min-height:28px !important; padding:3px 12px !important; font-size:12px !important;}
.st-key-fq_signout button p{font-size:12px !important;}
.fq-topbar{display:flex; align-items:flex-start; justify-content:space-between; gap:16px; flex-wrap:wrap;
  background:#fff; margin:0 -28px 0; padding:14px 28px 8px;}
.fq-topbar .title{font-size:19px; font-weight:700; color:#111827; letter-spacing:-.015em; line-height:1.25;}
.fq-topbar .subtitle{font-size:12px; color:#6b7280; margin-top:3px; max-width:420px; overflow:hidden; text-overflow:ellipsis; white-space:nowrap;}
.fq-pills{display:flex; gap:8px; flex-wrap:wrap; align-items:center; padding-top:2px;}
.fq-pill{display:inline-flex; align-items:center; gap:7px; font-size:11px; color:#4b5563; background:#fff; border:1px solid var(--fq-border);
  border-radius:999px; padding:4px 11px; white-space:nowrap;}
.fq-pill .dot{width:7px; height:7px; border-radius:50%; background:#22c55e;}
.fq-pill .dot.grey{background:#9ca3af;} .fq-pill .dot.amber{background:#f59e0b;}
.st-key-fq_toolbar{background:#fff; margin:-16px -28px 18px !important; padding:4px 28px 12px; border-bottom:1px solid var(--fq-border);
  width:calc(100% + 56px) !important; max-width:none !important; box-sizing:border-box;}
.st-key-fq_toolbar .stSelectbox{max-width:176px;}
.st-key-fq_toolbar [data-baseweb="select"] > div{min-height:32px !important;}
.st-key-fq_page_gap{height:18px;}
/* copilot */
.st-key-fq_chat{background:#fff; border:1px solid var(--fq-border); border-radius:14px; padding:18px 20px 20px !important; gap:14px !important;}
.fq-msg-user{display:flex; justify-content:flex-end;}
.fq-msg-user span{background:var(--fq-primary); color:#fff; padding:10px 14px; border-radius:12px 12px 4px 12px; font-size:13px;
  max-width:min(560px,85%); line-height:1.45; box-shadow:0 2px 8px rgba(91,91,214,.22); white-space:pre-wrap; word-break:break-word;}
.fq-answer{background:#fff; border:1px solid var(--fq-border); border-radius:12px; padding:12px 14px 14px; max-width:780px; box-shadow:0 1px 2px rgba(16,24,40,.04);}
.fq-trace{font-size:11px; color:#6b7280; line-height:1.75;}
.fq-trace div::before, .fq-trace summary::before{content:"\\203A\\00a0"; color:#9ca3af;}
.fq-trace details{margin:0;}
.fq-trace summary{list-style:none; cursor:pointer; color:#4f46e5; font-weight:600;}
.fq-trace summary::-webkit-details-marker{display:none;}
.fq-trace details[open] summary::before{content:"\\25BE\\00a0";}
.fq-trace summary::before{content:"\\25B8\\00a0"; color:#4f46e5;}
.fq-trace .src{padding-left:10px;}
.fq-trace pre{background:#f8fafc; border:1px solid var(--fq-border); border-radius:8px; padding:8px 10px; font-size:11px; color:#334155;
  white-space:pre-wrap; margin:4px 0 2px; font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;}
.fq-watch{border:1px solid var(--fq-border); border-radius:10px; margin:10px 0 12px; overflow:hidden;}
.fq-watch .row{display:grid; grid-template-columns:minmax(110px,150px) 80px 1fr; align-items:center; gap:12px; padding:9px 12px;
  border-bottom:1px solid #f0f1f5; font-size:12px; color:#374151;}
.fq-watch .row:last-child{border-bottom:0;}
.fq-watch .name{font-weight:500; color:#1f2937; overflow:hidden; text-overflow:ellipsis; white-space:nowrap;}
.fq-headline{font-size:12.5px !important; color:#374151; margin:0 0 8px; line-height:1.5;}
.fq-headline b{color:#111827;}
.fq-answer .note{font-size:12.5px; color:#374151; margin:2px 0 8px;}
.fq-welcome{padding:6px 2px 2px;}
.fq-welcome .h{margin:0 0 4px; font-size:15px !important; font-weight:600; color:#111827;}
.fq-welcome p{margin:0; font-size:12.5px; color:#6b7280;}
.st-key-fq_suggest{gap:8px !important; flex-wrap:wrap;}
.st-key-fq_suggest > div{width:auto !important;}
.st-key-fq_suggest button{border-radius:999px !important; min-height:28px !important; padding:4px 12px !important;}
.st-key-fq_suggest button p{font-size:12px !important;}
[data-testid="stChatInput"]{border-radius:12px !important; border:1px solid var(--fq-border-2) !important; background:#fff !important; box-shadow:0 2px 6px rgba(16,24,40,.05);}
[data-testid="stChatInput"]:focus-within{border-color:#a5b4fc !important;}
[data-testid="stChatInput"] textarea{font-size:13px !important;}
[data-testid="stBottomBlockContainer"]{padding:12px 28px 18px !important; background:var(--fq-bg);}
[data-testid="stBottom"] > div{background:var(--fq-bg);}
.fq-kpis{display:grid; grid-template-columns:repeat(auto-fit,minmax(170px,1fr)); gap:12px; margin:0 0 16px;}
.fq-kpi{background:#fff; border:1px solid var(--fq-border); border-radius:12px; padding:12px 14px;}
.fq-kpi .k{font-size:11.5px; color:#6b7280;} .fq-kpi .v{font-size:20px; font-weight:700; color:#111827; margin-top:2px;}
.fq-section-title{display:flex; align-items:center; justify-content:space-between; gap:10px; margin:4px 0 8px;}
.fq-section-title .h{margin:0; font-size:13.5px !important; font-weight:600; color:#111827;}
.fq-section-title span{font-size:11.5px; color:#6b7280;}
@media (max-width: 760px){
  .block-container, [data-testid="stMainBlockContainer"]{padding:0 16px 32px !important;}
  .fq-topbar{margin:0 -16px; padding:12px 16px 10px;}
  .st-key-fq_toolbar{margin:-16px -16px 14px !important; padding:4px 16px 10px; width:calc(100% + 32px) !important;}
  .fq-topbar .subtitle{white-space:normal;}
  .fq-watch .row{grid-template-columns:1fr auto; }
  .fq-watch .row .txt{grid-column:1 / -1;}
  .st-key-fq_chat{padding:14px 12px 16px !important;}
  [data-testid="stBottomBlockContainer"]{padding:10px 16px 14px !important;}
}
"""

LOGO_SVG = ('<svg width="18" height="18" viewBox="0 0 24 24" fill="none" stroke="#fff" stroke-width="2.2" '
            'stroke-linecap="round" stroke-linejoin="round"><path d="M12 3a9 9 0 1 0 9 9"/>'
            '<path d="M12 7a5 5 0 1 0 5 5"/><circle cx="12" cy="12" r="1.4" fill="#fff"/></svg>')


def _style(css):
    # st.html with only a <style> tag takes no vertical space
    st.html(f"<style>{css}</style>")


def inject_base_css():
    _style(BASE_CSS)


def inject_login_css(mode="Sign In"):
    pad = {"Sign In": "clamp(48px, 21vh, 210px)",
           "Register": "clamp(36px, 7vh, 72px)",
           "Forgot Password": "clamp(40px, 11vh, 120px)"}.get(mode, "clamp(48px, 18vh, 180px)")
    _style(LOGIN_CSS.replace("%(pad_top)s", pad))


def inject_app_css(active):
    cmd = ""
    if active == "command":
        cmd = (".st-key-fq_nav .st-key-nav_command button{box-shadow:0 0 0 3px rgba(99,102,241,.25),"
               "0 4px 12px rgba(99,102,241,.3) !important;}")
    key = "none" if active == "command" else re.sub(r"[^a-z0-9_]", "", active)
    _style(APP_CSS.replace("%(active)s", key).replace("%(command_active)s", cmd))


def esc(value):
    return html.escape("" if value is None else str(value))


def badge(level, text=None):
    level = (level or "none").lower()
    return f'<span class="fq-badge {esc(level)}">{esc(text or level.title())}</span>'


def html_table(columns, rows, numeric_cols=()):
    head = "".join(f"<th>{esc(_pretty(c))}</th>" for c in columns)
    body = []
    for r in rows:
        cells = []
        for c, v in zip(columns, r):
            if isinstance(v, float):
                v = f"{v:,.2f}"
            cells.append(f'<td title="{esc(v)}">{esc(v)}</td>')
        body.append("<tr>" + "".join(cells) + "</tr>")
    if not body:
        body = [f'<tr><td colspan="{max(1, len(columns))}" style="color:#9ca3af">No rows</td></tr>']
    return (f'<div class="fq-table-wrap"><table class="fq-table"><thead><tr>{head}</tr></thead>'
            f'<tbody>{"".join(body)}</tbody></table></div>')


def _pretty(name):
    name = str(name)
    return name[:1].upper() + name[1:]


# ================================================================ LOGIN ====

def render_login_hero():
    points = "".join(f"<li>{esc(p)}</li>" for p in HERO_POINTS)
    st.markdown(
        f'<div class="fq-hero"><h1>{esc(BRAND)}</h1>'
        f'<p class="lead">{esc(HERO_LEAD)}</p><ul>{points}</ul></div>',
        unsafe_allow_html=True,
    )


def render_auth_header(mode, status_ok=True, status_text=None):
    sub = "Freight operations platform · every sign-in is protected"
    heading = {"Register": "Create an account",
               "Forgot Password": "Reset your password"}.get(mode)
    if status_text is None:
        status_text = ("Ready: secure sign-in online (you can sign in)" if status_ok
                       else "Starting up: database (you can sign in shortly)")
    icon = '<span class="ok"></span>' if status_ok else '<span class="ring"></span>'
    head = f'<div class="fq-auth-heading">{esc(heading)}</div>' if heading else ""
    st.markdown(
        f'<div><div class="fq-auth-title">{esc(BRAND)}</div>'
        f'<div class="fq-auth-sub">{esc(sub)}</div>'
        f'<div class="fq-status">{icon}<span>{esc(status_text)}</span></div>{head}</div>',
        unsafe_allow_html=True,
    )


def render_role_chips(selected=None):
    """Workspace chips (same keys as before: login_role_*)."""
    def pick(label):
        st.session_state["selected_login_role"] = label

    st.markdown('<div class="fq-demo-label">Workspace (click to select)</div>', unsafe_allow_html=True)
    with st.container(horizontal=True, gap="small", key="fq_role_chips"):
        for label, key in ROLE_BUTTONS:
            st.button(label, key=key, on_click=pick, args=(label,),
                      type="primary" if selected == label else "secondary")


# ============================================================ DATA LAYER ====

def _qi(name):
    return '"' + str(name).replace('"', '""') + '"'


def _ro_connect(path):
    uri = Path(path).resolve().as_uri() + "?mode=ro"
    conn = sqlite3.connect(uri, uri=True, timeout=5)
    conn.execute("PRAGMA query_only = ON")
    return conn


def discover_databases(primary=None):
    found = []
    if primary:
        found.append(Path(primary))
    for folder in (BASE_DIR, BASE_DIR / "database", BASE_DIR / "data"):
        if folder.is_dir():
            for pattern in ("*.db", "*.sqlite", "*.sqlite3"):
                found.extend(sorted(folder.glob(pattern)))
    seen, out = set(), []
    for p in found:
        try:
            key = p.resolve()
        except OSError:
            continue
        if key in seen or not p.exists() or p.name == "security_auth.db":
            continue
        seen.add(key)
        out.append(p)
    return out


def _norm_role(role):
    return str(role or "").strip().casefold()


@st.cache_data(ttl=20, show_spinner=False)
def _catalog(db_paths):
    """[{db, table, columns:[(name,type)], rows}] for every readable table."""
    out = []
    for db in db_paths:
        try:
            with _ro_connect(db) as conn:
                tables = [r[0] for r in conn.execute(
                    "SELECT name FROM sqlite_master WHERE type IN ('table','view') "
                    "AND name NOT LIKE 'sqlite_%' ORDER BY name")]
                for t in tables:
                    if t.lower() in SENSITIVE_TABLES:
                        continue
                    cols = [(r[1], (r[2] or "").upper()) for r in conn.execute(f"PRAGMA table_info({_qi(t)})")]
                    try:
                        n = conn.execute(f"SELECT COUNT(*) FROM {_qi(t)}").fetchone()[0]
                    except sqlite3.Error:
                        n = None
                    out.append({"db": str(db), "table": t, "columns": cols, "rows": n})
        except sqlite3.Error:
            continue
    return out


def accessible_tables(user, db_paths):
    """Tables this user may read, with safe columns and an owner filter for Customers."""
    role = _norm_role(user.get("role"))
    result = []
    for entry in _catalog(tuple(str(p) for p in db_paths)):
        t = entry["table"].lower()
        if t == "users" and role not in ("admin", "auditor"):
            continue
        cols = [c for c, _ in entry["columns"] if not SENSITIVE_COL_RE.search(c)]
        if not cols:
            continue
        where, params = "", []
        if role == "customer":
            lower = {c.lower(): c for c, _ in entry["columns"]}
            idc = next((lower[c] for c in OWNER_ID_COLS if c in lower), None)
            emc = next((lower[c] for c in OWNER_EMAIL_COLS if c in lower), None)
            if t == "users":
                continue
            if idc and user.get("id") is not None:
                where, params = f"{_qi(idc)} = ?", [user["id"]]
            elif emc and user.get("email"):
                where, params = f"lower({_qi(emc)}) = lower(?)", [user["email"]]
            elif lower.get("username") and user.get("username"):
                where, params = f"lower({_qi(lower['username'])}) = lower(?)", [user["username"]]
            else:
                continue  # no way to scope rows to this customer
        types = {c: ty for c, ty in entry["columns"]}
        item = {**entry, "safe_cols": cols, "types": types, "where": where, "params": params}
        if where:  # report only this user's row count
            try:
                with _ro_connect(entry["db"]) as conn:
                    item["rows"] = conn.execute(
                        f"SELECT COUNT(*) FROM {_qi(entry['table'])} WHERE {where}", params).fetchone()[0]
            except sqlite3.Error:
                item["rows"] = None
        result.append(item)
    return result


def _status_col(tbl):
    lower = {c.lower(): c for c in tbl["safe_cols"]}
    return next((lower[c] for c in STATUS_COLS if c in lower), None)


def _date_col(tbl):
    for c in tbl["safe_cols"]:
        if DATE_COL_RE.search(c):
            return c
    return None


def _numeric_cols(tbl):
    return [c for c in tbl["safe_cols"]
            if any(k in tbl["types"].get(c, "") for k in ("INT", "REAL", "NUM", "FLOA", "DOUB", "DEC"))
            and not c.lower().endswith("id")]


def _where(tbl, extra=None, extra_params=()):
    parts, params = [], list(tbl["params"])
    if tbl["where"]:
        parts.append(tbl["where"])
    if extra:
        parts.append(extra)
        params.extend(extra_params)
    return (" WHERE " + " AND ".join(parts)) if parts else "", params


def watch(tbl):
    """Status summary for one table -> dict(name, level, text)."""
    name = re.sub(r"s$", "", tbl["table"].lower()) + "_watch"
    sc = _status_col(tbl)
    w, params = _where(tbl)
    try:
        with _ro_connect(tbl["db"]) as conn:
            total = conn.execute(f"SELECT COUNT(*) FROM {_qi(tbl['table'])}{w}", params).fetchone()[0]
            if not sc:
                return {"name": name, "level": "info" if total else "none",
                        "badge": "Live" if total else "Empty",
                        "text": f"{total:,} {tbl['table'].replace('_', ' ')} rows available"}
            groups = conn.execute(
                f"SELECT {_qi(sc)}, COUNT(*) FROM {_qi(tbl['table'])}{w} GROUP BY 1 ORDER BY 2 DESC", params
            ).fetchall()
    except sqlite3.Error:
        return {"name": name, "level": "none", "badge": "n/a", "text": "Could not read this table"}
    flagged = [(str(s), n) for s, n in groups if s is not None and ATTENTION_RE.search(str(s))]
    attention = sum(n for _, n in flagged)
    if total == 0:
        level, txt = "none", "No rows yet"
    elif attention == 0:
        level, txt = "low", f"0 of {total:,} need attention"
    else:
        share = attention / total
        level = "high" if share >= 0.3 else "medium"
        detail = ", ".join(f"{s} {n}" for s, n in flagged[:3])
        txt = f"{attention:,} of {total:,} need attention ({detail})"
    return {"name": name, "level": level, "badge": None, "text": txt}


def table_rows(tbl, limit=25, order_col=None, extra=None, extra_params=()):
    cols = tbl["safe_cols"]
    w, params = _where(tbl, extra, extra_params)
    order = f" ORDER BY {_qi(order_col)} DESC" if order_col else " ORDER BY rowid DESC"
    sql = f"SELECT {', '.join(_qi(c) for c in cols)} FROM {_qi(tbl['table'])}{w}{order} LIMIT {int(limit)}"
    with _ro_connect(tbl["db"]) as conn:
        try:
            rows = conn.execute(sql, params).fetchall()
        except sqlite3.OperationalError:
            sql = sql.replace(" ORDER BY rowid DESC", "")
            rows = conn.execute(sql, params).fetchall()
        total = conn.execute(f"SELECT COUNT(*) FROM {_qi(tbl['table'])}{w}", params).fetchone()[0]
    return cols, rows, total, sql, params


# =============================================================== COPILOT ====

def _tokens(text):
    words = re.findall(r"[a-z0-9]+", (text or "").lower())
    out = set(words)
    out |= {w[:-1] for w in words if w.endswith("s") and len(w) > 3}
    out |= {w[:-2] for w in words if w.endswith("es") and len(w) > 4}
    return out


def _parts(name):
    p = set(re.findall(r"[a-z0-9]+", name.lower().replace("_", " ")))
    return p | {w[:-1] for w in p if w.endswith("s") and len(w) > 3}


def plan(question, tables):
    toks = _tokens(question)
    scored = []
    for t in tables:
        s = 3 * len(_parts(t["table"]) & toks)
        s += sum(1 for c in t["safe_cols"] if _parts(c) & toks)
        scored.append((s, t["rows"] or 0, t))
    scored.sort(key=lambda x: (x[0], x[1]), reverse=True)
    return [t for s, _, t in scored if s > 0], toks


def answer(question, user, db_paths):
    t0 = time.perf_counter()
    tables = accessible_tables(user, db_paths)
    trace, sources = [], []
    if not tables:
        return {"q": question, "trace": ["No readable data tables for your workspace yet."],
                "sources": [], "sql": None, "watch": [], "headline": None, "columns": [], "rows": [],
                "note": "Nothing to answer from yet. Data will appear here as soon as quotes or other "
                        "records are saved.", "elapsed": time.perf_counter() - t0}

    picked, toks = plan(question, tables)
    watch_set = (picked or tables)[:5]
    trace.append("Planner selected: " + ", ".join(
        re.sub(r"s$", "", t["table"].lower()) + "_watch" for t in watch_set))
    trace.append("Running the selected agents and the read-only query engine...")
    watches = [watch(t) for t in watch_set]
    trace.append(f"Agents ready in {time.perf_counter() - t0:.1f}s")

    if not picked:
        rows = [(t["table"], t["rows"] if t["rows"] is not None else "?", len(t["safe_cols"]),
                 Path(t["db"]).name) for t in tables]
        rows.sort(key=lambda r: (r[1] if isinstance(r[1], int) else -1), reverse=True)
        trace.append("No table matched the question; showing what the Copilot can read")
        return {"q": question, "trace": trace, "watch": watches,
                "sources": [f"{t['table']} · {Path(t['db']).name}" for t in tables[:6]],
                "sql": "SELECT name FROM sqlite_master WHERE type = 'table'",
                "headline": f"<b>{len(tables)}</b> readable tables; mention one of them (or a column) to query it.",
                "columns": ["table", "rows", "columns", "database"], "rows": rows[:10],
                "note": None, "elapsed": time.perf_counter() - t0}

    main = picked[0]
    trace.append("Exact answer from your live data (read-only query)")
    lower = {c.lower(): c for c in main["safe_cols"]}
    sc = _status_col(main)
    extra, extra_params, filter_txt = None, [], ""

    # status filter, e.g. "pending quotes"
    if sc:
        try:
            w, params = _where(main)
            with _ro_connect(main["db"]) as conn:
                values = [r[0] for r in conn.execute(
                    f"SELECT DISTINCT {_qi(sc)} FROM {_qi(main['table'])}{w} LIMIT 200", params) if r[0] is not None]
            for v in values:
                vt = _tokens(str(v))
                if vt and vt <= toks or str(v).lower() in question.lower():
                    extra, extra_params, filter_txt = f"lower({_qi(sc)}) = lower(?)", [v], f" with {sc} = {v}"
                    break
        except sqlite3.Error:
            pass

    # group-by: "how many ... by/per <column>"
    group_col = None
    m = re.search(r"\b(?:by|per)\s+([a-z0-9_ ]+)", question.lower())
    counting = re.search(r"\b(how many|count|number of|per|breakdown|split|group)\b", question.lower())
    if m and counting:
        cand = _tokens(m.group(1))
        group_col = next((c for c in main["safe_cols"] if _parts(c) & cand), None)

    order_col, order_txt = None, ""
    nums = _numeric_cols(main)
    hit = next((c for c in nums if _parts(c) & toks), None)
    if hit:
        order_col, order_txt = hit, f", sorted by {hit.replace('_', ' ')}"
    elif toks & {"latest", "recent", "newest", "last", "new"} and _date_col(main):
        order_col = _date_col(main)
        order_txt = f", newest first by {order_col.replace('_', ' ')}"

    try:
        if group_col:
            w, params = _where(main, extra, extra_params)
            sql = (f"SELECT {_qi(group_col)}, COUNT(*) AS count FROM {_qi(main['table'])}{w} "
                   f"GROUP BY 1 ORDER BY 2 DESC LIMIT 50")
            with _ro_connect(main["db"]) as conn:
                rows = conn.execute(sql, params).fetchall()
            cols, total = [group_col, "count"], sum(r[1] for r in rows)
            headline = (f"<b>{total:,}</b> {esc(main['table'].replace('_', ' '))} rows{esc(filter_txt)}, "
                        f"grouped by {esc(group_col.replace('_', ' '))}.")
        else:
            cols, rows, total, sql, params = table_rows(main, limit=10, order_col=order_col,
                                                        extra=extra, extra_params=extra_params)
            headline = (f"<b>{total:,}</b> {esc(main['table'].replace('_', ' '))} entries{esc(filter_txt)}; "
                        f"showing {len(rows)}{esc(order_txt)}.")
    except sqlite3.Error as exc:
        return {"q": question, "trace": trace + [f"Query failed: {type(exc).__name__}"], "watch": watches,
                "sources": [], "sql": None, "headline": None, "columns": [], "rows": [],
                "note": "The query could not run against this table.", "elapsed": time.perf_counter() - t0}

    shown_sql = sql
    for p in params:
        shown_sql = shown_sql.replace("?", repr(p) if isinstance(p, str) else str(p), 1)
    for i, t in enumerate(picked[:3], 1):
        sources.append(f"[{i}] {t['table']} · {Path(t['db']).name} · {t['rows'] or 0:,} rows "
                       f"· match #{i}")
    elapsed = time.perf_counter() - t0
    return {"q": question, "trace": trace, "watch": watches, "sources": sources, "sql": shown_sql,
            "headline": headline, "columns": cols, "rows": rows, "note": None, "elapsed": elapsed}


def render_answer(res):
    tr = "".join(f"<div>{esc(line)}</div>" for line in res["trace"])
    if res.get("sources"):
        src = "".join(f'<div class="src">{esc(s)}</div>' for s in res["sources"])
        tr += f'<details open><summary>Sources ({len(res["sources"])})</summary>{src}</details>'
    if res.get("sql"):
        tr += f'<details><summary>Show the exact query</summary><pre>{esc(res["sql"])}</pre></details>'
    tr += f'<div>done in {res["elapsed"]:.1f}s</div>'
    parts = [f'<div class="fq-answer"><div class="fq-trace">{tr}</div>']
    if res.get("watch"):
        rows = "".join(
            f'<div class="row"><span class="name">{esc(w["name"])}</span>'
            f'<span>{badge(w["level"], w.get("badge"))}</span><span class="txt">{esc(w["text"])}</span></div>'
            for w in res["watch"])
        parts.append(f'<div class="fq-watch">{rows}</div>')
    if res.get("note"):
        parts.append(f'<div class="note">{esc(res["note"])}</div>')
    if res.get("headline"):
        parts.append(f'<div class="fq-headline">{res["headline"]}</div>')
    if res.get("columns"):
        parts.append(html_table(res["columns"], res["rows"]))
    parts.append("</div>")
    st.markdown("".join(parts), unsafe_allow_html=True)


def suggestions(user, db_paths):
    tables = accessible_tables(user, db_paths)
    out = []
    for t in sorted(tables, key=lambda t: t["rows"] or 0, reverse=True)[:4]:
        label = t["table"].replace("_", " ")
        sc = _status_col(t)
        out.append(f"How many {label} by {sc.replace('_', ' ')}?" if sc else f"Show the latest {label}")
    return out or ["What data can you read?"]


def page_copilot(user, db_paths):
    ss = st.session_state
    hist = ss.setdefault("fq_copilot_history", [])
    pending = ss.pop("fq_pending_q", None)
    typed = st.chat_input("Ask about your quotes, shipments, carriers or anything in your data...")
    question = (typed or pending or "").strip()
    if question:
        with st.spinner("Agents are running..."):
            res = answer(question[:500], user, db_paths)
        hist.append({"role": "user", "text": question[:500]})
        hist.append({"role": "assistant", "result": res})
        del hist[:-30]

    with st.container(key="fq_chat"):
        if not hist:
            st.markdown(
                '<div class="fq-welcome"><div class="h">Ask the Copilot</div>'
                '<p>Questions run read-only queries over your live FreightQuote data. '
                'Every answer shows its sources and the exact query.</p></div>',
                unsafe_allow_html=True)

            def ask(q):
                st.session_state["fq_pending_q"] = q

            with st.container(horizontal=True, gap="small", key="fq_suggest"):
                for i, q in enumerate(suggestions(user, db_paths)):
                    st.button(q, key=f"fq_sugg_{i}", on_click=ask, args=(q,))
        for item in hist:
            if item["role"] == "user":
                st.markdown(f'<div class="fq-msg-user"><span>{esc(item["text"])}</span></div>',
                            unsafe_allow_html=True)
            else:
                render_answer(item["result"])
        if hist:
            if st.button("Clear conversation", key="fq_clear_chat", type="tertiary", icon=":material/delete_sweep:"):
                ss["fq_copilot_history"] = []
                st.rerun()


# ============================================================ OTHER PAGES ====

def page_tables(pid, user, db_paths):
    tables = accessible_tables(user, db_paths)
    if pid == "reports":
        matches = tables
    else:
        keys = PAGE_KEYWORDS.get(pid, [])
        matches = [t for t in tables if any(k in t["table"].lower() for k in keys)]
    if not matches:
        st.markdown(
            f'<div class="fq-empty"><div class="h">No {esc(PAGE_TITLES.get(pid, pid))} data yet</div>'
            f'<p>This module fills in automatically once its agent or a data feed writes '
            f'{esc(", ".join(PAGE_KEYWORDS.get(pid, ["records"])[:3]))} records to the database. '
            f'Nothing is simulated here.</p></div>', unsafe_allow_html=True)
        return
    if pid == "reports":
        total_rows = sum(t["rows"] or 0 for t in matches)
        st.markdown(
            f'<div class="fq-kpis"><div class="fq-kpi"><div class="k">Readable tables</div><div class="v">{len(matches)}</div></div>'
            f'<div class="fq-kpi"><div class="k">Total rows</div><div class="v">{total_rows:,}</div></div>'
            f'<div class="fq-kpi"><div class="k">Databases</div><div class="v">{len({t["db"] for t in matches})}</div></div></div>',
            unsafe_allow_html=True)
    for t in matches[:8]:
        w = watch(t)
        try:
            cols, rows, total, _, _ = table_rows(t, limit=25, order_col=_date_col(t))
        except sqlite3.Error:
            st.warning(f"Could not read {t['table']}.")
            continue
        st.markdown(
            f'<div class="fq-section-title"><div class="h">{esc(t["table"].replace("_", " ").title())}</div>'
            f'<span>{badge(w["level"], w.get("badge"))}&nbsp; {esc(w["text"])} · showing {len(rows)} of {total:,}</span></div>'
            + html_table(cols, rows) + '<div style="height:18px"></div>', unsafe_allow_html=True)


def page_admin(user, db_path):
    try:
        with _ro_connect(db_path) as conn:
            conn.row_factory = sqlite3.Row
            cols = [r[1] for r in conn.execute("PRAGMA table_info(users)")]
            safe = [c for c in cols if not SENSITIVE_COL_RE.search(c)]
            users = conn.execute(f"SELECT {', '.join(_qi(c) for c in safe)} FROM users ORDER BY id").fetchall()
            roles = conn.execute("SELECT role, COUNT(*) FROM users GROUP BY role ORDER BY 2 DESC").fetchall()
            try:
                locked = conn.execute("SELECT COUNT(*) FROM login_attempts WHERE locked_until > ?",
                                      (time.strftime("%Y-%m-%dT%H:%M:%S", time.gmtime()),)).fetchone()[0]
            except sqlite3.Error:
                locked = 0
    except sqlite3.Error as exc:
        st.error(f"Could not read users: {type(exc).__name__}")
        return
    kpis = "".join(f'<div class="fq-kpi"><div class="k">{esc(r[0])}</div><div class="v">{r[1]}</div></div>' for r in roles)
    st.markdown(
        f'<div class="fq-kpis"><div class="fq-kpi"><div class="k">Users</div><div class="v">{len(users)}</div></div>'
        f'<div class="fq-kpi"><div class="k">Locked accounts</div><div class="v">{locked}</div></div>{kpis}</div>',
        unsafe_allow_html=True)
    st.markdown('<div class="fq-section-title"><div class="h">Users</div><span>Read-only · password and '
                'security-answer hashes are never shown</span></div>'
                + html_table(safe, [tuple(u) for u in users]), unsafe_allow_html=True)


def page_profile(user, db_path):
    created = None
    try:
        with _ro_connect(db_path) as conn:
            row = conn.execute("SELECT created_at FROM users WHERE id = ?", (user.get("id"),)).fetchone()
            created = row[0][:10] if row and row[0] else None
    except sqlite3.Error:
        pass
    initials = "".join(w[:1] for w in str(user.get("username", "U")).split()[:2]).upper() or "U"
    st.markdown(
        f'<div class="fq-card" style="display:flex;gap:16px;align-items:center;max-width:620px">'
        f'<div class="fq-logo" style="width:52px;height:52px;flex-basis:52px;border-radius:14px;color:#fff;'
        f'font-weight:700;font-size:18px">{esc(initials)}</div><div>'
        f'<div class="h">{esc(user.get("username"))}</div><p>{esc(user.get("email"))}</p>'
        f'<p style="margin-top:6px">{badge("info", user.get("role"))}'
        f'{"&nbsp; <span style=font-size:12px;color:#6b7280>Member since " + esc(created) + "</span>" if created else ""}'
        f'</p></div></div>', unsafe_allow_html=True)
    st.caption("To change your password, sign out and use Forgot password on the sign-in page.")


def page_command(user):
    try:
        from dashboard_views import render_dashboard
    except Exception as exc:  # keep the shell usable even if the module breaks
        st.error(f"Dashboard module could not be loaded: {type(exc).__name__}: {exc}")
        return
    render_dashboard(user)


def page_customer_quotes(user):
    try:
        from quote_ui import render_customer_quotes
    except Exception as exc:
        st.error(f"Quote module could not be loaded: {type(exc).__name__}: {exc}")
        return
    view = st.session_state.get("customer_view", "request")
    c1, c2, _ = st.columns([1, 1, 2])
    with c1:
        if st.button("Request a Freight Quote", key="fq_cq_request", width="stretch",
                     type="primary" if view == "request" else "secondary"):
            st.session_state["customer_view"] = "request"
            st.rerun()
    with c2:
        if st.button("My Quotes / Shipment Status", key="fq_cq_list", width="stretch",
                     type="primary" if view == "quotes" else "secondary"):
            st.session_state["customer_view"] = "quotes"
            st.rerun()
    try:
        render_customer_quotes(user, view_mode=view)
    except TypeError:
        render_customer_quotes(user)


# ================================================================= SHELL ====

def allowed(pid, role):
    return role in PAGE_ROLES.get(pid, ALL_ROLES)


def _extra_pages():
    folder = BASE_DIR / "pages"
    if not folder.is_dir():
        return []
    return sorted(p for p in folder.glob("*.py") if not p.name.startswith("_"))


def _on_search():
    q = (st.session_state.get("fq_search") or "").strip()
    if q:
        st.session_state["fq_pending_q"] = q
        st.session_state["fq_page"] = "copilot"
    st.session_state["fq_search"] = ""


def _go(pid):
    st.session_state["fq_page"] = pid


def render_sidebar(user, on_sign_out):
    role = _norm_role(user.get("role"))
    active = st.session_state.get("fq_page", "copilot")
    with st.sidebar:
        st.markdown(
            f'<div class="fq-brand"><div class="fq-logo">{LOGO_SVG}</div><div style="min-width:0">'
            f'<div class="name">{esc(BRAND)}</div>'
            f'<div class="who">{esc(user.get("role"))} · {esc(user.get("email"))}</div></div></div>',
            unsafe_allow_html=True)
        st.text_input("Search or ask", key="fq_search", placeholder="Search or ask...",
                      label_visibility="collapsed", on_change=_on_search)
        st.html("""<script>
          (function(){const d=window.parent.document; if(d.__fqCtrlK) return; d.__fqCtrlK=true;
            d.addEventListener('keydown',function(e){ if((e.ctrlKey||e.metaKey)&&e.key.toLowerCase()==='k'){
              const i=d.querySelector('.st-key-fq_search input'); if(i){e.preventDefault(); i.focus();}}});})();
          </script>""", unsafe_allow_javascript=True)
        with st.container(key="fq_nav"):
            section = None
            for sec, pid, label, icon in NAV:
                if not allowed(pid, role):
                    continue
                if sec != section:
                    section = sec
                    st.markdown(f'<div class="fq-nav-label">{esc(sec)}</div>', unsafe_allow_html=True)
                st.button(label, key=f"nav_{pid}", icon=f":material/{icon}:", width="stretch",
                          on_click=_go, args=(pid,))
            for page in _extra_pages():
                try:
                    st.page_link(f"pages/{page.name}", label=page.stem.split("_", 1)[-1].replace("_", " ").title(),
                                 icon=":material/description:")
                except Exception:
                    pass
            st.button("Command Center", key="nav_command", icon=":material/space_dashboard:",
                      width="stretch", on_click=_go, args=("command",))
        st.button("Sign out", key="fq_signout", on_click=on_sign_out)


def _status_pills(user, db_paths):
    tables = accessible_tables(user, db_paths)
    rows = sum(t["rows"] or 0 for t in tables)
    flagged = 0
    for t in tables[:6]:
        if _status_col(t):
            w = watch(t)
            flagged += 1 if w["level"] in ("high", "medium") else 0
    turns = len(st.session_state.get("fq_copilot_history", [])) // 2
    dot = "amber" if flagged else ""
    return (f'<span class="fq-pill"><span class="dot {dot}"></span>live · tables {len(tables)} · rows {rows:,}'
            f' · flagged {flagged} · cycle {turns}</span>'
            f'<span class="fq-pill"><span class="dot"></span>SQLite read-only · Copilot ready</span>')


def render_topbar(pid, user, db_paths):
    st.markdown(
        f'<div class="fq-topbar"><div style="min-width:0"><div class="title">{esc(PAGE_TITLES.get(pid, pid))}</div>'
        f'<div class="subtitle">{esc(PAGE_SUBTITLES.get(pid, ""))}</div></div>'
        f'<div class="fq-pills">{_status_pills(user, db_paths)}</div></div>',
        unsafe_allow_html=True)


def render_app(user, db_path, on_sign_out):
    ss = st.session_state
    role = _norm_role(user.get("role"))
    ss.setdefault("fq_page", "copilot")
    if ss["fq_page"] != "command" and (ss["fq_page"] not in PAGE_TITLES or not allowed(ss["fq_page"], role)):
        ss["fq_page"] = "copilot"
    # sidebar callbacks have already run, so this is the page to draw
    pid = ss["fq_page"]
    inject_app_css(pid)
    db_paths = discover_databases(db_path)

    render_sidebar(user, on_sign_out)
    render_topbar(pid, user, db_paths)

    with st.container(key="fq_toolbar"):
        if pid == "copilot":
            st.selectbox("Answer language", ["English"], key="fq_lang", label_visibility="collapsed",
                         help="Answers are generated in English.")
        else:
            st.markdown('<div style="height:2px"></div>', unsafe_allow_html=True)

    if pid == "copilot":
        page_copilot(user, db_paths)
    elif pid == "command":
        page_command(user)
    elif pid == "quotes" and role == "customer":
        page_customer_quotes(user)
    elif pid == "admin":
        page_admin(user, db_path)
    elif pid == "profile":
        page_profile(user, db_path)
    else:
        page_tables(pid, user, db_paths)
'''

UI_SECTION = r'''# ---------------- UI ----------------
# Enterprise UI (fq_ui.py). All authentication logic above is unchanged.

import fq_ui

ss = st.session_state
ss.setdefault("user", None)
ss.setdefault("flash", None)
ss.setdefault("auth_mode", "Sign In")

fq_ui.inject_base_css()


def _sign_out():
    for key in ("user", "fq_page", "fq_copilot_history", "fq_pending_q", "customer_view",
                "selected_login_role", "otp_email", "otp_verified_email"):
        ss.pop(key, None)
    ss.user = None
    ss.auth_mode = "Sign In"


# ---- Signed-in view ----
if ss.user:
    if ss.flash:
        st.toast(ss.flash)
        ss.flash = None
    fq_ui.render_app(ss.user, db_path=DB_PATH, on_sign_out=_sign_out)
    st.stop()


# ---- Signed-out view (Sign In / Register / Forgot Password) ----
if ss.get("next_mode"):
    ss.auth_mode = ss.pop("next_mode")
mode = ss.auth_mode


def _set_mode(new_mode):
    ss.auth_mode = new_mode


fq_ui.inject_login_css(mode)

# Phase 1 RBAC: the selected workspace must match the account's database role.
ROLE_ALIASES = {
    "admin": "admin",
    "operations manager": "operations manager",
    "freight broker": "freight broker",
    "auditor": "auditor",
    "customer": "customer",
}

with st.container(key="fq_login"):
    hero_col, form_col = st.columns([1.08, 1], gap=None)
    with hero_col:
        fq_ui.render_login_hero()
    with form_col:
        with st.container(key="fq_auth_panel"):
            fq_ui.render_auth_header(mode)

            if ss.flash:
                st.success(ss.flash)
                ss.flash = None

            # ---- Sign In ----
            if mode == "Sign In":
                with st.form("signin_form", border=False):
                    identity = st.text_input("Email or username", max_chars=254)
                    password = st.text_input("Password", type="password",
                                             max_chars=MAX_PASSWORD_LENGTH)
                    st.markdown('<div style="height:4px"></div>', unsafe_allow_html=True)
                    c_submit, c_links = st.columns([1, 2.2], vertical_alignment="center")
                    with c_submit:
                        submit = st.form_submit_button("Sign in", type="primary")
                    with c_links:
                        with st.container(horizontal=True, horizontal_alignment="right",
                                          key="fq_auth_links"):
                            st.form_submit_button("Register", type="tertiary",
                                                  on_click=_set_mode, args=("Register",))
                            st.form_submit_button("Forgot password", type="tertiary",
                                                  on_click=_set_mode, args=("Forgot Password",))

                if submit and ss.auth_mode == "Sign In":
                    user, error = authenticate(identity, password)
                    if user:
                        selected_role = ss.get("selected_login_role")
                        actual_role = str(user["role"]).strip().casefold()
                        expected_role = ROLE_ALIASES.get(
                            str(selected_role).strip().casefold()) if selected_role else None
                        verified_role = ROLE_ALIASES.get(actual_role)

                        if expected_role is None:
                            st.warning("Please select a workspace before signing in.")
                        elif verified_role is None:
                            st.error("Your account has an unrecognized role. "
                                     "Please contact an administrator.")
                        elif expected_role != verified_role:
                            st.error(f"Access denied: this account is assigned the "
                                     f"{user['role']} role, not {selected_role}. "
                                     "Select the workspace assigned to your account.")
                        else:
                            ss.user = {"id": user["id"], "username": user["username"],
                                       "email": user["email"], "role": user["role"]}
                            st.rerun()
                    else:
                        st.error(error)

                fq_ui.render_role_chips(ss.get("selected_login_role"))

            # ---- Register ----
            elif mode == "Register":
                with st.form("register_form", clear_on_submit=False, border=False):
                    username = st.text_input("Username", max_chars=50)
                    email = st.text_input("Email address", max_chars=254)
                    password = st.text_input("Password", type="password",
                                             max_chars=MAX_PASSWORD_LENGTH)
                    confirm = st.text_input("Confirm password", type="password",
                                            max_chars=MAX_PASSWORD_LENGTH)
                    st.markdown('<div class="fq-auth-heading" style="font-size:13px">Security questions'
                                ' <span style="font-weight:400;color:#6b7280">(used to reset your '
                                'password)</span></div>', unsafe_allow_html=True)
                    reg_answers = [
                        st.text_input(question, max_chars=100, key=f"reg_{column}")
                        for column, question in SECURITY_QUESTIONS
                    ]
                    c_submit, c_links = st.columns([1, 1.4], vertical_alignment="center")
                    with c_submit:
                        submit = st.form_submit_button("Create account", type="primary")
                    with c_links:
                        with st.container(horizontal=True, horizontal_alignment="right",
                                          key="fq_auth_links"):
                            st.form_submit_button("Back to sign in", type="tertiary",
                                                  on_click=_set_mode, args=("Sign In",))

                if submit and ss.auth_mode == "Register":
                    ok, msg = register_user(username, email, password, confirm, reg_answers)
                    if ok:
                        ss.flash = msg
                        ss.next_mode = "Sign In"
                        st.rerun()
                    else:
                        st.error(msg)

            # ---- Forgot Password ----
            else:
                method = st.radio("Reset method", ["Security questions", "Email OTP"],
                                  horizontal=True, key="reset_method",
                                  label_visibility="collapsed")

                # -- Method 1: security questions --
                if method == "Security questions":
                    with st.form("reset_answers_form", border=False):
                        identity = st.text_input("Email or username", max_chars=254)
                        reset_answers = [
                            st.text_input(question, max_chars=100, key=f"reset_{column}")
                            for column, question in SECURITY_QUESTIONS
                        ]
                        new_password = st.text_input("New password", type="password",
                                                     max_chars=MAX_PASSWORD_LENGTH, key="qa_new_pw")
                        confirm_password = st.text_input("Confirm new password", type="password",
                                                         max_chars=MAX_PASSWORD_LENGTH,
                                                         key="qa_confirm_pw")
                        change = st.form_submit_button("Reset password", type="primary")

                    if change:
                        ok, msg = reset_with_answers(identity, reset_answers, new_password,
                                                     confirm_password)
                        if ok:
                            ss.flash = msg
                            ss.next_mode = "Sign In"
                            st.rerun()
                        else:
                            st.error(msg)

                # -- Method 2: email OTP --
                else:
                    with st.form("request_otp_form", border=False):
                        email = st.text_input("Registered email address", max_chars=254)
                        send = st.form_submit_button("Send OTP", type="primary")

                    if send:
                        email = normalize_email(email)
                        if not EMAIL_RE.fullmatch(email):
                            st.error("Enter a valid email address.")
                        else:
                            try:
                                ok, msg = request_otp(email)
                                if ok:
                                    ss.otp_email = email
                                    ss.pop("otp_verified_email", None)
                                    st.success(msg)
                                else:
                                    st.warning(msg)
                            except Exception as exc:
                                st.error("Email delivery failed. Check SMTP_EMAIL / SMTP_APP_PASSWORD, "
                                         "or use the Security questions option.")
                                print("SMTP error:", type(exc).__name__, str(exc)[:200])

                    reset_email = ss.get("otp_email")

                    if reset_email and ss.get("otp_verified_email") != reset_email:
                        st.caption(f"OTP requested for: {reset_email}")
                        with st.form("verify_otp_form", border=False):
                            otp = st.text_input("6-digit OTP", max_chars=6)
                            verify = st.form_submit_button("Verify OTP")
                        if verify:
                            ok, msg = verify_otp(reset_email, otp)
                            if ok:
                                ss.otp_verified_email = reset_email
                                st.rerun()
                            else:
                                st.error(msg)

                    if reset_email and ss.get("otp_verified_email") == reset_email:
                        st.success("OTP verified. Set your new password.")
                        with st.form("reset_password_form", border=False):
                            new_password = st.text_input("New password", type="password",
                                                         max_chars=MAX_PASSWORD_LENGTH,
                                                         key="otp_new_pw")
                            confirm_password = st.text_input("Confirm new password", type="password",
                                                             max_chars=MAX_PASSWORD_LENGTH,
                                                             key="otp_confirm_pw")
                            change = st.form_submit_button("Reset password", type="primary")
                        if change:
                            ok, msg = reset_password(reset_email, new_password, confirm_password)
                            if ok:
                                ss.pop("otp_email", None)
                                ss.pop("otp_verified_email", None)
                                ss.flash = msg
                                ss.next_mode = "Sign In"
                                st.rerun()
                            else:
                                st.error(msg)

                st.button("Back to sign in", type="tertiary", key="fq_back_signin",
                          on_click=_set_mode, args=("Sign In",))
'''

CONFIG_TOML = """[theme]
base = "light"
primaryColor = "#5b5bd6"
backgroundColor = "#f7f8fb"
secondaryBackgroundColor = "#ffffff"
textColor = "#1f2937"

[client]
showSidebarNavigation = false
toolbarMode = "minimal"
"""

if not APP.exists():
    raise FileNotFoundError(APP)
source = APP.read_text(encoding="utf-8")
if source.count(MARKER) != 1:
    raise RuntimeError(f"Expected exactly one '{MARKER}' line in app.py; found {source.count(MARKER)}. Nothing changed.")

logic = source.split(MARKER)[0]
tree = ast.parse(logic)
defined = {n.name for n in ast.walk(tree) if isinstance(n, ast.FunctionDef)}
defined |= {t.id for n in ast.walk(tree) if isinstance(n, ast.Assign) for t in n.targets if isinstance(t, ast.Name)}
needed = {"authenticate", "register_user", "reset_with_answers", "request_otp", "verify_otp",
          "reset_password", "normalize_email", "EMAIL_RE", "SECURITY_QUESTIONS",
          "MAX_PASSWORD_LENGTH", "DB_PATH"}
missing = needed - defined
if missing:
    raise RuntimeError(f"app.py is missing {sorted(missing)}; nothing changed.")

# wide layout; sidebar open on desktop, collapsed on phones
logic = logic.replace('layout="centered"', 'layout="wide"')
logic = logic.replace('initial_sidebar_state="collapsed"', 'initial_sidebar_state="auto"')

new_source = logic + UI_SECTION
ast.parse(new_source)
ast.parse(FQ_UI)

stamp = time.strftime("%Y%m%d_%H%M%S")
backup = BACKUP_DIR / f"app.py.before_ui_redesign_{stamp}.bak"
shutil.copy2(APP, backup)
if (PROJECT / "fq_ui.py").exists():
    shutil.copy2(PROJECT / "fq_ui.py", BACKUP_DIR / f"fq_ui.py.{stamp}.bak")

(PROJECT / "fq_ui.py").write_text(FQ_UI, encoding="utf-8")
(UPGRADE_DIR / "fq_ui.py").write_text(FQ_UI, encoding="utf-8")          # reference copy
(UPGRADE_DIR / "app_ui_section.py").write_text(UI_SECTION, encoding="utf-8")
(UPGRADE_DIR / "README.txt").write_text(
    "FreightQuote AI enterprise UI upgrade\n"
    "- fq_ui.py: theme, sidebar shell, AI Copilot (live copy is ../fq_ui.py)\n"
    "- app_ui_section.py: the UI section now at the end of ../app.py\n"
    "- backups/: app.py as it was before each upgrade run\n"
    "Roll back: copy the newest backups/app.py.before_ui_redesign_*.bak over ../app.py\n",
    encoding="utf-8")
tmp = PROJECT / "app.py.ui_redesign.tmp"
tmp.write_text(new_source, encoding="utf-8")
ast.parse(tmp.read_text(encoding="utf-8"))
tmp.replace(APP)

cfg_dir = PROJECT / ".streamlit"
cfg = cfg_dir / "config.toml"
cfg_dir.mkdir(exist_ok=True)
if not cfg.exists():
    cfg.write_text(CONFIG_TOML, encoding="utf-8")
    print("Theme config written:", cfg)
elif "[theme]" not in cfg.read_text(encoding="utf-8"):
    with cfg.open("a", encoding="utf-8") as f:
        f.write("\n" + CONFIG_TOML.split("[client]")[0])
    print("Theme section added to existing", cfg)
else:
    print("Existing [theme] in", cfg, "left unchanged")

print("Folder:", UPGRADE_DIR)
print("Backup:", backup)
print("Wrote:", PROJECT / "fq_ui.py")
print("PASS: app.py UI section replaced; auth/database logic untouched; syntax OK.")
print("Next: restart the server (run the run_colab.py cell again) and refresh the page.")


Mounted at /content/drive
Theme config written: /content/drive/MyDrive/FreightQuote_AI/.streamlit/config.toml
Folder: /content/drive/MyDrive/FreightQuote_AI/ui_upgrade
Backup: /content/drive/MyDrive/FreightQuote_AI/ui_upgrade/backups/app.py.before_ui_redesign_20261025_151019.bak
Wrote: /content/drive/MyDrive/FreightQuote_AI/fq_ui.py
PASS: app.py UI section replaced; auth/database logic untouched; syntax OK.
Next: restart the server (run the run_colab.py cell again) and refresh the page.


In [ ]:
%pip install pyngrok

In [ ]:
from pathlib import Path

p = Path("/content/drive/MyDrive/FreightQuote_AI/run_colab.py")
lines = p.read_text(encoding="utf-8").splitlines()

for i, line in enumerate(lines, 1):
    if "localhost" in line or "ngrok.connect" in line or "addr=" in line:
        print(f"{i}: {line}")

84:     tunnel = ngrok.connect(f"127.0.0.1:{PORT}", "http")


In [ ]:
from pyngrok import ngrok, conf

print("PASS: pyngrok installed successfully")

PASS: pyngrok installed successfully


In [ ]:
from pathlib import Path

project = Path("/content/drive/MyDrive/FreightQuote_AI")

for filename in ["run_colab.py", "app.py"]:
    path = project / filename
    if path.exists():
        for n, line in enumerate(
            path.read_text(encoding="utf-8").splitlines(), 1
        ):
            if "localhost:127.0.0.1" in line or "localhost:" in line:
                print(f"{filename}:{n}: {line.strip()}")

print("Inspection complete.")

Inspection complete.


In [ ]:
from pathlib import Path
import os

print("NGROK CONFIG CHECK")

for key in ("NGROK_CONFIG", "NGROK_CONFIG_PATH"):
    value = os.environ.get(key)
    if value:
        print(f"{key} = {value}")

paths = [
    Path.home() / ".config/ngrok/ngrok.yml",
    Path.home() / ".config/ngrok/ngrok.yaml",
    Path("/content/drive/MyDrive/FreightQuote_AI/ngrok.yml"),
    Path("/content/drive/MyDrive/FreightQuote_AI/ngrok.yaml"),
]

for path in paths:
    if path.exists():
        print(f"Config file found: {path}")
        # Print only lines relevant to endpoint addresses.
        for line in path.read_text(
            encoding="utf-8", errors="replace"
        ).splitlines():
            if any(word in line.lower() for word in
                   ("addr:", "address:", "url:", "localhost")):
                print(" ", line.strip())

print("Check complete.")

NGROK CONFIG CHECK
Config file found: /root/.config/ngrok/ngrok.yml
Check complete.


In [ ]:
from pathlib import Path
import re

config = Path("/root/.config/ngrok/ngrok.yml")

if not config.exists():
    print("Config file not found.")
else:
    text = config.read_text(encoding="utf-8", errors="replace")

    # Hide any token or credential values before displaying.
    text = re.sub(
        r"(?im)^(\s*(?:authtoken|api_key|oauth_token)\s*:\s*).+$",
        r"\1[HIDDEN]",
        text,
    )

    print("Sanitized ngrok configuration:")
    print(text)

Sanitized ngrok configuration:
version: '3'



In [ ]:
from pathlib import Path

p = Path("/content/drive/MyDrive/FreightQuote_AI/run_colab.py")
lines = p.read_text(encoding="utf-8").splitlines()

print("=== PORT CONFIGURATION ===")
for i, line in enumerate(lines, 1):
    if "PORT" in line and not line.lstrip().startswith("#"):
        print(f"{i}: {line}")

print("\n=== NGROK SETUP ===")
for i in range(max(0, 70 - 1), min(len(lines), 100)):
    print(f"{i + 1}: {lines[i]}")

=== PORT CONFIGURATION ===
6: Optional: set PORT before running, e.g.  import os; os.environ["FQ_PORT"] = "8504"
19: PORT = int(os.environ.get("FQ_PORT", "8503"))
55:          "--server.port", str(PORT),
62:     health = f"http://127.0.0.1:{PORT}/_stcore/health"
68:                 print(f"Streamlit running on 127.0.0.1:{PORT} (pid {proc.pid})")
84:     tunnel = ngrok.connect(f"127.0.0.1:{PORT}", "http")

=== NGROK SETUP ===
70:         except requests.RequestException:
71:             pass
72:         time.sleep(1)
73:     print("Streamlit failed to start. Log:\n")
74:     print(LOG_FILE.read_text(errors="replace"))
75:     raise SystemExit(1)
76: 
77: 
78: def start_tunnel():
79:     token = get_secret("NGROK_AUTHTOKEN")
80:     if not token:
81:         raise SystemExit("NGROK_AUTHTOKEN missing. Add it in Colab Secrets and enable notebook access.")
82:     conf.get_default().auth_token = token
83:     # IPv4 address explicitly -> no "failed to open private leg [::1]" warnings
84:   

In [ ]:
import os
import requests
from google.colab import userdata
from pyngrok import ngrok, conf

PORT = int(os.environ.get("FQ_PORT", "8503"))

# 1. Check that the existing Streamlit server is alive.
health_url = f"http://127.0.0.1:{PORT}/_stcore/health"

try:
    response = requests.get(health_url, timeout=5)
    print("Streamlit health:", response.status_code)
    if not response.ok:
        raise RuntimeError(
            "Streamlit health check failed. Stop here; do not create a tunnel."
        )
except requests.RequestException as exc:
    raise RuntimeError(
        f"Streamlit is not reachable on port {PORT}. "
        "Do not restart ngrok until the server is running."
    ) from exc

# 2. Stop old ngrok tunnels, without stopping Streamlit.
ngrok.kill()

# 3. Load the existing token from Colab Secrets.
token = userdata.get("NGROK_AUTHTOKEN")
if not token:
    raise RuntimeError(
        "NGROK_AUTHTOKEN is missing from Colab Secrets."
    )

conf.get_default().auth_token = token

# 4. Create a fresh tunnel to the correct local address.
tunnel = ngrok.connect(
    addr=f"http://127.0.0.1:{PORT}",
    proto="http"
)

print("\nFreightQuote AI URL:", tunnel.public_url)
print("Open this URL in your browser.")

Streamlit health: 200

FreightQuote AI URL: https://reptile-unspoken-tuition.ngrok-free.dev
Open this URL in your browser.


In [ ]:
from pathlib import Path
import sqlite3

db = Path("/content/drive/MyDrive/FreightQuote_AI/freightquote.db")

if not db.exists():
    print("Database file not found:", db)
else:
    con = sqlite3.connect(str(db))
    con.row_factory = sqlite3.Row

    tables = [
        r["name"] for r in con.execute(
            "SELECT name FROM sqlite_master WHERE type='table'"
        )
    ]
    print("Users table exists:", "users" in tables)

    if "users" in tables:
        columns = [
            r["name"] for r in con.execute("PRAGMA table_info(users)")
        ]
        print("Available user columns:", columns)

        if "email" in columns and "role" in columns:
            rows = con.execute(
                """
                SELECT username, email, role
                FROM users
                WHERE lower(email) = lower(?)
                """,
                ("shabuktr83@gmail.com",)
            ).fetchall()

            if rows:
                for row in rows:
                    print(dict(row))
            else:
                print("No matching email found in this database.")
    con.close()

Users table exists: True
Available user columns: ['id', 'username', 'email', 'password_hash', 'role', 'created_at', 'pet_hash', 'school_hash', 'city_hash', 'failed_attempts', 'locked_until']
{'username': 'Junaid', 'email': 'shabuktr83@gmail.com', 'role': 'Customer'}


In [ ]:

from pathlib import Path
import sqlite3

DB = Path("/content/drive/MyDrive/FreightQuote_AI/freightquote.db")
TARGET_EMAIL = "shabuktr83@gmail.com"
EXPECTED_OLD_ROLE = "Customer"
NEW_ROLE = "Operations Manager"

if not DB.exists():
    raise FileNotFoundError(f"Database not found: {DB}")

with sqlite3.connect(str(DB), timeout=10) as con:
    con.row_factory = sqlite3.Row

    # Verify the exact account and current role first.
    user = con.execute(
        """
        SELECT id, username, email, role
        FROM users
        WHERE lower(email) = lower(?)
        """,
        (TARGET_EMAIL,),
    ).fetchone()

    if user is None:
        raise RuntimeError("Target account not found. No changes made.")

    print(
        f"Account found: {user['username']} | "
        f"{user['email']} | Current role: {user['role']}"
    )

    if user["role"] == NEW_ROLE:
        print("No change needed; the account already has this role.")
    elif user["role"] != EXPECTED_OLD_ROLE:
        raise RuntimeError(
            f"Expected role '{EXPECTED_OLD_ROLE}', found "
            f"'{user['role']}'. No changes made."
        )
    else:
        con.execute(
            """
            UPDATE users
            SET role = ?
            WHERE id = ?
              AND lower(email) = lower(?)
              AND role = ?
            """,
            (NEW_ROLE, user["id"], TARGET_EMAIL, EXPECTED_OLD_ROLE),
        )

        if con.execute("SELECT changes()").fetchone()[0] != 1:
            raise RuntimeError("Exactly one account was not updated.")

        updated = con.execute(
            "SELECT username, email, role FROM users WHERE id = ?",
            (user["id"],),
        ).fetchone()

        print("PASS: Role updated successfully.")
        print(dict(updated))

print("Finished.")


Account found: Junaid | shabuktr83@gmail.com | Current role: Customer
PASS: Role updated successfully.
{'username': 'Junaid', 'email': 'shabuktr83@gmail.com', 'role': 'Operations Manager'}
Finished.


In [ ]:
from pathlib import Path

project = Path("/content/drive/MyDrive/FreightQuote_AI")

print("=== CUSTOMS REFERENCES ===")
for path in project.rglob("*.py"):
    if "ui_upgrade/backups" in str(path):
        continue

    try:
        lines = path.read_text(
            encoding="utf-8", errors="replace"
        ).splitlines()
    except OSError:
        continue

    matches = [
        (i, line.strip())
        for i, line in enumerate(lines, 1)
        if any(word in line.lower() for word in (
            "customs", "clearance", "customs_hold",
            "customs-hold", "read_only"
        ))
    ]

    if matches:
        print(f"\n--- {path.name} ---")
        for number, line in matches[:15]:
            print(f"{number}: {line[:220]}")

=== CUSTOMS REFERENCES ===

--- fq_ui.py ---
50: ("Agents", "customs", "Customs", "description"),
70: "customs": "Customs holds and clearance status.",
87: "customs": ["custom", "clearance", "hold", "declaration"],
101: "customs": ALL_ROLES - {"customer"},

--- fq_ui.py ---
50: ("Agents", "customs", "Customs", "description"),
70: "customs": "Customs holds and clearance status.",
87: "customs": ["custom", "clearance", "hold", "declaration"],
101: "customs": ALL_ROLES - {"customer"},


In [ ]:

from pathlib import Path

project = Path("/content/drive/MyDrive/FreightQuote_AI")

# Inspect the UI code around Customs configuration and rendering.
ui = project / "fq_ui.py"
lines = ui.read_text(encoding="utf-8").splitlines()

print("=== fq_ui.py: Customs configuration ===")
for start, end in [(40, 110), (110, 240), (240, 380)]:
    if start < len(lines):
        print(f"\n--- Lines {start + 1}-{min(end, len(lines))} ---")
        for i in range(start, min(end, len(lines))):
            print(f"{i + 1}: {lines[i]}")

print("\n=== Database references in project Python files ===")
for path in project.rglob("*.py"):
    if "backups" in path.parts:
        continue
    try:
        content = path.read_text(encoding="utf-8", errors="replace")
    except OSError:
        continue

    hits = [
        (i, line.strip())
        for i, line in enumerate(content.splitlines(), 1)
        if any(term in line.lower() for term in (
            "sqlite3.connect", "freightquote.db",
            "customs_hold", "customs_records",
            "clearance_status", "customs_data"
        ))
    ]
    if hits:
        print(f"\n--- {path.relative_to(project)} ---")
        for number, line in hits[:20]:
            print(f"{number}: {line[:200]}")


=== fq_ui.py: Customs configuration ===

--- Lines 41-110 ---
41: # ------------------------------------------------------------- navigation ----
42: # (section, page id, label, material icon)
43: NAV = [
44:     ("Assistant", "copilot", "AI Copilot", "auto_awesome"),
45:     ("Agents", "route", "Route AI", "alt_route"),
46:     ("Agents", "quotes", "Spot Quotes", "attach_money"),
47:     ("Agents", "carriers", "Carriers", "local_shipping"),
48:     ("Agents", "weather", "Weather Risk", "thunderstorm"),
49:     ("Agents", "margin", "Margin", "trending_up"),
50:     ("Agents", "customs", "Customs", "description"),
51:     ("Insights", "reports", "Reports", "summarize"),
52:     ("Insights", "notifications", "Notifications", "notifications"),
53:     ("Insights", "graph", "Knowledge Graph", "hub"),
54:     ("Insights", "twin", "Digital Twin", "deployed_code"),
55:     ("Platform", "feeds", "Data Feed Center", "upload"),
56:     ("Platform", "admin", "Admin", "shield"),
57:     ("Platform

In [31]:
from pathlib import Path
import shutil

root = Path("/content/drive/MyDrive")

# Find the original notebook
matches = list(root.rglob("Untitled10.ipynb"))

if not matches:
    print("Untitled10.ipynb nahi mili. Drive mein exact location check karo.")
elif len(matches) > 1:
    print("Multiple matches mili hain; koi copy nahi banayi.")
    for p in matches:
        print(p)
else:
    source = matches[0]
    destination = source.with_name("FreightQoute_ai.ipynb")

    if destination.exists():
        print("Target file pehle se exist karti hai:", destination)
    else:
        shutil.copy2(source, destination)
        print("SUCCESS: Notebook copy created.")
        print("New path:", destination)

SUCCESS: Notebook copy created.
New path: /content/drive/MyDrive/Colab Notebooks/FreightQoute_ai.ipynb
